## (old) 300 km radius, smoothing

In [20]:
# -*- coding: utf-8 -*-
"""
Batch-process all BLO PKL files from 2024 and 2025.

The script recursively searches INPUT_ROOT for .pkl files. For each file, it:

1. Reads the perturbation cube from data[5][1].
2. Reads the local date/start time from data[7][1].
3. Processes the file only when the internal start year is in TARGET_YEARS.
4. Fits a 2-D plane to both the raw frame and a Gaussian-background-outlier-cleaned frame, using only pixels within FIT_RADIUS_KM of the image center.
5. Saves CSV/NPZ results, angle/slope plots, 2-D/3-D tilt plots,
   and optionally an animation.
6. Writes each PKL's outputs into its own folder.
7. Creates one batch_processing_summary.csv file.

Time convention:
    data[7][1] is treated as the beginning of the first frame interval.
    Each fitted point is assigned to the center of its interval. Therefore,
    with DT_SECONDS = 120, frame 0 occurs one minute after the PKL start time.

Angle convention:
    0 degrees    = perturbation increases to the right
    +90 degrees  = perturbation increases downward
    -90 degrees  = perturbation increases upward
    +/-180       = perturbation increases to the left
"""

import csv
import gc
import pickle
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.patches import Ellipse
from scipy.ndimage import binary_dilation, gaussian_filter


# ==================================================
# Batch settings
# ==================================================
INPUT_ROOT = Path.home() / "Downloads" / "pkl"
OUTPUT_ROOT = INPUT_ROOT / "BLO_plane_fit_batch_outputs"

# ==================================================
# Run mode
# ==================================================
# True:
#     Process only SINGLE_PKL_FILE. Use this while tuning one night/day.
#
# False:
#     Discover and process all PKL files under INPUT_ROOT.
PROCESS_SINGLE_FILE = True

# Change this to the one PKL you want to examine first.
# An absolute path is also fine.
SINGLE_PKL_FILE = INPUT_ROOT / "processed_data_full_03_25_2025.pkl"

TARGET_YEARS = {2024, 2025, 2026}
RECURSIVE_SEARCH = True

# Animations can make the full batch substantially slower and use much more
# disk space. Set this to False when only the numerical results and plots
# are needed.
MAKE_ANIMATION = True

# If plane_fit_results.npz already exists in a PKL's output folder, skip it.
# Keep this False after changing the fit radius/filter settings so old results
# are recalculated with the new method.
SKIP_IF_RESULTS_EXIST = False

# Continue processing the remaining PKLs after one file fails.
CONTINUE_ON_ERROR = True


# ==================================================
# Analysis settings
# ==================================================
DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

LOCAL_TIME_LABEL_FORMAT = "%H:%M"
LOCAL_TIME_EXACT_FORMAT = "%H:%M:%S"
LOCAL_DATETIME_PRINT_FORMAT = "%Y-%m-%d %H:%M:%S"

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Only pixels inside this radius, measured from the image center, are used
# in the plane fit.
FIT_RADIUS_KM = 300.0

# Preprocessing before fitting the plane.
#
# Keep only one cleaning operation:
#   1. Build a broad Gaussian background.
#   2. Compute high_pass = original - Gaussian background.
#   3. Estimate high-pass scatter with a robust MAD sigma.
#   4. Flag unusually sharp pixels.
#   5. Optionally dilate the mask to include a small halo.
#   6. Replace only flagged pixels with the Gaussian-background value.
#
# Unflagged pixels remain exactly unchanged.
APPLY_STAR_RESIDUAL_FILTER = True

BACKGROUND_GAUSSIAN_SIGMA_KM = 40.0
BACKGROUND_GAUSSIAN_TRUNCATE = 3.0
STAR_RESIDUAL_MAD_SIGMA = 2.5
STAR_MASK_DILATION_KM = 4.0

ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

# Diagnostic comparison arrows use the same scale for raw and preprocessed
# within each selected frame.
DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE = 0.28

# Removed component shown in the animation:
#     removed component = original frame - preprocessed frame
# This shows what the Gaussian-background outlier replacement removed.
REMOVED_COMPONENT_CMAP = "RdBu_r"
REMOVED_COMPONENT_SCALE_PERCENTILE = 99.0
REMOVED_COMPONENT_SCALE_SAMPLE_FRAMES = 12

# Optional static diagnostic figures for tuning one day.
# Each selected frame compares the original image, Gaussian background,
# high-pass residual, detected mask, removed component, final preprocessed,
# and slope vectors before/after preprocessing.
MAKE_PREPROCESSING_DIAGNOSTICS = True

# Explicit frame numbers to inspect.
DIAGNOSTIC_FRAME_INDICES = [0, 30, 60, 90, 120, 150, 180, 210]

# Also inspect the frames nearest these elapsed times in minutes.
DIAGNOSTIC_TIMES_MIN = [30.0, 90.0, 150.0, 210.0, 270.0, 330.0]

TILT_VECTOR_PLOT_STEP = 10
VECTOR_PLOT_PADDING_FRACTION = 0.12

TILT_PLOT_START_TIME_MIN = 90.0
TILT_PLOT_END_TIME_MIN = 500.0

HOURLY_MARKER_SIZE = 55
MIDNIGHT_MARKER_SIZE = 130
HOURLY_LABEL_OFFSET_POINTS = (6, 6)

# On the whole-time 2-D plot, every hour is marked, but fewer labels are drawn.
WHOLE_TIME_LABEL_EVERY_N_HOURS = 8
WHOLE_TIME_COLORBAR_EVERY_N_HOURS = 2

# Separate sizes for the whole-range and zoomed 2-D tilt plots.
WHOLE_TIME_TILT_FIGSIZE = (18, 9)
ZOOM_TIME_TILT_FIGSIZE = (14, 7.5)
VECTOR_3D_FIGSIZE = (13, 7.5)

# Thin colorbars for the vector plots.
VECTOR_COLORBAR_FRACTION = 0.022
VECTOR_COLORBAR_PAD = 0.025
VECTOR_COLORBAR_ASPECT = 35

# For the hourly-average tilt plots, only include local clock hours from
# 21:00 through 03:00 (inclusive), i.e. 9 PM to 3 AM.
HOURLY_AVERAGE_START_HOUR = 21
HOURLY_AVERAGE_END_HOUR = 3


# ==================================================
# Time helpers
# ==================================================
def parse_pkl_start_datetime(date_start):
    """Parse data[7][1] = [date, start_time] as a local datetime."""
    if (
        not isinstance(date_start, (list, tuple, np.ndarray))
        or len(date_start) < 2
    ):
        raise ValueError(
            "Expected data[7][1] to contain [date, start_time], "
            f"but got: {date_start!r}"
        )

    combined_text = f"{str(date_start[0]).strip()} {str(date_start[1]).strip()}"

    accepted_formats = (
        "%m_%d_%Y %H:%M:%S",
        "%m_%d_%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in accepted_formats:
        try:
            return datetime.strptime(combined_text, date_format)
        except ValueError:
            pass

    raise ValueError(
        "Could not parse the PKL date/start time: "
        f"{date_start!r}. Add its format to accepted_formats."
    )


def local_datetime_from_elapsed_minutes(start_datetime, elapsed_minutes):
    return start_datetime + timedelta(minutes=float(elapsed_minutes))


def make_hour_boundaries(start_datetime, time_values):
    """Return exact whole local hours within the elapsed-time range."""
    time_values = np.asarray(time_values, dtype=np.float64)
    finite_values = time_values[np.isfinite(time_values)]

    if finite_values.size == 0:
        return np.array([], dtype=object), np.array([], dtype=np.float64)

    minimum_elapsed = float(np.nanmin(finite_values))
    maximum_elapsed = float(np.nanmax(finite_values))

    minimum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, minimum_elapsed
    )
    maximum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, maximum_elapsed
    )

    first_hour = minimum_datetime.replace(
        minute=0, second=0, microsecond=0
    )
    if first_hour < minimum_datetime:
        first_hour += timedelta(hours=1)

    hour_datetimes = []
    current_hour = first_hour
    while current_hour <= maximum_datetime:
        hour_datetimes.append(current_hour)
        current_hour += timedelta(hours=1)

    elapsed_hour_minutes = np.array(
        [
            (hour_datetime - start_datetime).total_seconds() / 60.0
            for hour_datetime in hour_datetimes
        ],
        dtype=np.float64,
    )

    return np.array(hour_datetimes, dtype=object), elapsed_hour_minutes


def format_hour_tick(start_datetime, elapsed_minutes):
    local_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, elapsed_minutes
    )
    if local_datetime.hour == 0:
        local_label = "Midnight\n00:00"
    else:
        local_label = local_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)

    return f"{elapsed_minutes:.0f} min\n{local_label}"


def apply_hourly_elapsed_local_labels_to_colorbar(
    colorbar,
    start_datetime,
    time_values,
    label_every_n_hours=1,
):
    hour_datetimes, hour_elapsed_minutes = make_hour_boundaries(
        start_datetime, time_values
    )

    if hour_elapsed_minutes.size == 0:
        finite_values = np.asarray(time_values, dtype=np.float64)
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            return

        hour_elapsed_minutes = np.unique(
            np.array(
                [
                    float(np.nanmin(finite_values)),
                    float(np.nanmax(finite_values)),
                ]
            )
        )
        hour_datetimes = np.array(
            [
                local_datetime_from_elapsed_minutes(start_datetime, value)
                for value in hour_elapsed_minutes
            ],
            dtype=object,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    keep = np.array(
        [
            (hour_datetime.hour == 0)
            or (hour_datetime.hour % label_every_n_hours == 0)
            for hour_datetime in hour_datetimes
        ],
        dtype=bool,
    )

    if np.any(keep):
        hour_elapsed_minutes = hour_elapsed_minutes[keep]

    colorbar.set_ticks(hour_elapsed_minutes)
    colorbar.set_ticklabels(
        [
            format_hour_tick(start_datetime, elapsed_minutes)
            for elapsed_minutes in hour_elapsed_minutes
        ]
    )
    colorbar.set_label("Elapsed time / local time")


def find_nearest_frame_indices_for_hours(
    selected_indices,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    target_hour_datetimes, target_hour_minutes = make_hour_boundaries(
        start_datetime, times_minutes[selected_indices]
    )

    if target_hour_minutes.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    marker_indices = []

    for target_datetime in target_hour_datetimes:
        differences = np.array(
            [
                abs(
                    (
                        local_datetimes[frame_index] - target_datetime
                    ).total_seconds()
                )
                for frame_index in selected_indices
            ],
            dtype=np.float64,
        )
        marker_indices.append(selected_indices[int(np.argmin(differences))])

    marker_indices = np.asarray(marker_indices, dtype=int)

    keep = np.concatenate(
        [
            np.array([True]),
            marker_indices[1:] != marker_indices[:-1],
        ]
    )

    return marker_indices[keep], target_hour_datetimes[keep]


def add_hourly_markers_2d(
    ax,
    selected_indices,
    x_values,
    y_values,
    local_datetimes,
    start_datetime,
    times_minutes,
    label_every_n_hours=1,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.2,
            label="1-hour marks",
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            zorder=8,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        is_midnight = target_datetime.hour == 0
        should_label = (
            is_midnight
            or target_datetime.hour % label_every_n_hours == 0
        )

        if not should_label:
            continue

        label_text = (
            "Midnight\n00:00"
            if is_midnight
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.annotate(
            label_text,
            xy=(x_values[frame_index], y_values[frame_index]),
            xytext=HOURLY_LABEL_OFFSET_POINTS,
            textcoords="offset points",
            fontsize=8,
            fontweight="bold" if is_midnight else "normal",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=9,
        )


def add_hourly_markers_3d(
    ax,
    selected_indices,
    x_values,
    y_values,
    z_values,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            z_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.0,
            label="1-hour marks",
            depthshade=False,
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            z_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            depthshade=False,
            zorder=8,
        )

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        label_text = (
            "Midnight\n00:00"
            if target_datetime.hour == 0
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.text(
            x_values[frame_index],
            y_values[frame_index],
            z_values[frame_index],
            f"  {label_text}",
            fontsize=8,
            zorder=9,
        )


# ==================================================
# Data and fitting helpers
# ==================================================
def load_wave_cube(pkl_file):
    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
        date_start = data[7][1]
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "The PKL does not contain the expected data[5][1] perturbation "
            "cube and data[7][1] date/start time."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected perturbation data with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    return label, wave_cube, date_start


def robust_sigma_from_mad(values):
    """
    Estimate a Gaussian-equivalent sigma from the median absolute deviation.
    Returns NaN when the estimate cannot be formed.
    """
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))

    if not np.isfinite(mad) or mad <= 0:
        return np.nan

    return 1.4826 * mad


def preprocess_frame_for_plane_fit(frame, fit_radius_mask, return_steps=False):
    """
    Remove compact sharp residual peaks using a Gaussian-background detector.

    Only detected outlier pixels are changed. The rest of the image is left
    exactly as it was in the raw frame.
    """
    frame = np.asarray(frame, dtype=np.float64)
    finite = np.isfinite(frame)

    steps = {
        "original": frame.copy(),
        "gaussian_background": np.full_like(frame, np.nan, dtype=np.float64),
        "high_pass": np.full_like(frame, np.nan, dtype=np.float64),
        "star_mask": np.zeros_like(frame, dtype=bool),
        "removed": np.zeros_like(frame, dtype=np.float64),
        "preprocessed": frame.copy(),
    }

    if not APPLY_STAR_RESIDUAL_FILTER or BACKGROUND_GAUSSIAN_SIGMA_KM <= 0:
        return (frame.copy(), steps) if return_steps else frame.copy()

    statistics_mask = finite & fit_radius_mask
    if np.count_nonzero(statistics_mask) < MIN_VALID_PIXELS:
        return (frame.copy(), steps) if return_steps else frame.copy()

    sigma_y_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DY_KM
    sigma_x_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DX_KM

    # Normalized Gaussian filtering prevents NaNs/missing pixels from
    # artificially depressing the local background.
    filled_values = np.where(finite, frame, 0.0)
    finite_weights = finite.astype(np.float64)

    blurred_values = gaussian_filter(
        filled_values,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )
    blurred_weights = gaussian_filter(
        finite_weights,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )

    gaussian_background = np.full_like(frame, np.nan, dtype=np.float64)
    good_weight = blurred_weights > 1.0e-8
    gaussian_background[good_weight] = (
        blurred_values[good_weight] / blurred_weights[good_weight]
    )

    high_pass = frame - gaussian_background
    hp_values = high_pass[statistics_mask]
    hp_center = float(np.median(hp_values))
    hp_sigma = robust_sigma_from_mad(hp_values)

    star_mask = np.zeros_like(frame, dtype=bool)

    if np.isfinite(hp_sigma) and hp_sigma > 0:
        star_mask = (
            finite
            & np.isfinite(gaussian_background)
            & (
                np.abs(high_pass - hp_center)
                > STAR_RESIDUAL_MAD_SIGMA * hp_sigma
            )
        )

        if STAR_MASK_DILATION_KM is not None and STAR_MASK_DILATION_KM > 0:
            dilation_pixels = max(
                int(round(STAR_MASK_DILATION_KM / min(DX_KM, DY_KM))),
                1,
            )
            star_mask = binary_dilation(
                star_mask,
                iterations=dilation_pixels,
            ) & finite

    preprocessed = frame.copy()
    preprocessed[star_mask] = gaussian_background[star_mask]
    removed = frame - preprocessed

    steps["gaussian_background"] = gaussian_background
    steps["high_pass"] = high_pass
    steps["star_mask"] = star_mask
    steps["removed"] = removed
    steps["preprocessed"] = preprocessed.copy()

    if return_steps:
        return preprocessed, steps

    return preprocessed


def solve_plane_least_squares(x_values, y_values, z_values):
    """Return plane coefficients a, b, c for z = a*x + b*y + c."""
    design_matrix = np.column_stack(
        [
            x_values,
            y_values,
            np.ones(x_values.size, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_values,
        rcond=None,
    )

    return coefficients, design_matrix



def fit_plane_to_image(
    frame_for_fit,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
):
    """
    Fit z = a*x + b*y + c directly to the supplied image.

    No preprocessing or iterative robust rejection is performed here.
    This makes it possible to use exactly the same fitting method for the
    raw and preprocessed images.
    """
    frame_sample = frame_for_fit[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    radius_sample = fit_radius_mask[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        radius_sample
        & np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN
    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))
    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    coefficients, design_matrix = solve_plane_least_squares(
        x_valid,
        y_valid,
        z_valid,
    )

    a, b, c = coefficients
    residuals = z_valid - design_matrix @ coefficients
    rmse = np.sqrt(np.mean(residuals**2))
    slope = np.hypot(a, b)
    angle_deg = (
        (np.degrees(np.arctan2(b, a)) + 180.0) % 360.0
    ) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


def calculate_vector_limits(x_values, y_values, padding_fraction):
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_max + y_padding, y_min - y_padding),
    )


def safe_relative_output_name(pkl_file):
    """Create a unique folder name from the PKL's path relative to INPUT_ROOT."""
    try:
        relative_path = pkl_file.relative_to(INPUT_ROOT).with_suffix("")
        parts = relative_path.parts
    except ValueError:
        parts = (pkl_file.stem,)

    return "__".join(parts)


# ==================================================
# Plot helper
# ==================================================

# ==================================================
# Plot helpers
# ==================================================
def _draw_tilt_vector_2d_panel(
    ax,
    time_mask,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    panel_title,
    x_limits,
    y_limits,
    label_every_n_hours=1,
):
    selected_all = np.flatnonzero(time_mask)
    scatter = None

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(u_plot, v_plot, linewidth=0.8, alpha=0.30)

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]
        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        add_hourly_markers_2d(
            ax=ax,
            selected_indices=selected_all,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
            label_every_n_hours=label_every_n_hours,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(panel_title)
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    return scatter


def save_tilt_vector_2d_comparison_plot(
    raw_time_mask,
    preprocessed_time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    raw_x_slope_per_100km,
    raw_y_slope_per_100km,
    preprocessed_x_slope_per_100km,
    preprocessed_y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
    figure_size=(14, 18),
):
    """
    Save a three-panel tilt-vector comparison:

        Top:    Raw
        Middle: Preprocessed
        Bottom: Difference = Preprocessed - Raw

    Raw and preprocessed use identical X/Y limits.

    The difference panel uses its own tighter symmetric X/Y limits so that
    small preprocessing-induced changes are visible.
    """
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=figure_size,
    )

    scatter_raw = _draw_tilt_vector_2d_panel(
        ax=axes[0],
        time_mask=raw_time_mask,
        x_slope_per_100km=raw_x_slope_per_100km,
        y_slope_per_100km=raw_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Raw data",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    scatter_pre = _draw_tilt_vector_2d_panel(
        ax=axes[1],
        time_mask=preprocessed_time_mask,
        x_slope_per_100km=preprocessed_x_slope_per_100km,
        y_slope_per_100km=preprocessed_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Preprocessed",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    # --------------------------------------------------
    # Difference = preprocessed - raw
    # --------------------------------------------------
    difference_x = (
        preprocessed_x_slope_per_100km
        - raw_x_slope_per_100km
    )
    difference_y = (
        preprocessed_y_slope_per_100km
        - raw_y_slope_per_100km
    )

    difference_time_mask = (
        raw_time_mask
        & preprocessed_time_mask
        & np.isfinite(difference_x)
        & np.isfinite(difference_y)
    )

    difference_indices = np.flatnonzero(difference_time_mask)

    # Use one symmetric scale for both X and Y so direction/magnitude in the
    # difference panel are not visually distorted.
    if difference_indices.size > 0:
        difference_abs_values = np.concatenate(
            [
                np.abs(difference_x[difference_indices]),
                np.abs(difference_y[difference_indices]),
            ]
        )

        difference_limit = np.nanpercentile(
            difference_abs_values,
            99.0,
        )

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = np.nanmax(difference_abs_values)

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = 1.0e-6

        difference_limit *= 1.15
    else:
        difference_limit = 1.0e-6

    # Y is plotted with the same image-coordinate convention as the other
    # tilt-vector panels: positive Y points downward.
    difference_x_limits = (
        -difference_limit,
        +difference_limit,
    )
    difference_y_limits = (
        +difference_limit,
        -difference_limit,
    )

    scatter_difference = _draw_tilt_vector_2d_panel(
        ax=axes[2],
        time_mask=difference_time_mask,
        x_slope_per_100km=difference_x,
        y_slope_per_100km=difference_y,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title=(
            "Difference: Preprocessed - Raw\n"
            r"$\Delta x = x_{\mathrm{pre}} - x_{\mathrm{raw}},\quad "
            r"\Delta y = y_{\mathrm{pre}} - y_{\mathrm{raw}}$"
        ),
        x_limits=difference_x_limits,
        y_limits=difference_y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    axes[2].set_xlabel(
        "Difference in X slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )
    axes[2].set_ylabel(
        "Difference in Y slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )

    # Add summary statistics directly to the difference panel.
    if difference_indices.size > 0:
        median_dx = np.nanmedian(
            np.abs(difference_x[difference_indices])
        )
        median_dy = np.nanmedian(
            np.abs(difference_y[difference_indices])
        )
        max_dx = np.nanmax(
            np.abs(difference_x[difference_indices])
        )
        max_dy = np.nanmax(
            np.abs(difference_y[difference_indices])
        )

        axes[2].text(
            0.02,
            0.98,
            (
                f"median |Δx| = {median_dx:.6f}\n"
                f"median |Δy| = {median_dy:.6f}\n"
                f"max |Δx| = {max_dx:.6f}\n"
                f"max |Δy| = {max_dy:.6f}"
            ),
            transform=axes[2].transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=10,
        )

    # Use one shared time colorbar for all three panels.
    scatter_for_cbar = (
        scatter_difference
        if scatter_difference is not None
        else (
            scatter_pre
            if scatter_pre is not None
            else scatter_raw
        )
    )

    if scatter_for_cbar is not None:
        cbar = fig.colorbar(
            scatter_for_cbar,
            ax=axes,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )

        combined_time_mask = (
            raw_time_mask
            | preprocessed_time_mask
        )

        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            times_minutes[combined_time_mask],
            label_every_n_hours=colorbar_every_n_hours,
        )

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    fig.suptitle(
        f"{title_text}\n"
        f"{subtitle}\n"
        "Raw and preprocessed use the same scale; "
        "difference uses a tighter symmetric scale",
        fontsize=14,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.055,
        top=0.92,
        hspace=0.32,
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def local_hour_is_in_hourly_average_window(local_hour):
    """
    Return True when the local clock hour falls inside the requested nightly
    window. This supports windows that cross midnight, such as 21:00 to 03:00.
    """
    start_hour = int(HOURLY_AVERAGE_START_HOUR) % 24
    end_hour = int(HOURLY_AVERAGE_END_HOUR) % 24
    local_hour = int(local_hour) % 24

    if start_hour <= end_hour:
        return start_hour <= local_hour <= end_hour

    return (local_hour >= start_hour) or (local_hour <= end_hour)


def compute_hourly_average_rows(
    selected_indices,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    input_file,
    output_folder,
    start_local_datetime,
):
    """
    Group the selected frame-center points by local clock hour and average
    the tilt-vector endpoint for each hour.
    """
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return []

    hour_groups = {}

    for frame_index in selected_indices:
        if not local_hour_is_in_hourly_average_window(
            local_datetimes[frame_index].hour
        ):
            continue

        hour_start = local_datetimes[frame_index].replace(
            minute=0,
            second=0,
            microsecond=0,
        )
        hour_groups.setdefault(hour_start, []).append(int(frame_index))

    rows = []

    night_label = start_local_datetime.strftime("%Y-%m-%d")

    for hour_start in sorted(hour_groups):
        frame_indices = np.asarray(hour_groups[hour_start], dtype=int)

        mean_x = float(np.nanmean(x_slope_per_100km[frame_indices]))
        mean_y = float(np.nanmean(y_slope_per_100km[frame_indices]))
        mean_elapsed_minutes = float(np.nanmean(times_minutes[frame_indices]))

        rows.append(
            {
                "input_file": str(input_file),
                "output_folder": str(output_folder),
                "start_local_datetime": start_local_datetime.isoformat(sep=" "),
                "night_label": night_label,
                "local_date": hour_start.strftime("%Y-%m-%d"),
                "hour_start_local_datetime": hour_start.isoformat(sep=" "),
                "hour_label": hour_start.strftime(LOCAL_TIME_LABEL_FORMAT),
                "is_midnight": int(hour_start.hour == 0),
                "mean_elapsed_minutes": mean_elapsed_minutes,
                "mean_x_slope_per_100km": mean_x,
                "mean_y_slope_per_100km": mean_y,
                "mean_slope_magnitude_per_100km": float(np.hypot(mean_x, mean_y)),
                "frame_count_in_average": int(frame_indices.size),
                "first_frame": int(frame_indices[0]),
                "last_frame": int(frame_indices[-1]),
            }
        )

    return rows


def save_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def load_hourly_average_csv(input_file):
    if not Path(input_file).exists():
        return []

    with open(input_file, "r", newline="") as file:
        return list(csv.DictReader(file))


def save_hourly_average_tilt_plot(
    hourly_rows,
    output_file,
    title_text,
    x_limits,
    y_limits,
    start_local_datetime,
    figure_size=(9, 8),
):
    """
    Save a 2-D tilt plot using one averaged point per selected local hour.
    """
    fig, ax = plt.subplots(figsize=figure_size)

    if hourly_rows:
        mean_x = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_y = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_elapsed = np.array(
            [float(row["mean_elapsed_minutes"]) for row in hourly_rows],
            dtype=np.float64,
        )

        ax.plot(
            mean_x,
            mean_y,
            linewidth=1.2,
            alpha=0.55,
            color="0.35",
            zorder=2,
        )

        scatter = ax.scatter(
            mean_x,
            mean_y,
            c=mean_elapsed,
            cmap="viridis",
            s=95,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            mean_elapsed,
            label_every_n_hours=1,
        )

        midnight_mask = np.array(
            [int(row["is_midnight"]) == 1 for row in hourly_rows],
            dtype=bool,
        )

        if np.any(midnight_mask):
            ax.scatter(
                mean_x[midnight_mask],
                mean_y[midnight_mask],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color="black",
                edgecolors="white",
                linewidths=0.8,
                label="Midnight",
                zorder=4,
            )

        for row, x_here, y_here in zip(hourly_rows, mean_x, mean_y):
            if int(row["is_midnight"]) == 1:
                label_text = "Midnight\n00:00"
                font_weight = "bold"
            else:
                label_text = row["hour_label"]
                font_weight = "normal"

            ax.annotate(
                label_text,
                xy=(x_here, y_here),
                xytext=HOURLY_LABEL_OFFSET_POINTS,
                textcoords="offset points",
                fontsize=8,
                fontweight=font_weight,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    edgecolor="0.5",
                    alpha=0.85,
                ),
                zorder=5,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{title_text}\n"
        "One averaged tilt-vector endpoint is shown for each local hour "
        f"from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_all_days_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def save_all_days_hourly_average_plot(hourly_rows, output_file):
    """
    Plot the hourly averaged tilt points from all processed PKLs together.

    Each night/day is assigned its own color. The rows should already be filtered
    to the requested local-hour window.
    """
    if not hourly_rows:
        return

    x_values = np.array(
        [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )
    y_values = np.array(
        [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )

    x_limits, y_limits = calculate_vector_limits(
        x_values,
        y_values,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    rows_by_night = {}
    for row in hourly_rows:
        night_key = row.get("night_label", row["start_local_datetime"][:10])
        rows_by_night.setdefault(night_key, []).append(row)

    night_keys = sorted(rows_by_night.keys())

    if len(night_keys) <= 20:
        cmap = plt.get_cmap("tab20", max(len(night_keys), 1))
        colors = [cmap(i) for i in range(len(night_keys))]
    else:
        cmap = plt.get_cmap("nipy_spectral", max(len(night_keys), 1))
        colors = [cmap(i / max(len(night_keys) - 1, 1)) for i in range(len(night_keys))]

    for color, night_key in zip(colors, night_keys):
        night_rows = sorted(
            rows_by_night[night_key],
            key=lambda row: row["hour_start_local_datetime"],
        )

        x_here = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        y_here = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        midnight_mask_here = np.array(
            [int(row["is_midnight"]) == 1 for row in night_rows],
            dtype=bool,
        )

        ax.plot(
            x_here,
            y_here,
            linewidth=1.4,
            alpha=0.85,
            color=color,
            zorder=2,
            label=night_key,
        )

        ax.scatter(
            x_here,
            y_here,
            s=75,
            color=color,
            edgecolors="black",
            linewidths=0.5,
            alpha=0.95,
            zorder=3,
        )

        if np.any(midnight_mask_here):
            ax.scatter(
                x_here[midnight_mask_here],
                y_here[midnight_mask_here],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color=color,
                edgecolors="black",
                linewidths=0.8,
                zorder=4,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        "Hourly Averaged Tilt Points: All Days\n"
        f"Local-hour averages from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00; each day has a different color"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=8,
        framealpha=0.9,
        title="Day",
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Preprocessing diagnostics
# ==================================================
def _symmetric_plot_limit(values, percentile=99.0, mask=None):
    values = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(values)
    if mask is not None:
        valid &= mask
    if not np.any(valid):
        return 1.0
    vmax = np.nanpercentile(np.abs(values[valid]), percentile)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    return float(vmax)


def add_diagnostic_slope_vector(
    ax,
    a_here,
    b_here,
    vector_reference_slope,
    nx,
    ny,
):
    """
    Overlay one plane-slope vector on an image.

    vector_reference_slope is shared by the original/preprocessed diagnostic
    panels for the current frame, so the two arrow lengths are directly
    comparable.
    """
    if (
        not np.isfinite(a_here)
        or not np.isfinite(b_here)
        or not np.isfinite(vector_reference_slope)
        or vector_reference_slope <= 0
    ):
        return

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    max_arrow_length_pix = (
        DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)
    )

    arrow_u = (a_here / vector_reference_slope) * max_arrow_length_pix
    arrow_v = (b_here / vector_reference_slope) * max_arrow_length_pix

    ax.scatter(
        [center_x],
        [center_y],
        s=CENTER_MARKER_SIZE,
        c=CENTER_MARKER_COLOR,
        marker="o",
        zorder=6,
    )

    ax.quiver(
        [center_x],
        [center_y],
        [arrow_u],
        [arrow_v],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color=ARROW_COLOR,
        width=0.008,
        headwidth=4.5,
        headlength=6.0,
        headaxislength=5.5,
        zorder=7,
    )


def save_preprocessing_diagnostic_plot(
    frame_index,
    wave_cube,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
    raw_a_values,
    raw_b_values,
    preprocessed_a_values,
    preprocessed_b_values,
    times_minutes,
    local_datetimes,
    out_dir,
):
    """
    Save a compact diagnostic figure for the retained preprocessing method.

    Top row:
      1. Raw + raw slope vector
      2. Gaussian background
      3. High-pass = raw - background

    Bottom row:
      4. Detected replacement mask
      5. Removed component
      6. Final preprocessed + preprocessed slope vector
    """
    nt, ny, nx = wave_cube.shape
    if frame_index < 0 or frame_index >= nt:
        return

    original = np.asarray(wave_cube[frame_index], dtype=np.float64)
    preprocessed, steps = preprocess_frame_for_plane_fit(
        original,
        fit_radius_mask,
        return_steps=True,
    )

    raw_a = raw_a_values[frame_index]
    raw_b = raw_b_values[frame_index]
    pre_a = preprocessed_a_values[frame_index]
    pre_b = preprocessed_b_values[frame_index]

    raw_slope = np.hypot(raw_a, raw_b)
    pre_slope = np.hypot(pre_a, pre_b)

    comparison_slopes = np.array([raw_slope, pre_slope], dtype=np.float64)
    comparison_slopes = comparison_slopes[np.isfinite(comparison_slopes)]
    vector_reference_slope = (
        float(np.nanmax(comparison_slopes))
        if comparison_slopes.size > 0
        else 1.0
    )
    if not np.isfinite(vector_reference_slope) or vector_reference_slope <= 0:
        vector_reference_slope = 1.0

    main_vmax = _symmetric_plot_limit(
        np.concatenate(
            [
                original[np.isfinite(original)],
                preprocessed[np.isfinite(preprocessed)],
                steps["gaussian_background"][
                    np.isfinite(steps["gaussian_background"])
                ],
            ]
        ),
        percentile=99.0,
    )
    high_pass_vmax = _symmetric_plot_limit(
        steps["high_pass"],
        percentile=99.0,
        mask=fit_radius_mask,
    )
    removed_vmax = _symmetric_plot_limit(
        steps["removed"],
        percentile=REMOVED_COMPONENT_SCALE_PERCENTILE,
        mask=fit_radius_mask,
    )

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 11),
    )

    panels = [
        (
            axes[0, 0],
            original,
            (
                "Raw + slope vector\n"
                f"x = {100.0 * raw_a:+.5f}, "
                f"y = {100.0 * raw_b:+.5f}, "
                f"|s| = {100.0 * raw_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
        (
            axes[0, 1],
            steps["gaussian_background"],
            f"Gaussian background\nsigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km",
            CMAP,
            main_vmax,
            "Background amplitude",
        ),
        (
            axes[0, 2],
            steps["high_pass"],
            "High-pass = raw - background",
            REMOVED_COMPONENT_CMAP,
            high_pass_vmax,
            "High-pass amplitude",
        ),
        (
            axes[1, 1],
            steps["removed"],
            "Removed component\n(raw - preprocessed)",
            REMOVED_COMPONENT_CMAP,
            removed_vmax,
            "Removed amplitude",
        ),
        (
            axes[1, 2],
            preprocessed,
            (
                "Final preprocessed + slope vector\n"
                f"x = {100.0 * pre_a:+.5f}, "
                f"y = {100.0 * pre_b:+.5f}, "
                f"|s| = {100.0 * pre_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
    ]

    for ax, panel_data, title_text, cmap, vmax_here, cbar_label in panels:
        image = ax.imshow(
            panel_data,
            origin="upper",
            cmap=cmap,
            vmin=-vmax_here,
            vmax=vmax_here,
        )
        fit_circle = Ellipse(
            xy=(center_x, center_y),
            width=2.0 * FIT_RADIUS_KM / DX_KM,
            height=2.0 * FIT_RADIUS_KM / DY_KM,
            fill=False,
            edgecolor="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.85,
            zorder=4,
        )
        ax.add_patch(fit_circle)
        ax.set_title(title_text, fontsize=10.5)
        ax.set_xlabel("X pixel")
        ax.set_ylabel("Y pixel")
        cbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.040,
            pad=0.025,
            aspect=30,
        )
        cbar.set_label(cbar_label)

    mask_image = axes[1, 0].imshow(
        steps["star_mask"].astype(np.float64),
        origin="upper",
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    fit_circle = Ellipse(
        xy=(center_x, center_y),
        width=2.0 * FIT_RADIUS_KM / DX_KM,
        height=2.0 * FIT_RADIUS_KM / DY_KM,
        fill=False,
        edgecolor="red",
        linestyle="--",
        linewidth=1.1,
        alpha=0.85,
        zorder=4,
    )
    axes[1, 0].add_patch(fit_circle)
    axes[1, 0].set_title("Detected peak mask\nwhite = replaced pixels")
    axes[1, 0].set_xlabel("X pixel")
    axes[1, 0].set_ylabel("Y pixel")
    cbar_mask = fig.colorbar(
        mask_image,
        ax=axes[1, 0],
        fraction=0.040,
        pad=0.025,
        aspect=30,
    )
    cbar_mask.set_ticks([0.0, 1.0])
    cbar_mask.set_ticklabels(["0", "1"])
    cbar_mask.set_label("Mask value")

    add_diagnostic_slope_vector(
        axes[0, 0],
        raw_a,
        raw_b,
        vector_reference_slope,
        nx,
        ny,
    )
    add_diagnostic_slope_vector(
        axes[1, 2],
        pre_a,
        pre_b,
        vector_reference_slope,
        nx,
        ny,
    )

    local_time_text = local_datetimes[frame_index].strftime(
        LOCAL_TIME_EXACT_FORMAT
    )
    replaced_pixel_count = int(np.count_nonzero(steps["star_mask"]))
    inside_fit_count = int(
        np.count_nonzero(steps["star_mask"] & fit_radius_mask)
    )

    fig.suptitle(
        f"Gaussian-Background Peak Diagnostic | "
        f"Frame {frame_index} | "
        f"t = {times_minutes[frame_index]:.1f} min | "
        f"Local time = {local_time_text}\n"
        f"Background sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km | "
        f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma | "
        f"mask dilation = {STAR_MASK_DILATION_KM:g} km | "
        f"fit radius = {FIT_RADIUS_KM:g} km | "
        f"replaced = {replaced_pixel_count} total "
        f"({inside_fit_count} inside fit radius)",
        fontsize=13,
    )

    diagnostic_dir = out_dir / "preprocessing_diagnostics"
    diagnostic_dir.mkdir(parents=True, exist_ok=True)

    output_file = (
        diagnostic_dir
        / f"diagnostic_frame_{frame_index:04d}_"
          f"t_{times_minutes[frame_index]:07.1f}_min.png"
    )

    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def build_diagnostic_frame_indices(
    nt,
    times_minutes,
):
    """
    Combine requested frame numbers with frames nearest the requested
    elapsed times.
    """
    requested_indices = set()

    for frame_index in DIAGNOSTIC_FRAME_INDICES:
        frame_index = int(frame_index)

        if 0 <= frame_index < nt:
            requested_indices.add(frame_index)

    finite_time = np.isfinite(times_minutes)

    if np.any(finite_time):
        valid_indices = np.flatnonzero(
            finite_time
        )

        for target_time_min in DIAGNOSTIC_TIMES_MIN:
            nearest_position = int(
                np.argmin(
                    np.abs(
                        times_minutes[valid_indices]
                        - float(target_time_min)
                    )
                )
            )

            requested_indices.add(
                int(
                    valid_indices[
                        nearest_position
                    ]
                )
            )

    return sorted(requested_indices)


# ==================================================
# Process one PKL
# ==================================================
def process_one_pkl(pkl_file):
    print("\n" + "=" * 80)
    print(f"Reading: {pkl_file}")

    label, wave_cube, date_start = load_wave_cube(pkl_file)
    start_local_datetime = parse_pkl_start_datetime(date_start)

    if start_local_datetime.year not in TARGET_YEARS:
        print(
            f"Skipped: internal year {start_local_datetime.year} "
            f"is not in {sorted(TARGET_YEARS)}"
        )
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_year",
                "output_folder": "",
                "frame_count": "",
                "successful_fits": "",
                "hourly_average_csv": "",
                "message": f"Year {start_local_datetime.year} not selected",
            },
            [],
        )

    output_name = safe_relative_output_name(pkl_file)
    out_dir = OUTPUT_ROOT / str(start_local_datetime.year) / output_name
    out_dir.mkdir(parents=True, exist_ok=True)

    npz_file = out_dir / "plane_fit_results.npz"
    hourly_average_csv_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )
    hourly_average_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    required_existing_files = [
        npz_file,
        hourly_average_csv_file,
        hourly_average_plot_file,
    ]

    if SKIP_IF_RESULTS_EXIST and all(file.exists() for file in required_existing_files):
        print(f"Skipped existing results in: {out_dir}")
        hourly_rows = load_hourly_average_csv(hourly_average_csv_file)
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_existing",
                "output_folder": str(out_dir),
                "frame_count": wave_cube.shape[0],
                "successful_fits": "",
                "hourly_average_csv": str(hourly_average_csv_file),
                "message": "All required result files already exist",
            },
            hourly_rows,
        )

    nt, ny, nx = wave_cube.shape
    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")
    print(
        "PKL interval start datetime: "
        f"{start_local_datetime.strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    frame_numbers = np.arange(nt, dtype=int)

    # data[7][1] is the beginning of the first frame interval.
    # Each fitted point is assigned to the center of its interval.
    times_seconds = (frame_numbers + 0.5) * DT_SECONDS
    times_minutes = times_seconds / 60.0

    local_datetimes = np.array(
        [
            local_datetime_from_elapsed_minutes(
                start_local_datetime,
                elapsed_minutes,
            )
            for elapsed_minutes in times_minutes
        ],
        dtype=object,
    )

    local_datetime_iso = np.array(
        [
            local_datetime.isoformat(sep=" ")
            for local_datetime in local_datetimes
        ],
        dtype=str,
    )

    print(
        "First frame-center datetime: "
        f"{local_datetimes[0].strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    x_1d_km = (
        np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
    ) * DX_KM
    y_1d_km = (
        np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
    ) * DY_KM
    x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)

    radial_distance_km = np.hypot(x_grid_km, y_grid_km)
    fit_radius_mask = radial_distance_km <= FIT_RADIUS_KM
    fit_radius_pixel_count = int(np.count_nonzero(fit_radius_mask))

    print(
        f"Plane-fit region: radius <= {FIT_RADIUS_KM:.1f} km "
        f"({fit_radius_pixel_count} pixels before other validity filters)"
    )
    if APPLY_STAR_RESIDUAL_FILTER:
        print(
            "Spatial preprocessing: Gaussian-background peak replacement "
            f"(sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km, "
            f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma, "
            f"mask dilation = {STAR_MASK_DILATION_KM:g} km)"
        )
    else:
        print("Spatial preprocessing: disabled")

    # Raw plane-fit results
    raw_a_values = np.full(nt, np.nan, dtype=np.float64)
    raw_b_values = np.full(nt, np.nan, dtype=np.float64)
    raw_c_values = np.full(nt, np.nan, dtype=np.float64)
    raw_slope_values = np.full(nt, np.nan, dtype=np.float64)
    raw_angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    raw_rmse_values = np.full(nt, np.nan, dtype=np.float64)
    raw_valid_pixel_counts = np.zeros(nt, dtype=int)

    # Preprocessed plane-fit results
    a_values = np.full(nt, np.nan, dtype=np.float64)
    b_values = np.full(nt, np.nan, dtype=np.float64)
    c_values = np.full(nt, np.nan, dtype=np.float64)
    slope_values = np.full(nt, np.nan, dtype=np.float64)
    angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    rmse_values = np.full(nt, np.nan, dtype=np.float64)
    valid_pixel_counts = np.zeros(nt, dtype=int)

    print("Fitting planes to raw and preprocessed frames...")

    for frame_index in range(nt):
        raw_frame = wave_cube[frame_index]

        (
            raw_a_values[frame_index],
            raw_b_values[frame_index],
            raw_c_values[frame_index],
            raw_slope_values[frame_index],
            raw_angle_values_deg[frame_index],
            raw_rmse_values[frame_index],
            raw_valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            raw_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        preprocessed_frame = preprocess_frame_for_plane_fit(
            raw_frame,
            fit_radius_mask,
        )

        (
            a_values[frame_index],
            b_values[frame_index],
            c_values[frame_index],
            slope_values[frame_index],
            angle_values_deg[frame_index],
            rmse_values[frame_index],
            valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            preprocessed_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        if (
            frame_index == 0
            or frame_index == nt - 1
            or (frame_index + 1) % 25 == 0
        ):
            print(f"  Processed frame {frame_index + 1}/{nt}")

    successful_fits = int(
        np.count_nonzero(
            np.isfinite(slope_values) & np.isfinite(angle_values_deg)
        )
    )

    # ----------------------------------------------
    # Optional preprocessing / fit diagnostics
    # ----------------------------------------------
    if MAKE_PREPROCESSING_DIAGNOSTICS:
        diagnostic_indices = build_diagnostic_frame_indices(
            nt,
            times_minutes,
        )

        print(
            "Saving preprocessing diagnostic plots for frames: "
            + ", ".join(
                str(index)
                for index in diagnostic_indices
            )
        )

        for diagnostic_frame_index in diagnostic_indices:
            save_preprocessing_diagnostic_plot(
                frame_index=diagnostic_frame_index,
                wave_cube=wave_cube,
                x_grid_km=x_grid_km,
                y_grid_km=y_grid_km,
                fit_radius_mask=fit_radius_mask,
                raw_a_values=raw_a_values,
                raw_b_values=raw_b_values,
                preprocessed_a_values=a_values,
                preprocessed_b_values=b_values,
                times_minutes=times_minutes,
                local_datetimes=local_datetimes,
                out_dir=out_dir,
            )

    # ----------------------------------------------
    # Save CSV and NPZ
    # ----------------------------------------------
    csv_file = out_dir / "plane_fit_results.csv"

    with open(csv_file, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "raw_slope_x_amplitude_per_km",
                "raw_slope_y_amplitude_per_km",
                "raw_plane_intercept",
                "raw_slope_magnitude_amplitude_per_km",
                "raw_increase_direction_deg",
                "raw_fit_rmse",
                "raw_valid_pixel_count",
                "preprocessed_slope_x_amplitude_per_km",
                "preprocessed_slope_y_amplitude_per_km",
                "preprocessed_plane_intercept",
                "preprocessed_slope_magnitude_amplitude_per_km",
                "preprocessed_increase_direction_deg",
                "preprocessed_fit_rmse",
                "preprocessed_valid_pixel_count",
            ]
        )

        for frame_index in range(nt):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(
                        LOCAL_TIME_EXACT_FORMAT
                    ),
                    raw_a_values[frame_index],
                    raw_b_values[frame_index],
                    raw_c_values[frame_index],
                    raw_slope_values[frame_index],
                    raw_angle_values_deg[frame_index],
                    raw_rmse_values[frame_index],
                    raw_valid_pixel_counts[frame_index],
                    a_values[frame_index],
                    b_values[frame_index],
                    c_values[frame_index],
                    slope_values[frame_index],
                    angle_values_deg[frame_index],
                    rmse_values[frame_index],
                    valid_pixel_counts[frame_index],
                ]
            )

    np.savez(
        npz_file,
        input_file=str(pkl_file),
        frame=frame_numbers,
        time_seconds=times_seconds,
        time_minutes=times_minutes,
        time_reference="frame_center",
        frame_duration_seconds=DT_SECONDS,
        fit_radius_km=FIT_RADIUS_KM,
        preprocessing_method="gaussian_background_outlier_replacement",
        background_gaussian_sigma_km=BACKGROUND_GAUSSIAN_SIGMA_KM,
        background_gaussian_truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
        star_residual_filter_enabled=APPLY_STAR_RESIDUAL_FILTER,
        star_residual_mad_sigma=STAR_RESIDUAL_MAD_SIGMA,
        star_mask_dilation_km=STAR_MASK_DILATION_KM,
        local_datetime=local_datetime_iso,
        start_local_datetime=start_local_datetime.isoformat(sep=" "),

        raw_slope_x_amplitude_per_km=raw_a_values,
        raw_slope_y_amplitude_per_km=raw_b_values,
        raw_plane_intercept=raw_c_values,
        raw_slope_magnitude_amplitude_per_km=raw_slope_values,
        raw_increase_direction_deg=raw_angle_values_deg,
        raw_fit_rmse=raw_rmse_values,
        raw_valid_pixel_count=raw_valid_pixel_counts,

        preprocessed_slope_x_amplitude_per_km=a_values,
        preprocessed_slope_y_amplitude_per_km=b_values,
        preprocessed_plane_intercept=c_values,
        preprocessed_slope_magnitude_amplitude_per_km=slope_values,
        preprocessed_increase_direction_deg=angle_values_deg,
        preprocessed_fit_rmse=rmse_values,
        preprocessed_valid_pixel_count=valid_pixel_counts,

        # Backward-compatible aliases: these are PREPROCESSED values.
        slope_x_amplitude_per_km=a_values,
        slope_y_amplitude_per_km=b_values,
        plane_intercept=c_values,
        slope_magnitude_amplitude_per_km=slope_values,
        increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
        fit_rmse=rmse_values,
        valid_pixel_count=valid_pixel_counts,
    )

    # ----------------------------------------------
    # Angle plot
    # ----------------------------------------------
    fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

    ax_angle.plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_angle.plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )

    ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
    ax_angle.set_xlabel("Frame")
    ax_angle.set_ylabel(
        "Direction of perturbation increase "
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks(np.arange(-180, 181, 45))
    ax_angle.grid(True, alpha=0.3)
    ax_angle.legend()

    fig_angle.tight_layout()
    angle_plot_file = out_dir / "plane_angle_vs_frame.png"
    fig_angle.savefig(angle_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_angle)

    # ----------------------------------------------
    # Slope plot
    # ----------------------------------------------
    fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

    ax_slope.plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_slope.plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        label="Preprocessed",
    )

    ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
    ax_slope.grid(True, alpha=0.3)
    ax_slope.legend()

    fig_slope.tight_layout()
    slope_plot_file = out_dir / "plane_slope_vs_frame.png"
    fig_slope.savefig(slope_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_slope)

    # ----------------------------------------------
    # Combined angle/slope comparison
    # ----------------------------------------------
    fig_combined, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 8),
        sharex=True,
    )

    axes[0].plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[0].plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[0].set_title("Plane-fit Direction and Slope: Raw vs Preprocessed")
    axes[0].set_ylabel(
        "Increase direction\n"
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    axes[0].set_ylim(-180, 180)
    axes[0].set_yticks(np.arange(-180, 181, 45))
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[1].plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig_combined.tight_layout()
    combined_plot_file = out_dir / "plane_angle_and_slope_vs_frame.png"
    fig_combined.savefig(
        combined_plot_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_combined)

    # ----------------------------------------------
    # Physical tilt vectors: raw vs preprocessed
    # ----------------------------------------------
    raw_x_slope_per_100km = 100.0 * raw_a_values
    raw_y_slope_per_100km = 100.0 * raw_b_values

    x_slope_per_100km = 100.0 * a_values
    y_slope_per_100km = 100.0 * b_values

    raw_finite_vector = (
        np.isfinite(raw_x_slope_per_100km)
        & np.isfinite(raw_y_slope_per_100km)
        & np.isfinite(times_minutes)
    )
    preprocessed_finite_vector = (
        np.isfinite(x_slope_per_100km)
        & np.isfinite(y_slope_per_100km)
        & np.isfinite(times_minutes)
    )

    raw_tilt_plot_time_mask = (
        raw_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )
    preprocessed_tilt_plot_time_mask = (
        preprocessed_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )

    whole_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_finite_vector],
            x_slope_per_100km[preprocessed_finite_vector],
        ]
    )
    whole_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_finite_vector],
            y_slope_per_100km[preprocessed_finite_vector],
        ]
    )

    whole_vector_x_limits, whole_vector_y_limits = calculate_vector_limits(
        whole_combined_x,
        whole_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    zoom_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_tilt_plot_time_mask],
            x_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )
    zoom_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_tilt_plot_time_mask],
            y_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )

    zoom_vector_x_limits, zoom_vector_y_limits = calculate_vector_limits(
        zoom_combined_x,
        zoom_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    # ----------------------------------------------
    # 3-D tilt-vector-vs-time comparison
    # top = raw, bottom = preprocessed
    # ----------------------------------------------
    fig_tilt_3d = plt.figure(figsize=(14, 15))
    ax_raw_3d = fig_tilt_3d.add_subplot(211, projection="3d")
    ax_pre_3d = fig_tilt_3d.add_subplot(212, projection="3d")

    comparison_3d = [
        (
            ax_raw_3d,
            raw_tilt_plot_time_mask,
            raw_x_slope_per_100km,
            raw_y_slope_per_100km,
            "Raw data",
        ),
        (
            ax_pre_3d,
            preprocessed_tilt_plot_time_mask,
            x_slope_per_100km,
            y_slope_per_100km,
            "Preprocessed",
        ),
    ]

    scatter_for_3d_cbar = None

    for (
        ax_here,
        mask_here,
        x_here_all,
        y_here_all,
        panel_title,
    ) in comparison_3d:
        selected_indices = np.flatnonzero(mask_here)

        if selected_indices.size > 0:
            x_here = x_here_all[selected_indices]
            y_here = y_here_all[selected_indices]
            t_here = times_minutes[selected_indices]

            ax_here.plot(
                x_here,
                y_here,
                t_here,
                linewidth=1.0,
                alpha=0.55,
            )

            scatter_here = ax_here.scatter(
                x_here,
                y_here,
                t_here,
                c=t_here,
                cmap="viridis",
                s=18,
                depthshade=False,
            )
            scatter_for_3d_cbar = scatter_here

            add_hourly_markers_3d(
                ax=ax_here,
                selected_indices=selected_indices,
                x_values=x_here_all,
                y_values=y_here_all,
                z_values=times_minutes,
                local_datetimes=local_datetimes,
                start_datetime=start_local_datetime,
                times_minutes=times_minutes,
            )

            hour_datetimes_3d, hour_elapsed_3d = make_hour_boundaries(
                start_local_datetime,
                times_minutes[selected_indices],
            )
            if hour_elapsed_3d.size > 0:
                ax_here.set_zticks(hour_elapsed_3d)
                ax_here.set_zticklabels(
                    [
                        (
                            "Midnight\n00:00"
                            if hour_datetime.hour == 0
                            else hour_datetime.strftime(
                                LOCAL_TIME_LABEL_FORMAT
                            )
                        )
                        + f"\n({elapsed_minutes:.0f} min)"
                        for hour_datetime, elapsed_minutes in zip(
                            hour_datetimes_3d,
                            hour_elapsed_3d,
                        )
                    ]
                )

        ax_here.set_xlim(*zoom_vector_x_limits)
        ax_here.set_ylim(*zoom_vector_y_limits)
        ax_here.set_box_aspect((1, 1, 1))
        ax_here.set_title(panel_title)
        ax_here.set_xlabel(
            "X slope (amplitude per 100 km; right positive)"
        )
        ax_here.set_ylabel(
            "Y slope (amplitude per 100 km; down positive)"
        )
        ax_here.set_zlabel("Elapsed time / local time")
        ax_here.grid(True, alpha=0.3)

        handles_here, _ = ax_here.get_legend_handles_labels()
        if handles_here:
            ax_here.legend()

    if scatter_for_3d_cbar is not None:
        combined_zoom_mask = (
            raw_tilt_plot_time_mask | preprocessed_tilt_plot_time_mask
        )
        cbar_3d = fig_tilt_3d.colorbar(
            scatter_for_3d_cbar,
            ax=[ax_raw_3d, ax_pre_3d],
            pad=0.05,
            shrink=0.72,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar_3d,
            start_local_datetime,
            times_minutes[combined_zoom_mask],
        )

    fig_tilt_3d.suptitle(
        "Plane-fit Tilt Vector vs Time: Raw vs Preprocessed\n"
        "Top = raw data; bottom = preprocessed; same X/Y limits",
        fontsize=14,
    )
    fig_tilt_3d.subplots_adjust(
        left=0.04,
        right=0.88,
        bottom=0.04,
        top=0.93,
        hspace=0.18,
    )

    tilt_vector_3d_file = out_dir / "plane_tilt_vector_3d_vs_time.png"
    fig_tilt_3d.savefig(
        tilt_vector_3d_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_tilt_3d)

    # ----------------------------------------------
    # 2-D whole-time and zoomed comparisons
    # ----------------------------------------------
    tilt_vector_2d_whole_file = (
        out_dir / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
    )

    tilt_vector_2d_zoom_file = (
        out_dir
        / f"plane_tilt_vector_2d_zoom_{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min_colored_by_time.png"
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_finite_vector,
        preprocessed_time_mask=preprocessed_finite_vector,
        output_file=tilt_vector_2d_whole_file,
        title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
        x_limits=whole_vector_x_limits,
        y_limits=whole_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
        figure_size=(18, 21),
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_tilt_plot_time_mask,
        preprocessed_time_mask=preprocessed_tilt_plot_time_mask,
        output_file=tilt_vector_2d_zoom_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: "
            f"{TILT_PLOT_START_TIME_MIN:.0f}–"
            f"{TILT_PLOT_END_TIME_MIN:.0f} Minutes"
        ),
        x_limits=zoom_vector_x_limits,
        y_limits=zoom_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
        figure_size=(14, 18),
    )

    # New: hourly-averaged zoomed tilt points
    hourly_average_rows = compute_hourly_average_rows(
        selected_indices=np.flatnonzero(preprocessed_finite_vector),
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        input_file=pkl_file,
        output_folder=out_dir,
        start_local_datetime=start_local_datetime,
    )

    save_hourly_average_csv(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_csv_file,
    )

    if hourly_average_rows:
        hourly_average_x = np.array(
            [
                float(row["mean_x_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )
        hourly_average_y = np.array(
            [
                float(row["mean_y_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )

        hourly_average_x_limits, hourly_average_y_limits = (
            calculate_vector_limits(
                hourly_average_x,
                hourly_average_y,
                VECTOR_PLOT_PADDING_FRACTION,
            )
        )
    else:
        hourly_average_x_limits = (-1.0, 1.0)
        hourly_average_y_limits = (1.0, -1.0)

    save_hourly_average_tilt_plot(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_plot_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: Hourly Averages\n"
            f"{HOURLY_AVERAGE_START_HOUR:02d}:00–"
            f"{HOURLY_AVERAGE_END_HOUR:02d}:00 Local Time"
        ),
        x_limits=hourly_average_x_limits,
        y_limits=hourly_average_y_limits,
        start_local_datetime=start_local_datetime,
        figure_size=ZOOM_TIME_TILT_FIGSIZE,
    )

    # ----------------------------------------------
    # Animation: raw vs preprocessed
    # ----------------------------------------------
    saved_animation_file = ""

    if MAKE_ANIMATION:
        all_slopes_for_scale = np.concatenate(
            [
                raw_slope_values[np.isfinite(raw_slope_values)],
                slope_values[np.isfinite(slope_values)],
            ]
        )

        if all_slopes_for_scale.size > 0:
            slope_ref = np.nanpercentile(
                all_slopes_for_scale,
                ARROW_LENGTH_PERCENTILE,
            )
        else:
            slope_ref = 1.0

        if not np.isfinite(slope_ref) or slope_ref <= 0:
            slope_ref = 1.0

        max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

        raw_arrow_u = np.zeros(nt, dtype=np.float64)
        raw_arrow_v = np.zeros(nt, dtype=np.float64)
        pre_arrow_u = np.zeros(nt, dtype=np.float64)
        pre_arrow_v = np.zeros(nt, dtype=np.float64)

        finite_raw_ab = np.isfinite(raw_a_values) & np.isfinite(raw_b_values)
        finite_pre_ab = np.isfinite(a_values) & np.isfinite(b_values)

        raw_arrow_u[finite_raw_ab] = (
            raw_a_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix
        raw_arrow_v[finite_raw_ab] = (
            raw_b_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix

        pre_arrow_u[finite_pre_ab] = (
            a_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix
        pre_arrow_v[finite_pre_ab] = (
            b_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix

        vmax = np.nanpercentile(np.abs(wave_cube), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax

        preprocessed_frame_0 = preprocess_frame_for_plane_fit(
            wave_cube[0],
            fit_radius_mask,
        )

        fig_anim, (ax_raw, ax_pre) = plt.subplots(
            nrows=1,
            ncols=2,
            figsize=(13, 6),
        )

        im_raw = ax_raw.imshow(
            wave_cube[0],
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )
        im_pre = ax_pre.imshow(
            preprocessed_frame_0,
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )

        cbar_raw = fig_anim.colorbar(
            im_raw,
            ax=ax_raw,
            fraction=0.046,
            pad=0.04,
        )
        cbar_raw.set_label("Perturbation amplitude")

        cbar_pre = fig_anim.colorbar(
            im_pre,
            ax=ax_pre,
            fraction=0.046,
            pad=0.04,
        )
        cbar_pre.set_label("Perturbation amplitude")

        center_x = (nx - 1) / 2.0
        center_y = (ny - 1) / 2.0

        for ax_here in (ax_raw, ax_pre):
            fit_outline = Ellipse(
                xy=(center_x, center_y),
                width=2.0 * FIT_RADIUS_KM / DX_KM,
                height=2.0 * FIT_RADIUS_KM / DY_KM,
                fill=False,
                edgecolor="black",
                linestyle="--",
                linewidth=1.2,
                alpha=0.8,
                zorder=4,
            )
            ax_here.add_patch(fit_outline)
            ax_here.scatter(
                [center_x],
                [center_y],
                s=CENTER_MARKER_SIZE,
                c=CENTER_MARKER_COLOR,
                marker="o",
            )
            ax_here.set_xlabel("X pixel")
            ax_here.set_ylabel("Y pixel")

        raw_quiver = ax_raw.quiver(
            [center_x],
            [center_y],
            [raw_arrow_u[0]],
            [raw_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        pre_quiver = ax_pre.quiver(
            [center_x],
            [center_y],
            [pre_arrow_u[0]],
            [pre_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        ax_raw.set_title("Raw data + raw slope vector")
        ax_pre.set_title("Preprocessed + preprocessed slope vector")

        title = fig_anim.suptitle("")

        raw_info = ax_raw.text(
            0.02,
            0.98,
            "",
            transform=ax_raw.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        pre_info = ax_pre.text(
            0.02,
            0.98,
            "",
            transform=ax_pre.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        def update(frame_index):
            raw_frame = wave_cube[frame_index]
            preprocessed_frame = preprocess_frame_for_plane_fit(
                raw_frame,
                fit_radius_mask,
            )

            im_raw.set_data(raw_frame)
            im_pre.set_data(preprocessed_frame)

            raw_quiver.set_UVC(
                [raw_arrow_u[frame_index]],
                [raw_arrow_v[frame_index]],
            )
            pre_quiver.set_UVC(
                [pre_arrow_u[frame_index]],
                [pre_arrow_v[frame_index]],
            )

            local_time_text = local_datetimes[frame_index].strftime(
                LOCAL_TIME_EXACT_FORMAT
            )
            title.set_text(
                f"Raw vs Preprocessed Plane-fit Vector | "
                f"Frame {frame_index}/{nt - 1} | "
                f"t = {times_minutes[frame_index]:.1f} min | "
                f"Local time = {local_time_text}"
            )

            raw_info.set_text(
                f"Direction = {raw_angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {raw_slope_values[frame_index]:.4g} amp/km\n"
                f"a = {raw_a_values[frame_index]:.4g}\n"
                f"b = {raw_b_values[frame_index]:.4g}"
            )
            pre_info.set_text(
                f"Direction = {angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {slope_values[frame_index]:.4g} amp/km\n"
                f"a = {a_values[frame_index]:.4g}\n"
                f"b = {b_values[frame_index]:.4g}"
            )

            return (
                im_raw,
                im_pre,
                raw_quiver,
                pre_quiver,
                title,
                raw_info,
                pre_info,
            )

        ani = FuncAnimation(
            fig_anim,
            update,
            frames=nt,
            interval=1000 / FPS,
            blit=False,
        )

        fig_anim.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))

        animation_mp4_file = out_dir / "plane_fit_vector_animation.mp4"
        animation_gif_file = out_dir / "plane_fit_vector_animation.gif"

        try:
            writer = FFMpegWriter(
                fps=FPS,
                bitrate=3000,
            )
            ani.save(
                animation_mp4_file,
                writer=writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_mp4_file)
        except Exception as exc:
            print(
                "Could not save MP4 with ffmpeg. "
                "Falling back to GIF."
            )
            print(f"Reason: {exc}")

            gif_writer = PillowWriter(fps=FPS)
            ani.save(
                animation_gif_file,
                writer=gif_writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_gif_file)

        plt.close(fig_anim)

    print(f"Finished: {pkl_file.name}")
    print(f"Successful fits: {successful_fits}/{nt}")
    print(f"Output folder: {out_dir}")

    return (
        {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "processed",
            "output_folder": str(out_dir),
            "frame_count": nt,
            "successful_fits": successful_fits,
            "hourly_average_csv": str(hourly_average_csv_file),
            "message": (
                f"Animation: {saved_animation_file}"
                if MAKE_ANIMATION
                else "Animation disabled"
            ),
        },
        hourly_average_rows,
    )


# ==================================================
# Batch driver
# ==================================================
def discover_pkl_files():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f"Input folder does not exist:\n{INPUT_ROOT}"
        )

    iterator = (
        INPUT_ROOT.rglob("*.pkl")
        if RECURSIVE_SEARCH
        else INPUT_ROOT.glob("*.pkl")
    )

    files = []
    output_root_resolved = OUTPUT_ROOT.resolve()

    for file_path in iterator:
        try:
            file_resolved = file_path.resolve()
            if (
                file_resolved == output_root_resolved
                or output_root_resolved in file_resolved.parents
            ):
                continue
        except OSError:
            pass

        files.append(file_path)

    return sorted(files)


def write_batch_summary(summary_rows):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary_file = OUTPUT_ROOT / "batch_processing_summary.csv"

    fieldnames = [
        "input_file",
        "internal_start_datetime",
        "status",
        "output_folder",
        "frame_count",
        "successful_fits",
        "hourly_average_csv",
        "message",
    ]

    with open(summary_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_file


def main():
    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Input root: {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Target years: {sorted(TARGET_YEARS)}")
    print(f"Animations enabled: {MAKE_ANIMATION}")
    print(
        "Preprocessing diagnostics enabled: "
        f"{MAKE_PREPROCESSING_DIAGNOSTICS}"
    )

    # ==================================================
    # Single-file mode
    # ==================================================
    if PROCESS_SINGLE_FILE:
        single_file = Path(
            SINGLE_PKL_FILE
        ).expanduser()

        print("\n" + "=" * 80)
        print("RUN MODE: SINGLE FILE")
        print(f"Selected PKL: {single_file}")

        if not single_file.exists():
            raise FileNotFoundError(
                "PROCESS_SINGLE_FILE is True, but the selected PKL "
                "does not exist:\n"
                f"{single_file}\n\n"
                "Change SINGLE_PKL_FILE near the top of the script."
            )

        summary_rows = []

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                single_file
            )

            summary_rows.append(
                summary_row
            )

        except Exception as exc:
            print(f"FAILED: {single_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(single_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            summary_file = write_batch_summary(
                summary_rows
            )
            print(
                f"Summary saved: {summary_file}"
            )

            if not CONTINUE_ON_ERROR:
                raise

            return

        finally:
            plt.close("all")
            gc.collect()

        summary_file = write_batch_summary(
            summary_rows
        )

        print("\n" + "=" * 80)
        print("Single-file processing finished.")
        print(f"Summary: {summary_file}")

        if (
            summary_rows
            and summary_rows[0]["status"] == "processed"
        ):
            print(
                "Output folder: "
                f"{summary_rows[0]['output_folder']}"
            )

        print(
            "\nWhen you are satisfied with this day's preprocessing and "
            "plane fit, set PROCESS_SINGLE_FILE = False to run the batch."
        )

        return

    # ==================================================
    # Batch mode
    # ==================================================
    print("\n" + "=" * 80)
    print("RUN MODE: BATCH")

    pkl_files = discover_pkl_files()

    print(
        f"PKL files found: {len(pkl_files)}"
    )

    if not pkl_files:
        print("No PKL files were found.")
        return

    summary_rows = []
    all_days_hourly_average_rows = []

    for (
        file_number,
        pkl_file,
    ) in enumerate(
        pkl_files,
        start=1,
    ):
        print(
            f"\nBatch file {file_number}/{len(pkl_files)}: "
            f"{pkl_file.name}"
        )

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                pkl_file
            )

            summary_rows.append(
                summary_row
            )

            all_days_hourly_average_rows.extend(
                hourly_rows
            )

        except Exception as exc:
            print(f"FAILED: {pkl_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(pkl_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            if not CONTINUE_ON_ERROR:
                summary_file = write_batch_summary(
                    summary_rows
                )
                print(
                    f"Partial summary saved: {summary_file}"
                )
                raise

        finally:
            plt.close("all")
            gc.collect()

        write_batch_summary(
            summary_rows
        )

    summary_file = write_batch_summary(
        summary_rows
    )

    all_days_hourly_average_csv_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )

    all_days_hourly_average_plot_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    save_all_days_hourly_average_csv(
        all_days_hourly_average_rows,
        all_days_hourly_average_csv_file,
    )

    save_all_days_hourly_average_plot(
        all_days_hourly_average_rows,
        all_days_hourly_average_plot_file,
    )

    processed_count = sum(
        row["status"] == "processed"
        for row in summary_rows
    )

    skipped_count = sum(
        str(row["status"]).startswith(
            "skipped"
        )
        for row in summary_rows
    )

    failed_count = sum(
        row["status"] == "failed"
        for row in summary_rows
    )

    print("\n" + "=" * 80)
    print("Batch finished.")
    print(f"Processed: {processed_count}")
    print(f"Skipped:   {skipped_count}")
    print(f"Failed:    {failed_count}")
    print(f"Summary:   {summary_file}")

    print(
        "All-days hourly-average CSV:  "
        f"{all_days_hourly_average_csv_file}"
    )

    print(
        "All-days hourly-average plot: "
        f"{all_days_hourly_average_plot_file}"
    )

    print(
        "Hourly-average local-hour window: "
        f"{HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )


if __name__ == "__main__":
    main()


Input root: /Users/anhphan/Downloads/pkl
Output root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs
Target years: [2024, 2025, 2026]
Animations enabled: True
Preprocessing diagnostics enabled: True

RUN MODE: SINGLE FILE
Selected PKL: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)
PKL interval start datetime: 2025-03-25 20:29:27
First frame-center datetime: 2025-03-25 20:30:27
Plane-fit region: radius <= 300.0 km (282792 pixels before other validity filters)
Spatial preprocessing: Gaussian-background peak replacement (sigma = 40 km, threshold = 2.5 MAD-sigma, mask dilation = 4 km)
Fitting planes to raw and preprocessed frames...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Proce

## Save cube, 2hr average

In [4]:
# -*- coding: utf-8 -*-
"""
Batch-process all BLO PKL files from 2024 and 2025.

The script recursively searches INPUT_ROOT for .pkl files. For each file, it:

1. Reads the perturbation cube from data[5][1].
2. Reads the local date/start time from data[7][1].
3. Processes the file only when the internal start year is in TARGET_YEARS.
4. Fits a 2-D plane to both the raw frame and a Gaussian-background-outlier-cleaned frame, using only pixels within FIT_RADIUS_KM of the image center.
5. Saves the final preprocessed perturbation cube as a memory-mappable .npy
   file, plus metadata needed for later keogram generation.
6. Calculates and saves a centered 2-hour running average of the raw and
   preprocessed slope vectors.
7. Saves CSV/NPZ results, angle/slope plots, 2-D/3-D tilt plots,
   and optionally an animation.
8. Writes each PKL's outputs into its own folder.
9. Creates one batch_processing_summary.csv file.

Time convention:
    data[7][1] is treated as the beginning of the first frame interval.
    Each fitted point is assigned to the center of its interval. Therefore,
    with DT_SECONDS = 120, frame 0 occurs one minute after the PKL start time.

Angle convention:
    0 degrees    = perturbation increases to the right
    +90 degrees  = perturbation increases downward
    -90 degrees  = perturbation increases upward
    +/-180       = perturbation increases to the left
"""

import csv
import gc
import pickle
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.patches import Ellipse
from scipy.ndimage import binary_dilation, gaussian_filter


# ==================================================
# Batch settings
# ==================================================
INPUT_ROOT = Path.home() / "Downloads" / "pkl"
OUTPUT_ROOT = INPUT_ROOT / "BLO_plane_fit_batch_outputs"

# ==================================================
# Run mode
# ==================================================
# True:
#     Process only SINGLE_PKL_FILE. Use this while tuning one night/day.
#
# False:
#     Discover and process all PKL files under INPUT_ROOT.
PROCESS_SINGLE_FILE = True

# Change this to the one PKL you want to examine first.
# An absolute path is also fine.
SINGLE_PKL_FILE = INPUT_ROOT / "processed_data_full_03_25_2025.pkl"

TARGET_YEARS = {2024, 2025, 2026}
RECURSIVE_SEARCH = True

# Animations can make the full batch substantially slower and use much more
# disk space. Set this to False when only the numerical results and plots
# are needed.
MAKE_ANIMATION = True

# If plane_fit_results.npz already exists in a PKL's output folder, skip it.
# Keep this False after changing the fit radius/filter settings so old results
# are recalculated with the new method.
SKIP_IF_RESULTS_EXIST = False

# Continue processing the remaining PKLs after one file fails.
CONTINUE_ON_ERROR = True


# ==================================================
# Analysis settings
# ==================================================
DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

# Save the FINAL PREPROCESSED perturbation cube so it can be used later for
# keograms without re-running the preprocessing.  The cube is written one
# frame at a time using a disk-backed .npy memmap, so a second full cube does
# not have to be held in RAM.
SAVE_PREPROCESSED_CUBE = True
PREPROCESSED_CUBE_DTYPE = np.float32

# Running-average settings for the fitted slope vector.
# The default is a centered 2-hour window: each point averages samples within
# +/- 1 hour of that frame-center time. Near the beginning/end of a night, the
# available portion of the window is used and the number of contributing
# samples is saved so those edge points can be identified later.
RUNNING_AVERAGE_HOURS = 2.0

LOCAL_TIME_LABEL_FORMAT = "%H:%M"
LOCAL_TIME_EXACT_FORMAT = "%H:%M:%S"
LOCAL_DATETIME_PRINT_FORMAT = "%Y-%m-%d %H:%M:%S"

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Only pixels inside this radius, measured from the image center, are used
# in the plane fit.
FIT_RADIUS_KM = 300.0

# Preprocessing before fitting the plane.
#
# Keep only one cleaning operation:
#   1. Build a broad Gaussian background.
#   2. Compute high_pass = original - Gaussian background.
#   3. Estimate high-pass scatter with a robust MAD sigma.
#   4. Flag unusually sharp pixels.
#   5. Optionally dilate the mask to include a small halo.
#   6. Replace only flagged pixels with the Gaussian-background value.
#
# Unflagged pixels remain exactly unchanged.
APPLY_STAR_RESIDUAL_FILTER = True

BACKGROUND_GAUSSIAN_SIGMA_KM = 40.0
BACKGROUND_GAUSSIAN_TRUNCATE = 3.0
STAR_RESIDUAL_MAD_SIGMA = 2.5
STAR_MASK_DILATION_KM = 4.0

ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

# Diagnostic comparison arrows use the same scale for raw and preprocessed
# within each selected frame.
DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE = 0.28

# Removed component shown in the animation:
#     removed component = original frame - preprocessed frame
# This shows what the Gaussian-background outlier replacement removed.
REMOVED_COMPONENT_CMAP = "RdBu_r"
REMOVED_COMPONENT_SCALE_PERCENTILE = 99.0
REMOVED_COMPONENT_SCALE_SAMPLE_FRAMES = 12

# Optional static diagnostic figures for tuning one day.
# Each selected frame compares the original image, Gaussian background,
# high-pass residual, detected mask, removed component, final preprocessed,
# and slope vectors before/after preprocessing.
MAKE_PREPROCESSING_DIAGNOSTICS = True

# Explicit frame numbers to inspect.
DIAGNOSTIC_FRAME_INDICES = [0, 30, 60, 90, 120, 150, 180, 210]

# Also inspect the frames nearest these elapsed times in minutes.
DIAGNOSTIC_TIMES_MIN = [30.0, 90.0, 150.0, 210.0, 270.0, 330.0]

TILT_VECTOR_PLOT_STEP = 10
VECTOR_PLOT_PADDING_FRACTION = 0.12

TILT_PLOT_START_TIME_MIN = 90.0
TILT_PLOT_END_TIME_MIN = 500.0

HOURLY_MARKER_SIZE = 55
MIDNIGHT_MARKER_SIZE = 130
HOURLY_LABEL_OFFSET_POINTS = (6, 6)

# On the whole-time 2-D plot, every hour is marked, but fewer labels are drawn.
WHOLE_TIME_LABEL_EVERY_N_HOURS = 8
WHOLE_TIME_COLORBAR_EVERY_N_HOURS = 2

# Separate sizes for the whole-range and zoomed 2-D tilt plots.
WHOLE_TIME_TILT_FIGSIZE = (18, 9)
ZOOM_TIME_TILT_FIGSIZE = (14, 7.5)
VECTOR_3D_FIGSIZE = (13, 7.5)

# Thin colorbars for the vector plots.
VECTOR_COLORBAR_FRACTION = 0.022
VECTOR_COLORBAR_PAD = 0.025
VECTOR_COLORBAR_ASPECT = 35

# For the hourly-average tilt plots, only include local clock hours from
# 21:00 through 03:00 (inclusive), i.e. 9 PM to 3 AM.
HOURLY_AVERAGE_START_HOUR = 21
HOURLY_AVERAGE_END_HOUR = 3


# ==================================================
# Time helpers
# ==================================================
def parse_pkl_start_datetime(date_start):
    """Parse data[7][1] = [date, start_time] as a local datetime."""
    if (
        not isinstance(date_start, (list, tuple, np.ndarray))
        or len(date_start) < 2
    ):
        raise ValueError(
            "Expected data[7][1] to contain [date, start_time], "
            f"but got: {date_start!r}"
        )

    combined_text = f"{str(date_start[0]).strip()} {str(date_start[1]).strip()}"

    accepted_formats = (
        "%m_%d_%Y %H:%M:%S",
        "%m_%d_%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in accepted_formats:
        try:
            return datetime.strptime(combined_text, date_format)
        except ValueError:
            pass

    raise ValueError(
        "Could not parse the PKL date/start time: "
        f"{date_start!r}. Add its format to accepted_formats."
    )


def local_datetime_from_elapsed_minutes(start_datetime, elapsed_minutes):
    return start_datetime + timedelta(minutes=float(elapsed_minutes))


def make_hour_boundaries(start_datetime, time_values):
    """Return exact whole local hours within the elapsed-time range."""
    time_values = np.asarray(time_values, dtype=np.float64)
    finite_values = time_values[np.isfinite(time_values)]

    if finite_values.size == 0:
        return np.array([], dtype=object), np.array([], dtype=np.float64)

    minimum_elapsed = float(np.nanmin(finite_values))
    maximum_elapsed = float(np.nanmax(finite_values))

    minimum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, minimum_elapsed
    )
    maximum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, maximum_elapsed
    )

    first_hour = minimum_datetime.replace(
        minute=0, second=0, microsecond=0
    )
    if first_hour < minimum_datetime:
        first_hour += timedelta(hours=1)

    hour_datetimes = []
    current_hour = first_hour
    while current_hour <= maximum_datetime:
        hour_datetimes.append(current_hour)
        current_hour += timedelta(hours=1)

    elapsed_hour_minutes = np.array(
        [
            (hour_datetime - start_datetime).total_seconds() / 60.0
            for hour_datetime in hour_datetimes
        ],
        dtype=np.float64,
    )

    return np.array(hour_datetimes, dtype=object), elapsed_hour_minutes


def format_hour_tick(start_datetime, elapsed_minutes):
    local_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, elapsed_minutes
    )
    if local_datetime.hour == 0:
        local_label = "Midnight\n00:00"
    else:
        local_label = local_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)

    return f"{elapsed_minutes:.0f} min\n{local_label}"


def apply_hourly_elapsed_local_labels_to_colorbar(
    colorbar,
    start_datetime,
    time_values,
    label_every_n_hours=1,
):
    hour_datetimes, hour_elapsed_minutes = make_hour_boundaries(
        start_datetime, time_values
    )

    if hour_elapsed_minutes.size == 0:
        finite_values = np.asarray(time_values, dtype=np.float64)
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            return

        hour_elapsed_minutes = np.unique(
            np.array(
                [
                    float(np.nanmin(finite_values)),
                    float(np.nanmax(finite_values)),
                ]
            )
        )
        hour_datetimes = np.array(
            [
                local_datetime_from_elapsed_minutes(start_datetime, value)
                for value in hour_elapsed_minutes
            ],
            dtype=object,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    keep = np.array(
        [
            (hour_datetime.hour == 0)
            or (hour_datetime.hour % label_every_n_hours == 0)
            for hour_datetime in hour_datetimes
        ],
        dtype=bool,
    )

    if np.any(keep):
        hour_elapsed_minutes = hour_elapsed_minutes[keep]

    colorbar.set_ticks(hour_elapsed_minutes)
    colorbar.set_ticklabels(
        [
            format_hour_tick(start_datetime, elapsed_minutes)
            for elapsed_minutes in hour_elapsed_minutes
        ]
    )
    colorbar.set_label("Elapsed time / local time")


def find_nearest_frame_indices_for_hours(
    selected_indices,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    target_hour_datetimes, target_hour_minutes = make_hour_boundaries(
        start_datetime, times_minutes[selected_indices]
    )

    if target_hour_minutes.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    marker_indices = []

    for target_datetime in target_hour_datetimes:
        differences = np.array(
            [
                abs(
                    (
                        local_datetimes[frame_index] - target_datetime
                    ).total_seconds()
                )
                for frame_index in selected_indices
            ],
            dtype=np.float64,
        )
        marker_indices.append(selected_indices[int(np.argmin(differences))])

    marker_indices = np.asarray(marker_indices, dtype=int)

    keep = np.concatenate(
        [
            np.array([True]),
            marker_indices[1:] != marker_indices[:-1],
        ]
    )

    return marker_indices[keep], target_hour_datetimes[keep]


def add_hourly_markers_2d(
    ax,
    selected_indices,
    x_values,
    y_values,
    local_datetimes,
    start_datetime,
    times_minutes,
    label_every_n_hours=1,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.2,
            label="1-hour marks",
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            zorder=8,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        is_midnight = target_datetime.hour == 0
        should_label = (
            is_midnight
            or target_datetime.hour % label_every_n_hours == 0
        )

        if not should_label:
            continue

        label_text = (
            "Midnight\n00:00"
            if is_midnight
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.annotate(
            label_text,
            xy=(x_values[frame_index], y_values[frame_index]),
            xytext=HOURLY_LABEL_OFFSET_POINTS,
            textcoords="offset points",
            fontsize=8,
            fontweight="bold" if is_midnight else "normal",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=9,
        )


def add_hourly_markers_3d(
    ax,
    selected_indices,
    x_values,
    y_values,
    z_values,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            z_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.0,
            label="1-hour marks",
            depthshade=False,
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            z_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            depthshade=False,
            zorder=8,
        )

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        label_text = (
            "Midnight\n00:00"
            if target_datetime.hour == 0
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.text(
            x_values[frame_index],
            y_values[frame_index],
            z_values[frame_index],
            f"  {label_text}",
            fontsize=8,
            zorder=9,
        )


# ==================================================
# Data and fitting helpers
# ==================================================
def load_wave_cube(pkl_file):
    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
        date_start = data[7][1]
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "The PKL does not contain the expected data[5][1] perturbation "
            "cube and data[7][1] date/start time."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected perturbation data with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    return label, wave_cube, date_start


def robust_sigma_from_mad(values):
    """
    Estimate a Gaussian-equivalent sigma from the median absolute deviation.
    Returns NaN when the estimate cannot be formed.
    """
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))

    if not np.isfinite(mad) or mad <= 0:
        return np.nan

    return 1.4826 * mad


def calculate_centered_running_average(values, times_minutes, window_hours):
    """
    Calculate a centered time-based running average.

    For a 2-hour window, the value at time t uses all valid samples from
    t - 1 hour through t + 1 hour. At the beginning/end of the time series,
    the available part of the window is used.

    Returns
    -------
    running_mean : ndarray
        Running average at each input time.
    valid_counts : ndarray of int
        Number of finite samples contributing to each running average.
    """
    values = np.asarray(values, dtype=np.float64)
    times_minutes = np.asarray(times_minutes, dtype=np.float64)

    running_mean = np.full(values.shape, np.nan, dtype=np.float64)
    valid_counts = np.zeros(values.shape, dtype=int)

    half_window_minutes = 0.5 * float(window_hours) * 60.0

    finite_time = np.isfinite(times_minutes)
    finite_values = np.isfinite(values)

    for index, time_here in enumerate(times_minutes):
        if not np.isfinite(time_here):
            continue

        in_window = (
            finite_time
            & finite_values
            & (np.abs(times_minutes - time_here) <= half_window_minutes)
        )

        count = int(np.count_nonzero(in_window))
        valid_counts[index] = count

        if count > 0:
            running_mean[index] = float(np.mean(values[in_window]))

    return running_mean, valid_counts


def calculate_centered_running_average_vector(
    x_values,
    y_values,
    times_minutes,
    window_hours,
):
    """
    Centered running average of a 2-D slope vector.

    X and Y are averaged using exactly the same set of finite samples. The
    magnitude and direction are then calculated FROM the averaged vector.
    """
    x_values = np.asarray(x_values, dtype=np.float64)
    y_values = np.asarray(y_values, dtype=np.float64)
    times_minutes = np.asarray(times_minutes, dtype=np.float64)

    mean_x = np.full(x_values.shape, np.nan, dtype=np.float64)
    mean_y = np.full(y_values.shape, np.nan, dtype=np.float64)
    valid_counts = np.zeros(x_values.shape, dtype=int)

    half_window_minutes = 0.5 * float(window_hours) * 60.0

    finite_time = np.isfinite(times_minutes)
    finite_vector = np.isfinite(x_values) & np.isfinite(y_values)

    for index, time_here in enumerate(times_minutes):
        if not np.isfinite(time_here):
            continue

        in_window = (
            finite_time
            & finite_vector
            & (np.abs(times_minutes - time_here) <= half_window_minutes)
        )

        count = int(np.count_nonzero(in_window))
        valid_counts[index] = count

        if count > 0:
            mean_x[index] = float(np.mean(x_values[in_window]))
            mean_y[index] = float(np.mean(y_values[in_window]))

    magnitude = np.hypot(mean_x, mean_y)
    direction_deg = (
        (np.degrees(np.arctan2(mean_y, mean_x)) + 180.0) % 360.0
    ) - 180.0

    invalid = ~np.isfinite(mean_x) | ~np.isfinite(mean_y)
    magnitude[invalid] = np.nan
    direction_deg[invalid] = np.nan

    return mean_x, mean_y, magnitude, direction_deg, valid_counts


def save_running_average_csv(
    output_file,
    frame_numbers,
    times_seconds,
    times_minutes,
    local_datetime_iso,
    local_datetimes,
    raw_mean_x,
    raw_mean_y,
    raw_vector_magnitude,
    raw_vector_direction_deg,
    raw_mean_scalar_magnitude,
    raw_counts,
    pre_mean_x,
    pre_mean_y,
    pre_vector_magnitude,
    pre_vector_direction_deg,
    pre_mean_scalar_magnitude,
    pre_counts,
):
    """Save the running-average slope products in a standalone CSV."""
    with open(output_file, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "running_average_window_hours",
                "raw_running_mean_x_slope_amplitude_per_km",
                "raw_running_mean_y_slope_amplitude_per_km",
                "raw_running_mean_vector_magnitude_amplitude_per_km",
                "raw_running_mean_vector_direction_deg",
                "raw_running_mean_of_scalar_slope_magnitude_amplitude_per_km",
                "raw_running_average_valid_sample_count",
                "preprocessed_running_mean_x_slope_amplitude_per_km",
                "preprocessed_running_mean_y_slope_amplitude_per_km",
                "preprocessed_running_mean_vector_magnitude_amplitude_per_km",
                "preprocessed_running_mean_vector_direction_deg",
                "preprocessed_running_mean_of_scalar_slope_magnitude_amplitude_per_km",
                "preprocessed_running_average_valid_sample_count",
            ]
        )

        for frame_index in range(len(frame_numbers)):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(LOCAL_TIME_EXACT_FORMAT),
                    RUNNING_AVERAGE_HOURS,
                    raw_mean_x[frame_index],
                    raw_mean_y[frame_index],
                    raw_vector_magnitude[frame_index],
                    raw_vector_direction_deg[frame_index],
                    raw_mean_scalar_magnitude[frame_index],
                    raw_counts[frame_index],
                    pre_mean_x[frame_index],
                    pre_mean_y[frame_index],
                    pre_vector_magnitude[frame_index],
                    pre_vector_direction_deg[frame_index],
                    pre_mean_scalar_magnitude[frame_index],
                    pre_counts[frame_index],
                ]
            )


def save_running_average_timeseries_plot(
    output_file,
    times_minutes,
    raw_mean_x,
    raw_mean_y,
    raw_vector_magnitude,
    raw_vector_direction_deg,
    pre_mean_x,
    pre_mean_y,
    pre_vector_magnitude,
    pre_vector_direction_deg,
):
    """
    Save the original component-vs-time diagnostic for the centered running
    average. This is retained as a secondary diagnostic.
    """
    scale = 100.0

    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(12, 12),
        sharex=True,
    )

    panels = [
        (
            axes[0],
            scale * raw_mean_x,
            scale * pre_mean_x,
            "X slope (amplitude per 100 km)",
        ),
        (
            axes[1],
            scale * raw_mean_y,
            scale * pre_mean_y,
            "Y slope (amplitude per 100 km)",
        ),
        (
            axes[2],
            scale * raw_vector_magnitude,
            scale * pre_vector_magnitude,
            "Magnitude of averaged vector (amplitude per 100 km)",
        ),
        (
            axes[3],
            raw_vector_direction_deg,
            pre_vector_direction_deg,
            "Direction of averaged vector (deg)",
        ),
    ]

    for ax, raw_values, pre_values, ylabel in panels:
        ax.plot(
            times_minutes,
            raw_values,
            linewidth=1.0,
            alpha=0.75,
            label="Raw",
        )
        ax.plot(
            times_minutes,
            pre_values,
            linewidth=1.4,
            label="Preprocessed",
        )
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)

    axes[0].legend()
    axes[3].set_ylim(-180.0, 180.0)
    axes[3].set_yticks(np.arange(-180, 181, 45))
    axes[3].set_xlabel("Elapsed time (minutes; frame-center time)")

    fig.suptitle(
        f"Centered {RUNNING_AVERAGE_HOURS:g}-Hour Running Average "
        "of Plane-Fit Slope Vector\n"
        "Magnitude/direction are calculated from the averaged X/Y vector"
    )
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.96))
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_running_average_vector_plot(
    output_file,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    processed_x,
    processed_y,
    running_mean_x,
    running_mean_y,
    time_min=None,
    time_max=None,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
    figure_size=(14, 12),
):
    """
    Plot the centered running-average slope vector in a two-panel layout:

        Top:    preprocessed (processed) slope-vector endpoints
        Bottom: centered running-average slope-vector endpoints

    This matches the ordinary slope-vector style, but the comparison is now
    between the processed data and its 2-hour running average.
    """
    processed_x_per_100km = 100.0 * np.asarray(processed_x, dtype=np.float64)
    processed_y_per_100km = 100.0 * np.asarray(processed_y, dtype=np.float64)

    running_x_per_100km = 100.0 * np.asarray(running_mean_x, dtype=np.float64)
    running_y_per_100km = 100.0 * np.asarray(running_mean_y, dtype=np.float64)

    times_minutes = np.asarray(times_minutes, dtype=np.float64)

    processed_mask = (
        np.isfinite(processed_x_per_100km)
        & np.isfinite(processed_y_per_100km)
        & np.isfinite(times_minutes)
    )
    running_mask = (
        np.isfinite(running_x_per_100km)
        & np.isfinite(running_y_per_100km)
        & np.isfinite(times_minutes)
    )

    if time_min is not None:
        processed_mask &= times_minutes >= float(time_min)
        running_mask &= times_minutes >= float(time_min)

    if time_max is not None:
        processed_mask &= times_minutes <= float(time_max)
        running_mask &= times_minutes <= float(time_max)

    combined_x_parts = []
    combined_y_parts = []

    if np.any(processed_mask):
        combined_x_parts.append(processed_x_per_100km[processed_mask])
        combined_y_parts.append(processed_y_per_100km[processed_mask])

    if np.any(running_mask):
        combined_x_parts.append(running_x_per_100km[running_mask])
        combined_y_parts.append(running_y_per_100km[running_mask])

    if combined_x_parts:
        combined_x = np.concatenate(combined_x_parts)
        combined_y = np.concatenate(combined_y_parts)
        x_limits, y_limits = calculate_vector_limits(
            combined_x,
            combined_y,
            VECTOR_PLOT_PADDING_FRACTION,
        )
    else:
        x_limits = (-1.0, 1.0)
        y_limits = (1.0, -1.0)

    if time_min is None and time_max is None:
        time_description = "Whole Time Range"
    elif time_min is not None and time_max is not None:
        time_description = f"{float(time_min):.0f}–{float(time_max):.0f} Minutes"
    elif time_min is not None:
        time_description = f"From {float(time_min):.0f} Minutes"
    else:
        time_description = f"Through {float(time_max):.0f} Minutes"

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=figure_size,
    )

    scatter_processed = _draw_tilt_vector_2d_panel(
        ax=axes[0],
        time_mask=processed_mask,
        x_slope_per_100km=processed_x_per_100km,
        y_slope_per_100km=processed_y_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Processed data",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    scatter_running = _draw_tilt_vector_2d_panel(
        ax=axes[1],
        time_mask=running_mask,
        x_slope_per_100km=running_x_per_100km,
        y_slope_per_100km=running_y_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title=(
            f"Centered {RUNNING_AVERAGE_HOURS:g}-hour average"
        ),
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    scatter_for_cbar = (
        scatter_running if scatter_running is not None else scatter_processed
    )

    if scatter_for_cbar is not None:
        combined_time_mask = processed_mask | running_mask
        cbar = fig.colorbar(
            scatter_for_cbar,
            ax=axes,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            times_minutes[combined_time_mask],
            label_every_n_hours=colorbar_every_n_hours,
        )

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    fig.suptitle(
        f"Processed vs {RUNNING_AVERAGE_HOURS:g}-Hour Running-Average Tilt Vector Endpoints: {time_description}\n"
        f"{subtitle}",
        fontsize=14,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.055,
        top=0.93,
        hspace=0.28,
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

def preprocess_frame_for_plane_fit(frame, fit_radius_mask, return_steps=False):
    """
    Remove compact sharp residual peaks using a Gaussian-background detector.

    Only detected outlier pixels are changed. The rest of the image is left
    exactly as it was in the raw frame.
    """
    frame = np.asarray(frame, dtype=np.float64)
    finite = np.isfinite(frame)

    steps = {
        "original": frame.copy(),
        "gaussian_background": np.full_like(frame, np.nan, dtype=np.float64),
        "high_pass": np.full_like(frame, np.nan, dtype=np.float64),
        "star_mask": np.zeros_like(frame, dtype=bool),
        "removed": np.zeros_like(frame, dtype=np.float64),
        "preprocessed": frame.copy(),
    }

    if not APPLY_STAR_RESIDUAL_FILTER or BACKGROUND_GAUSSIAN_SIGMA_KM <= 0:
        return (frame.copy(), steps) if return_steps else frame.copy()

    statistics_mask = finite & fit_radius_mask
    if np.count_nonzero(statistics_mask) < MIN_VALID_PIXELS:
        return (frame.copy(), steps) if return_steps else frame.copy()

    sigma_y_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DY_KM
    sigma_x_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DX_KM

    # Normalized Gaussian filtering prevents NaNs/missing pixels from
    # artificially depressing the local background.
    filled_values = np.where(finite, frame, 0.0)
    finite_weights = finite.astype(np.float64)

    blurred_values = gaussian_filter(
        filled_values,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )
    blurred_weights = gaussian_filter(
        finite_weights,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )

    gaussian_background = np.full_like(frame, np.nan, dtype=np.float64)
    good_weight = blurred_weights > 1.0e-8
    gaussian_background[good_weight] = (
        blurred_values[good_weight] / blurred_weights[good_weight]
    )

    high_pass = frame - gaussian_background
    hp_values = high_pass[statistics_mask]
    hp_center = float(np.median(hp_values))
    hp_sigma = robust_sigma_from_mad(hp_values)

    star_mask = np.zeros_like(frame, dtype=bool)

    if np.isfinite(hp_sigma) and hp_sigma > 0:
        star_mask = (
            finite
            & np.isfinite(gaussian_background)
            & (
                np.abs(high_pass - hp_center)
                > STAR_RESIDUAL_MAD_SIGMA * hp_sigma
            )
        )

        if STAR_MASK_DILATION_KM is not None and STAR_MASK_DILATION_KM > 0:
            dilation_pixels = max(
                int(round(STAR_MASK_DILATION_KM / min(DX_KM, DY_KM))),
                1,
            )
            star_mask = binary_dilation(
                star_mask,
                iterations=dilation_pixels,
            ) & finite

    preprocessed = frame.copy()
    preprocessed[star_mask] = gaussian_background[star_mask]
    removed = frame - preprocessed

    steps["gaussian_background"] = gaussian_background
    steps["high_pass"] = high_pass
    steps["star_mask"] = star_mask
    steps["removed"] = removed
    steps["preprocessed"] = preprocessed.copy()

    if return_steps:
        return preprocessed, steps

    return preprocessed


def solve_plane_least_squares(x_values, y_values, z_values):
    """Return plane coefficients a, b, c for z = a*x + b*y + c."""
    design_matrix = np.column_stack(
        [
            x_values,
            y_values,
            np.ones(x_values.size, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_values,
        rcond=None,
    )

    return coefficients, design_matrix



def fit_plane_to_image(
    frame_for_fit,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
):
    """
    Fit z = a*x + b*y + c directly to the supplied image.

    No preprocessing or iterative robust rejection is performed here.
    This makes it possible to use exactly the same fitting method for the
    raw and preprocessed images.
    """
    frame_sample = frame_for_fit[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    radius_sample = fit_radius_mask[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        radius_sample
        & np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN
    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))
    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    coefficients, design_matrix = solve_plane_least_squares(
        x_valid,
        y_valid,
        z_valid,
    )

    a, b, c = coefficients
    residuals = z_valid - design_matrix @ coefficients
    rmse = np.sqrt(np.mean(residuals**2))
    slope = np.hypot(a, b)
    angle_deg = (
        (np.degrees(np.arctan2(b, a)) + 180.0) % 360.0
    ) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


def calculate_vector_limits(x_values, y_values, padding_fraction):
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_max + y_padding, y_min - y_padding),
    )


def safe_relative_output_name(pkl_file):
    """Create a unique folder name from the PKL's path relative to INPUT_ROOT."""
    try:
        relative_path = pkl_file.relative_to(INPUT_ROOT).with_suffix("")
        parts = relative_path.parts
    except ValueError:
        parts = (pkl_file.stem,)

    return "__".join(parts)


# ==================================================
# Plot helper
# ==================================================

# ==================================================
# Plot helpers
# ==================================================
def _draw_tilt_vector_2d_panel(
    ax,
    time_mask,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    panel_title,
    x_limits,
    y_limits,
    label_every_n_hours=1,
):
    selected_all = np.flatnonzero(time_mask)
    scatter = None

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(u_plot, v_plot, linewidth=0.8, alpha=0.30)

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]
        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        add_hourly_markers_2d(
            ax=ax,
            selected_indices=selected_all,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
            label_every_n_hours=label_every_n_hours,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(panel_title)
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    return scatter


def save_tilt_vector_2d_comparison_plot(
    raw_time_mask,
    preprocessed_time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    raw_x_slope_per_100km,
    raw_y_slope_per_100km,
    preprocessed_x_slope_per_100km,
    preprocessed_y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
    figure_size=(14, 18),
):
    """
    Save a three-panel tilt-vector comparison:

        Top:    Raw
        Middle: Preprocessed
        Bottom: Difference = Preprocessed - Raw

    Raw and preprocessed use identical X/Y limits.

    The difference panel uses its own tighter symmetric X/Y limits so that
    small preprocessing-induced changes are visible.
    """
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=figure_size,
    )

    scatter_raw = _draw_tilt_vector_2d_panel(
        ax=axes[0],
        time_mask=raw_time_mask,
        x_slope_per_100km=raw_x_slope_per_100km,
        y_slope_per_100km=raw_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Raw data",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    scatter_pre = _draw_tilt_vector_2d_panel(
        ax=axes[1],
        time_mask=preprocessed_time_mask,
        x_slope_per_100km=preprocessed_x_slope_per_100km,
        y_slope_per_100km=preprocessed_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Preprocessed",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    # --------------------------------------------------
    # Difference = preprocessed - raw
    # --------------------------------------------------
    difference_x = (
        preprocessed_x_slope_per_100km
        - raw_x_slope_per_100km
    )
    difference_y = (
        preprocessed_y_slope_per_100km
        - raw_y_slope_per_100km
    )

    difference_time_mask = (
        raw_time_mask
        & preprocessed_time_mask
        & np.isfinite(difference_x)
        & np.isfinite(difference_y)
    )

    difference_indices = np.flatnonzero(difference_time_mask)

    # Use one symmetric scale for both X and Y so direction/magnitude in the
    # difference panel are not visually distorted.
    if difference_indices.size > 0:
        difference_abs_values = np.concatenate(
            [
                np.abs(difference_x[difference_indices]),
                np.abs(difference_y[difference_indices]),
            ]
        )

        difference_limit = np.nanpercentile(
            difference_abs_values,
            99.0,
        )

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = np.nanmax(difference_abs_values)

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = 1.0e-6

        difference_limit *= 1.15
    else:
        difference_limit = 1.0e-6

    # Y is plotted with the same image-coordinate convention as the other
    # tilt-vector panels: positive Y points downward.
    difference_x_limits = (
        -difference_limit,
        +difference_limit,
    )
    difference_y_limits = (
        +difference_limit,
        -difference_limit,
    )

    scatter_difference = _draw_tilt_vector_2d_panel(
        ax=axes[2],
        time_mask=difference_time_mask,
        x_slope_per_100km=difference_x,
        y_slope_per_100km=difference_y,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title=(
            "Difference: Preprocessed - Raw\n"
            r"$\Delta x = x_{\mathrm{pre}} - x_{\mathrm{raw}},\quad "
            r"\Delta y = y_{\mathrm{pre}} - y_{\mathrm{raw}}$"
        ),
        x_limits=difference_x_limits,
        y_limits=difference_y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    axes[2].set_xlabel(
        "Difference in X slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )
    axes[2].set_ylabel(
        "Difference in Y slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )

    # Add summary statistics directly to the difference panel.
    if difference_indices.size > 0:
        median_dx = np.nanmedian(
            np.abs(difference_x[difference_indices])
        )
        median_dy = np.nanmedian(
            np.abs(difference_y[difference_indices])
        )
        max_dx = np.nanmax(
            np.abs(difference_x[difference_indices])
        )
        max_dy = np.nanmax(
            np.abs(difference_y[difference_indices])
        )

        axes[2].text(
            0.02,
            0.98,
            (
                f"median |Δx| = {median_dx:.6f}\n"
                f"median |Δy| = {median_dy:.6f}\n"
                f"max |Δx| = {max_dx:.6f}\n"
                f"max |Δy| = {max_dy:.6f}"
            ),
            transform=axes[2].transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=10,
        )

    # Use one shared time colorbar for all three panels.
    scatter_for_cbar = (
        scatter_difference
        if scatter_difference is not None
        else (
            scatter_pre
            if scatter_pre is not None
            else scatter_raw
        )
    )

    if scatter_for_cbar is not None:
        cbar = fig.colorbar(
            scatter_for_cbar,
            ax=axes,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )

        combined_time_mask = (
            raw_time_mask
            | preprocessed_time_mask
        )

        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            times_minutes[combined_time_mask],
            label_every_n_hours=colorbar_every_n_hours,
        )

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    fig.suptitle(
        f"{title_text}\n"
        f"{subtitle}\n"
        "Raw and preprocessed use the same scale; "
        "difference uses a tighter symmetric scale",
        fontsize=14,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.055,
        top=0.92,
        hspace=0.32,
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def local_hour_is_in_hourly_average_window(local_hour):
    """
    Return True when the local clock hour falls inside the requested nightly
    window. This supports windows that cross midnight, such as 21:00 to 03:00.
    """
    start_hour = int(HOURLY_AVERAGE_START_HOUR) % 24
    end_hour = int(HOURLY_AVERAGE_END_HOUR) % 24
    local_hour = int(local_hour) % 24

    if start_hour <= end_hour:
        return start_hour <= local_hour <= end_hour

    return (local_hour >= start_hour) or (local_hour <= end_hour)


def compute_hourly_average_rows(
    selected_indices,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    input_file,
    output_folder,
    start_local_datetime,
):
    """
    Group the selected frame-center points by local clock hour and average
    the tilt-vector endpoint for each hour.
    """
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return []

    hour_groups = {}

    for frame_index in selected_indices:
        if not local_hour_is_in_hourly_average_window(
            local_datetimes[frame_index].hour
        ):
            continue

        hour_start = local_datetimes[frame_index].replace(
            minute=0,
            second=0,
            microsecond=0,
        )
        hour_groups.setdefault(hour_start, []).append(int(frame_index))

    rows = []

    night_label = start_local_datetime.strftime("%Y-%m-%d")

    for hour_start in sorted(hour_groups):
        frame_indices = np.asarray(hour_groups[hour_start], dtype=int)

        mean_x = float(np.nanmean(x_slope_per_100km[frame_indices]))
        mean_y = float(np.nanmean(y_slope_per_100km[frame_indices]))
        mean_elapsed_minutes = float(np.nanmean(times_minutes[frame_indices]))

        rows.append(
            {
                "input_file": str(input_file),
                "output_folder": str(output_folder),
                "start_local_datetime": start_local_datetime.isoformat(sep=" "),
                "night_label": night_label,
                "local_date": hour_start.strftime("%Y-%m-%d"),
                "hour_start_local_datetime": hour_start.isoformat(sep=" "),
                "hour_label": hour_start.strftime(LOCAL_TIME_LABEL_FORMAT),
                "is_midnight": int(hour_start.hour == 0),
                "mean_elapsed_minutes": mean_elapsed_minutes,
                "mean_x_slope_per_100km": mean_x,
                "mean_y_slope_per_100km": mean_y,
                "mean_slope_magnitude_per_100km": float(np.hypot(mean_x, mean_y)),
                "frame_count_in_average": int(frame_indices.size),
                "first_frame": int(frame_indices[0]),
                "last_frame": int(frame_indices[-1]),
            }
        )

    return rows


def save_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def load_hourly_average_csv(input_file):
    if not Path(input_file).exists():
        return []

    with open(input_file, "r", newline="") as file:
        return list(csv.DictReader(file))


def save_hourly_average_tilt_plot(
    hourly_rows,
    output_file,
    title_text,
    x_limits,
    y_limits,
    start_local_datetime,
    figure_size=(9, 8),
):
    """
    Save a 2-D tilt plot using one averaged point per selected local hour.
    """
    fig, ax = plt.subplots(figsize=figure_size)

    if hourly_rows:
        mean_x = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_y = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_elapsed = np.array(
            [float(row["mean_elapsed_minutes"]) for row in hourly_rows],
            dtype=np.float64,
        )

        ax.plot(
            mean_x,
            mean_y,
            linewidth=1.2,
            alpha=0.55,
            color="0.35",
            zorder=2,
        )

        scatter = ax.scatter(
            mean_x,
            mean_y,
            c=mean_elapsed,
            cmap="viridis",
            s=95,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            mean_elapsed,
            label_every_n_hours=1,
        )

        midnight_mask = np.array(
            [int(row["is_midnight"]) == 1 for row in hourly_rows],
            dtype=bool,
        )

        if np.any(midnight_mask):
            ax.scatter(
                mean_x[midnight_mask],
                mean_y[midnight_mask],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color="black",
                edgecolors="white",
                linewidths=0.8,
                label="Midnight",
                zorder=4,
            )

        for row, x_here, y_here in zip(hourly_rows, mean_x, mean_y):
            if int(row["is_midnight"]) == 1:
                label_text = "Midnight\n00:00"
                font_weight = "bold"
            else:
                label_text = row["hour_label"]
                font_weight = "normal"

            ax.annotate(
                label_text,
                xy=(x_here, y_here),
                xytext=HOURLY_LABEL_OFFSET_POINTS,
                textcoords="offset points",
                fontsize=8,
                fontweight=font_weight,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    edgecolor="0.5",
                    alpha=0.85,
                ),
                zorder=5,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{title_text}\n"
        "One averaged tilt-vector endpoint is shown for each local hour "
        f"from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_all_days_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def save_all_days_hourly_average_plot(hourly_rows, output_file):
    """
    Plot the hourly averaged tilt points from all processed PKLs together.

    Each night/day is assigned its own color. The rows should already be filtered
    to the requested local-hour window.
    """
    if not hourly_rows:
        return

    x_values = np.array(
        [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )
    y_values = np.array(
        [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )

    x_limits, y_limits = calculate_vector_limits(
        x_values,
        y_values,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    rows_by_night = {}
    for row in hourly_rows:
        night_key = row.get("night_label", row["start_local_datetime"][:10])
        rows_by_night.setdefault(night_key, []).append(row)

    night_keys = sorted(rows_by_night.keys())

    if len(night_keys) <= 20:
        cmap = plt.get_cmap("tab20", max(len(night_keys), 1))
        colors = [cmap(i) for i in range(len(night_keys))]
    else:
        cmap = plt.get_cmap("nipy_spectral", max(len(night_keys), 1))
        colors = [cmap(i / max(len(night_keys) - 1, 1)) for i in range(len(night_keys))]

    for color, night_key in zip(colors, night_keys):
        night_rows = sorted(
            rows_by_night[night_key],
            key=lambda row: row["hour_start_local_datetime"],
        )

        x_here = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        y_here = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        midnight_mask_here = np.array(
            [int(row["is_midnight"]) == 1 for row in night_rows],
            dtype=bool,
        )

        ax.plot(
            x_here,
            y_here,
            linewidth=1.4,
            alpha=0.85,
            color=color,
            zorder=2,
            label=night_key,
        )

        ax.scatter(
            x_here,
            y_here,
            s=75,
            color=color,
            edgecolors="black",
            linewidths=0.5,
            alpha=0.95,
            zorder=3,
        )

        if np.any(midnight_mask_here):
            ax.scatter(
                x_here[midnight_mask_here],
                y_here[midnight_mask_here],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color=color,
                edgecolors="black",
                linewidths=0.8,
                zorder=4,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        "Hourly Averaged Tilt Points: All Days\n"
        f"Local-hour averages from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00; each day has a different color"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=8,
        framealpha=0.9,
        title="Day",
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Preprocessing diagnostics
# ==================================================
def _symmetric_plot_limit(values, percentile=99.0, mask=None):
    values = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(values)
    if mask is not None:
        valid &= mask
    if not np.any(valid):
        return 1.0
    vmax = np.nanpercentile(np.abs(values[valid]), percentile)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    return float(vmax)


def add_diagnostic_slope_vector(
    ax,
    a_here,
    b_here,
    vector_reference_slope,
    nx,
    ny,
):
    """
    Overlay one plane-slope vector on an image.

    vector_reference_slope is shared by the original/preprocessed diagnostic
    panels for the current frame, so the two arrow lengths are directly
    comparable.
    """
    if (
        not np.isfinite(a_here)
        or not np.isfinite(b_here)
        or not np.isfinite(vector_reference_slope)
        or vector_reference_slope <= 0
    ):
        return

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    max_arrow_length_pix = (
        DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)
    )

    arrow_u = (a_here / vector_reference_slope) * max_arrow_length_pix
    arrow_v = (b_here / vector_reference_slope) * max_arrow_length_pix

    ax.scatter(
        [center_x],
        [center_y],
        s=CENTER_MARKER_SIZE,
        c=CENTER_MARKER_COLOR,
        marker="o",
        zorder=6,
    )

    ax.quiver(
        [center_x],
        [center_y],
        [arrow_u],
        [arrow_v],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color=ARROW_COLOR,
        width=0.008,
        headwidth=4.5,
        headlength=6.0,
        headaxislength=5.5,
        zorder=7,
    )


def save_preprocessing_diagnostic_plot(
    frame_index,
    wave_cube,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
    raw_a_values,
    raw_b_values,
    preprocessed_a_values,
    preprocessed_b_values,
    times_minutes,
    local_datetimes,
    out_dir,
):
    """
    Save a compact diagnostic figure for the retained preprocessing method.

    Top row:
      1. Raw + raw slope vector
      2. Gaussian background
      3. High-pass = raw - background

    Bottom row:
      4. Detected replacement mask
      5. Removed component
      6. Final preprocessed + preprocessed slope vector
    """
    nt, ny, nx = wave_cube.shape
    if frame_index < 0 or frame_index >= nt:
        return

    original = np.asarray(wave_cube[frame_index], dtype=np.float64)
    preprocessed, steps = preprocess_frame_for_plane_fit(
        original,
        fit_radius_mask,
        return_steps=True,
    )

    raw_a = raw_a_values[frame_index]
    raw_b = raw_b_values[frame_index]
    pre_a = preprocessed_a_values[frame_index]
    pre_b = preprocessed_b_values[frame_index]

    raw_slope = np.hypot(raw_a, raw_b)
    pre_slope = np.hypot(pre_a, pre_b)

    comparison_slopes = np.array([raw_slope, pre_slope], dtype=np.float64)
    comparison_slopes = comparison_slopes[np.isfinite(comparison_slopes)]
    vector_reference_slope = (
        float(np.nanmax(comparison_slopes))
        if comparison_slopes.size > 0
        else 1.0
    )
    if not np.isfinite(vector_reference_slope) or vector_reference_slope <= 0:
        vector_reference_slope = 1.0

    main_vmax = _symmetric_plot_limit(
        np.concatenate(
            [
                original[np.isfinite(original)],
                preprocessed[np.isfinite(preprocessed)],
                steps["gaussian_background"][
                    np.isfinite(steps["gaussian_background"])
                ],
            ]
        ),
        percentile=99.0,
    )
    high_pass_vmax = _symmetric_plot_limit(
        steps["high_pass"],
        percentile=99.0,
        mask=fit_radius_mask,
    )
    removed_vmax = _symmetric_plot_limit(
        steps["removed"],
        percentile=REMOVED_COMPONENT_SCALE_PERCENTILE,
        mask=fit_radius_mask,
    )

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 11),
    )

    panels = [
        (
            axes[0, 0],
            original,
            (
                "Raw + slope vector\n"
                f"x = {100.0 * raw_a:+.5f}, "
                f"y = {100.0 * raw_b:+.5f}, "
                f"|s| = {100.0 * raw_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
        (
            axes[0, 1],
            steps["gaussian_background"],
            f"Gaussian background\nsigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km",
            CMAP,
            main_vmax,
            "Background amplitude",
        ),
        (
            axes[0, 2],
            steps["high_pass"],
            "High-pass = raw - background",
            REMOVED_COMPONENT_CMAP,
            high_pass_vmax,
            "High-pass amplitude",
        ),
        (
            axes[1, 1],
            steps["removed"],
            "Removed component\n(raw - preprocessed)",
            REMOVED_COMPONENT_CMAP,
            removed_vmax,
            "Removed amplitude",
        ),
        (
            axes[1, 2],
            preprocessed,
            (
                "Final preprocessed + slope vector\n"
                f"x = {100.0 * pre_a:+.5f}, "
                f"y = {100.0 * pre_b:+.5f}, "
                f"|s| = {100.0 * pre_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
    ]

    for ax, panel_data, title_text, cmap, vmax_here, cbar_label in panels:
        image = ax.imshow(
            panel_data,
            origin="upper",
            cmap=cmap,
            vmin=-vmax_here,
            vmax=vmax_here,
        )
        fit_circle = Ellipse(
            xy=(center_x, center_y),
            width=2.0 * FIT_RADIUS_KM / DX_KM,
            height=2.0 * FIT_RADIUS_KM / DY_KM,
            fill=False,
            edgecolor="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.85,
            zorder=4,
        )
        ax.add_patch(fit_circle)
        ax.set_title(title_text, fontsize=10.5)
        ax.set_xlabel("X pixel")
        ax.set_ylabel("Y pixel")
        cbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.040,
            pad=0.025,
            aspect=30,
        )
        cbar.set_label(cbar_label)

    mask_image = axes[1, 0].imshow(
        steps["star_mask"].astype(np.float64),
        origin="upper",
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    fit_circle = Ellipse(
        xy=(center_x, center_y),
        width=2.0 * FIT_RADIUS_KM / DX_KM,
        height=2.0 * FIT_RADIUS_KM / DY_KM,
        fill=False,
        edgecolor="red",
        linestyle="--",
        linewidth=1.1,
        alpha=0.85,
        zorder=4,
    )
    axes[1, 0].add_patch(fit_circle)
    axes[1, 0].set_title("Detected peak mask\nwhite = replaced pixels")
    axes[1, 0].set_xlabel("X pixel")
    axes[1, 0].set_ylabel("Y pixel")
    cbar_mask = fig.colorbar(
        mask_image,
        ax=axes[1, 0],
        fraction=0.040,
        pad=0.025,
        aspect=30,
    )
    cbar_mask.set_ticks([0.0, 1.0])
    cbar_mask.set_ticklabels(["0", "1"])
    cbar_mask.set_label("Mask value")

    add_diagnostic_slope_vector(
        axes[0, 0],
        raw_a,
        raw_b,
        vector_reference_slope,
        nx,
        ny,
    )
    add_diagnostic_slope_vector(
        axes[1, 2],
        pre_a,
        pre_b,
        vector_reference_slope,
        nx,
        ny,
    )

    local_time_text = local_datetimes[frame_index].strftime(
        LOCAL_TIME_EXACT_FORMAT
    )
    replaced_pixel_count = int(np.count_nonzero(steps["star_mask"]))
    inside_fit_count = int(
        np.count_nonzero(steps["star_mask"] & fit_radius_mask)
    )

    fig.suptitle(
        f"Gaussian-Background Peak Diagnostic | "
        f"Frame {frame_index} | "
        f"t = {times_minutes[frame_index]:.1f} min | "
        f"Local time = {local_time_text}\n"
        f"Background sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km | "
        f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma | "
        f"mask dilation = {STAR_MASK_DILATION_KM:g} km | "
        f"fit radius = {FIT_RADIUS_KM:g} km | "
        f"replaced = {replaced_pixel_count} total "
        f"({inside_fit_count} inside fit radius)",
        fontsize=13,
    )

    diagnostic_dir = out_dir / "preprocessing_diagnostics"
    diagnostic_dir.mkdir(parents=True, exist_ok=True)

    output_file = (
        diagnostic_dir
        / f"diagnostic_frame_{frame_index:04d}_"
          f"t_{times_minutes[frame_index]:07.1f}_min.png"
    )

    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def build_diagnostic_frame_indices(
    nt,
    times_minutes,
):
    """
    Combine requested frame numbers with frames nearest the requested
    elapsed times.
    """
    requested_indices = set()

    for frame_index in DIAGNOSTIC_FRAME_INDICES:
        frame_index = int(frame_index)

        if 0 <= frame_index < nt:
            requested_indices.add(frame_index)

    finite_time = np.isfinite(times_minutes)

    if np.any(finite_time):
        valid_indices = np.flatnonzero(
            finite_time
        )

        for target_time_min in DIAGNOSTIC_TIMES_MIN:
            nearest_position = int(
                np.argmin(
                    np.abs(
                        times_minutes[valid_indices]
                        - float(target_time_min)
                    )
                )
            )

            requested_indices.add(
                int(
                    valid_indices[
                        nearest_position
                    ]
                )
            )

    return sorted(requested_indices)


# ==================================================
# Process one PKL
# ==================================================
def process_one_pkl(pkl_file):
    print("\n" + "=" * 80)
    print(f"Reading: {pkl_file}")

    label, wave_cube, date_start = load_wave_cube(pkl_file)
    start_local_datetime = parse_pkl_start_datetime(date_start)

    if start_local_datetime.year not in TARGET_YEARS:
        print(
            f"Skipped: internal year {start_local_datetime.year} "
            f"is not in {sorted(TARGET_YEARS)}"
        )
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_year",
                "output_folder": "",
                "frame_count": "",
                "successful_fits": "",
                "hourly_average_csv": "",
                "preprocessed_cube_file": "",
                "running_average_csv": "",
                "message": f"Year {start_local_datetime.year} not selected",
            },
            [],
        )

    output_name = safe_relative_output_name(pkl_file)
    out_dir = OUTPUT_ROOT / str(start_local_datetime.year) / output_name
    out_dir.mkdir(parents=True, exist_ok=True)

    npz_file = out_dir / "plane_fit_results.npz"

    running_average_tag = f"{RUNNING_AVERAGE_HOURS:g}hr".replace(".", "p")
    running_average_csv_file = (
        out_dir / f"plane_slope_{running_average_tag}_running_average.csv"
    )
    running_average_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_{running_average_tag}_running_average_"
          f"{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min.png"
    )
    running_average_whole_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_{running_average_tag}_running_average_"
          "whole_time.png"
    )
    running_average_timeseries_plot_file = (
        out_dir
        / f"plane_slope_{running_average_tag}_running_average_"
          "components_vs_time.png"
    )

    preprocessed_cube_file = out_dir / "preprocessed_perturbation_cube.npy"
    preprocessed_cube_metadata_file = (
        out_dir / "preprocessed_perturbation_cube_metadata.npz"
    )

    hourly_average_csv_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )
    hourly_average_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    required_existing_files = [
        npz_file,
        running_average_csv_file,
        running_average_plot_file,
        running_average_whole_plot_file,
        running_average_timeseries_plot_file,
        hourly_average_csv_file,
        hourly_average_plot_file,
    ]

    if SAVE_PREPROCESSED_CUBE:
        required_existing_files.extend(
            [
                preprocessed_cube_file,
                preprocessed_cube_metadata_file,
            ]
        )

    if SKIP_IF_RESULTS_EXIST and all(file.exists() for file in required_existing_files):
        print(f"Skipped existing results in: {out_dir}")
        hourly_rows = load_hourly_average_csv(hourly_average_csv_file)
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_existing",
                "output_folder": str(out_dir),
                "frame_count": wave_cube.shape[0],
                "successful_fits": "",
                "hourly_average_csv": str(hourly_average_csv_file),
                "preprocessed_cube_file": (str(preprocessed_cube_file) if SAVE_PREPROCESSED_CUBE else ""),
                "running_average_csv": str(running_average_csv_file),
                "message": "All required result files already exist",
            },
            hourly_rows,
        )

    nt, ny, nx = wave_cube.shape
    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")
    print(
        "PKL interval start datetime: "
        f"{start_local_datetime.strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    frame_numbers = np.arange(nt, dtype=int)

    # data[7][1] is the beginning of the first frame interval.
    # Each fitted point is assigned to the center of its interval.
    times_seconds = (frame_numbers + 0.5) * DT_SECONDS
    times_minutes = times_seconds / 60.0

    local_datetimes = np.array(
        [
            local_datetime_from_elapsed_minutes(
                start_local_datetime,
                elapsed_minutes,
            )
            for elapsed_minutes in times_minutes
        ],
        dtype=object,
    )

    local_datetime_iso = np.array(
        [
            local_datetime.isoformat(sep=" ")
            for local_datetime in local_datetimes
        ],
        dtype=str,
    )

    print(
        "First frame-center datetime: "
        f"{local_datetimes[0].strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    x_1d_km = (
        np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
    ) * DX_KM
    y_1d_km = (
        np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
    ) * DY_KM
    x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)

    radial_distance_km = np.hypot(x_grid_km, y_grid_km)
    fit_radius_mask = radial_distance_km <= FIT_RADIUS_KM
    fit_radius_pixel_count = int(np.count_nonzero(fit_radius_mask))

    print(
        f"Plane-fit region: radius <= {FIT_RADIUS_KM:.1f} km "
        f"({fit_radius_pixel_count} pixels before other validity filters)"
    )
    if APPLY_STAR_RESIDUAL_FILTER:
        print(
            "Spatial preprocessing: Gaussian-background peak replacement "
            f"(sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km, "
            f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma, "
            f"mask dilation = {STAR_MASK_DILATION_KM:g} km)"
        )
    else:
        print("Spatial preprocessing: disabled")

    # Raw plane-fit results
    raw_a_values = np.full(nt, np.nan, dtype=np.float64)
    raw_b_values = np.full(nt, np.nan, dtype=np.float64)
    raw_c_values = np.full(nt, np.nan, dtype=np.float64)
    raw_slope_values = np.full(nt, np.nan, dtype=np.float64)
    raw_angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    raw_rmse_values = np.full(nt, np.nan, dtype=np.float64)
    raw_valid_pixel_counts = np.zeros(nt, dtype=int)

    # Preprocessed plane-fit results
    a_values = np.full(nt, np.nan, dtype=np.float64)
    b_values = np.full(nt, np.nan, dtype=np.float64)
    c_values = np.full(nt, np.nan, dtype=np.float64)
    slope_values = np.full(nt, np.nan, dtype=np.float64)
    angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    rmse_values = np.full(nt, np.nan, dtype=np.float64)
    valid_pixel_counts = np.zeros(nt, dtype=int)

    preprocessed_cube_memmap = None
    if SAVE_PREPROCESSED_CUBE:
        print(
            "Saving final preprocessed cube while processing frames: "
            f"{preprocessed_cube_file.name} "
            f"({np.dtype(PREPROCESSED_CUBE_DTYPE).name})"
        )
        preprocessed_cube_memmap = np.lib.format.open_memmap(
            preprocessed_cube_file,
            mode="w+",
            dtype=PREPROCESSED_CUBE_DTYPE,
            shape=(nt, ny, nx),
        )

    print("Fitting planes to raw and preprocessed frames...")

    for frame_index in range(nt):
        raw_frame = wave_cube[frame_index]

        (
            raw_a_values[frame_index],
            raw_b_values[frame_index],
            raw_c_values[frame_index],
            raw_slope_values[frame_index],
            raw_angle_values_deg[frame_index],
            raw_rmse_values[frame_index],
            raw_valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            raw_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        preprocessed_frame = preprocess_frame_for_plane_fit(
            raw_frame,
            fit_radius_mask,
        )

        if preprocessed_cube_memmap is not None:
            preprocessed_cube_memmap[frame_index] = preprocessed_frame.astype(
                PREPROCESSED_CUBE_DTYPE,
                copy=False,
            )

        (
            a_values[frame_index],
            b_values[frame_index],
            c_values[frame_index],
            slope_values[frame_index],
            angle_values_deg[frame_index],
            rmse_values[frame_index],
            valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            preprocessed_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        if (
            frame_index == 0
            or frame_index == nt - 1
            or (frame_index + 1) % 25 == 0
        ):
            print(f"  Processed frame {frame_index + 1}/{nt}")

    if preprocessed_cube_memmap is not None:
        preprocessed_cube_memmap.flush()
        del preprocessed_cube_memmap
        preprocessed_cube_memmap = None

        np.savez(
            preprocessed_cube_metadata_file,
            input_file=str(pkl_file),
            cube_file=preprocessed_cube_file.name,
            cube_shape=np.array([nt, ny, nx], dtype=int),
            cube_dtype=np.dtype(PREPROCESSED_CUBE_DTYPE).name,
            cube_contents="final_preprocessed_perturbation",
            axis_order="time_y_x",
            frame=frame_numbers,
            time_seconds=times_seconds,
            time_minutes=times_minutes,
            time_reference="frame_center",
            frame_duration_seconds=DT_SECONDS,
            local_datetime=local_datetime_iso,
            start_local_datetime=start_local_datetime.isoformat(sep=" "),
            x_km=x_1d_km,
            y_km=y_1d_km,
            dx_km=DX_KM,
            dy_km=DY_KM,
            origin_convention="x right positive; y down positive; image center at 0 km",
            preprocessing_method="gaussian_background_outlier_replacement",
            background_gaussian_sigma_km=BACKGROUND_GAUSSIAN_SIGMA_KM,
            background_gaussian_truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
            star_residual_filter_enabled=APPLY_STAR_RESIDUAL_FILTER,
            star_residual_mad_sigma=STAR_RESIDUAL_MAD_SIGMA,
            star_mask_dilation_km=STAR_MASK_DILATION_KM,
            preprocessing_statistics_radius_km=FIT_RADIUS_KM,
        )

        print(f"Saved preprocessed cube: {preprocessed_cube_file}")
        print(f"Saved cube metadata:     {preprocessed_cube_metadata_file}")

    successful_fits = int(
        np.count_nonzero(
            np.isfinite(slope_values) & np.isfinite(angle_values_deg)
        )
    )

    # ----------------------------------------------
    # Centered running average of the slope vector
    # ----------------------------------------------
    (
        raw_running_mean_a,
        raw_running_mean_b,
        raw_running_vector_magnitude,
        raw_running_vector_direction_deg,
        raw_running_valid_counts,
    ) = calculate_centered_running_average_vector(
        raw_a_values,
        raw_b_values,
        times_minutes,
        RUNNING_AVERAGE_HOURS,
    )

    (
        running_mean_a,
        running_mean_b,
        running_vector_magnitude,
        running_vector_direction_deg,
        running_valid_counts,
    ) = calculate_centered_running_average_vector(
        a_values,
        b_values,
        times_minutes,
        RUNNING_AVERAGE_HOURS,
    )

    # Also save the direct running mean of the scalar slope magnitude.
    # This is not identical to |mean vector| when the direction varies.
    raw_running_scalar_magnitude, _ = calculate_centered_running_average(
        raw_slope_values,
        times_minutes,
        RUNNING_AVERAGE_HOURS,
    )
    running_scalar_magnitude, _ = calculate_centered_running_average(
        slope_values,
        times_minutes,
        RUNNING_AVERAGE_HOURS,
    )

    save_running_average_csv(
        output_file=running_average_csv_file,
        frame_numbers=frame_numbers,
        times_seconds=times_seconds,
        times_minutes=times_minutes,
        local_datetime_iso=local_datetime_iso,
        local_datetimes=local_datetimes,
        raw_mean_x=raw_running_mean_a,
        raw_mean_y=raw_running_mean_b,
        raw_vector_magnitude=raw_running_vector_magnitude,
        raw_vector_direction_deg=raw_running_vector_direction_deg,
        raw_mean_scalar_magnitude=raw_running_scalar_magnitude,
        raw_counts=raw_running_valid_counts,
        pre_mean_x=running_mean_a,
        pre_mean_y=running_mean_b,
        pre_vector_magnitude=running_vector_magnitude,
        pre_vector_direction_deg=running_vector_direction_deg,
        pre_mean_scalar_magnitude=running_scalar_magnitude,
        pre_counts=running_valid_counts,
    )

    # Main running-average plots: use the SAME 2-D vector-endpoint style
    # as the ordinary slope-vector plots.
    save_running_average_vector_plot(
        output_file=running_average_plot_file,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        processed_x=a_values,
        processed_y=b_values,
        running_mean_x=running_mean_a,
        running_mean_y=running_mean_b,
        time_min=TILT_PLOT_START_TIME_MIN,
        time_max=TILT_PLOT_END_TIME_MIN,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
        figure_size=(14, 12),
    )

    save_running_average_vector_plot(
        output_file=running_average_whole_plot_file,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        processed_x=a_values,
        processed_y=b_values,
        running_mean_x=running_mean_a,
        running_mean_y=running_mean_b,
        time_min=None,
        time_max=None,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
        figure_size=(18, 14),
    )

    # Keep the old component-vs-time plot as a secondary diagnostic.
    save_running_average_timeseries_plot(
        output_file=running_average_timeseries_plot_file,
        times_minutes=times_minutes,
        raw_mean_x=raw_running_mean_a,
        raw_mean_y=raw_running_mean_b,
        raw_vector_magnitude=raw_running_vector_magnitude,
        raw_vector_direction_deg=raw_running_vector_direction_deg,
        pre_mean_x=running_mean_a,
        pre_mean_y=running_mean_b,
        pre_vector_magnitude=running_vector_magnitude,
        pre_vector_direction_deg=running_vector_direction_deg,
    )

    # ----------------------------------------------
    # Optional preprocessing / fit diagnostics
    # ----------------------------------------------
    if MAKE_PREPROCESSING_DIAGNOSTICS:
        diagnostic_indices = build_diagnostic_frame_indices(
            nt,
            times_minutes,
        )

        print(
            "Saving preprocessing diagnostic plots for frames: "
            + ", ".join(
                str(index)
                for index in diagnostic_indices
            )
        )

        for diagnostic_frame_index in diagnostic_indices:
            save_preprocessing_diagnostic_plot(
                frame_index=diagnostic_frame_index,
                wave_cube=wave_cube,
                x_grid_km=x_grid_km,
                y_grid_km=y_grid_km,
                fit_radius_mask=fit_radius_mask,
                raw_a_values=raw_a_values,
                raw_b_values=raw_b_values,
                preprocessed_a_values=a_values,
                preprocessed_b_values=b_values,
                times_minutes=times_minutes,
                local_datetimes=local_datetimes,
                out_dir=out_dir,
            )

    # ----------------------------------------------
    # Save CSV and NPZ
    # ----------------------------------------------
    csv_file = out_dir / "plane_fit_results.csv"

    with open(csv_file, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "raw_slope_x_amplitude_per_km",
                "raw_slope_y_amplitude_per_km",
                "raw_plane_intercept",
                "raw_slope_magnitude_amplitude_per_km",
                "raw_increase_direction_deg",
                "raw_fit_rmse",
                "raw_valid_pixel_count",
                "preprocessed_slope_x_amplitude_per_km",
                "preprocessed_slope_y_amplitude_per_km",
                "preprocessed_plane_intercept",
                "preprocessed_slope_magnitude_amplitude_per_km",
                "preprocessed_increase_direction_deg",
                "preprocessed_fit_rmse",
                "preprocessed_valid_pixel_count",
                "running_average_window_hours",
                "raw_running_mean_x_slope_amplitude_per_km",
                "raw_running_mean_y_slope_amplitude_per_km",
                "raw_running_mean_vector_magnitude_amplitude_per_km",
                "raw_running_mean_vector_direction_deg",
                "raw_running_mean_of_scalar_slope_magnitude_amplitude_per_km",
                "raw_running_average_valid_sample_count",
                "preprocessed_running_mean_x_slope_amplitude_per_km",
                "preprocessed_running_mean_y_slope_amplitude_per_km",
                "preprocessed_running_mean_vector_magnitude_amplitude_per_km",
                "preprocessed_running_mean_vector_direction_deg",
                "preprocessed_running_mean_of_scalar_slope_magnitude_amplitude_per_km",
                "preprocessed_running_average_valid_sample_count",
            ]
        )

        for frame_index in range(nt):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(
                        LOCAL_TIME_EXACT_FORMAT
                    ),
                    raw_a_values[frame_index],
                    raw_b_values[frame_index],
                    raw_c_values[frame_index],
                    raw_slope_values[frame_index],
                    raw_angle_values_deg[frame_index],
                    raw_rmse_values[frame_index],
                    raw_valid_pixel_counts[frame_index],
                    a_values[frame_index],
                    b_values[frame_index],
                    c_values[frame_index],
                    slope_values[frame_index],
                    angle_values_deg[frame_index],
                    rmse_values[frame_index],
                    valid_pixel_counts[frame_index],
                    RUNNING_AVERAGE_HOURS,
                    raw_running_mean_a[frame_index],
                    raw_running_mean_b[frame_index],
                    raw_running_vector_magnitude[frame_index],
                    raw_running_vector_direction_deg[frame_index],
                    raw_running_scalar_magnitude[frame_index],
                    raw_running_valid_counts[frame_index],
                    running_mean_a[frame_index],
                    running_mean_b[frame_index],
                    running_vector_magnitude[frame_index],
                    running_vector_direction_deg[frame_index],
                    running_scalar_magnitude[frame_index],
                    running_valid_counts[frame_index],
                ]
            )

    np.savez(
        npz_file,
        input_file=str(pkl_file),
        frame=frame_numbers,
        time_seconds=times_seconds,
        time_minutes=times_minutes,
        time_reference="frame_center",
        frame_duration_seconds=DT_SECONDS,
        fit_radius_km=FIT_RADIUS_KM,
        preprocessing_method="gaussian_background_outlier_replacement",
        background_gaussian_sigma_km=BACKGROUND_GAUSSIAN_SIGMA_KM,
        background_gaussian_truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
        star_residual_filter_enabled=APPLY_STAR_RESIDUAL_FILTER,
        star_residual_mad_sigma=STAR_RESIDUAL_MAD_SIGMA,
        star_mask_dilation_km=STAR_MASK_DILATION_KM,
        local_datetime=local_datetime_iso,
        start_local_datetime=start_local_datetime.isoformat(sep=" "),

        raw_slope_x_amplitude_per_km=raw_a_values,
        raw_slope_y_amplitude_per_km=raw_b_values,
        raw_plane_intercept=raw_c_values,
        raw_slope_magnitude_amplitude_per_km=raw_slope_values,
        raw_increase_direction_deg=raw_angle_values_deg,
        raw_fit_rmse=raw_rmse_values,
        raw_valid_pixel_count=raw_valid_pixel_counts,

        preprocessed_slope_x_amplitude_per_km=a_values,
        preprocessed_slope_y_amplitude_per_km=b_values,
        preprocessed_plane_intercept=c_values,
        preprocessed_slope_magnitude_amplitude_per_km=slope_values,
        preprocessed_increase_direction_deg=angle_values_deg,
        preprocessed_fit_rmse=rmse_values,
        preprocessed_valid_pixel_count=valid_pixel_counts,

        running_average_window_hours=RUNNING_AVERAGE_HOURS,
        running_average_definition="centered time window; +/- half window; edges use available samples",

        raw_running_mean_x_slope_amplitude_per_km=raw_running_mean_a,
        raw_running_mean_y_slope_amplitude_per_km=raw_running_mean_b,
        raw_running_mean_vector_magnitude_amplitude_per_km=raw_running_vector_magnitude,
        raw_running_mean_vector_direction_deg=raw_running_vector_direction_deg,
        raw_running_mean_of_scalar_slope_magnitude_amplitude_per_km=raw_running_scalar_magnitude,
        raw_running_average_valid_sample_count=raw_running_valid_counts,

        preprocessed_running_mean_x_slope_amplitude_per_km=running_mean_a,
        preprocessed_running_mean_y_slope_amplitude_per_km=running_mean_b,
        preprocessed_running_mean_vector_magnitude_amplitude_per_km=running_vector_magnitude,
        preprocessed_running_mean_vector_direction_deg=running_vector_direction_deg,
        preprocessed_running_mean_of_scalar_slope_magnitude_amplitude_per_km=running_scalar_magnitude,
        preprocessed_running_average_valid_sample_count=running_valid_counts,

        preprocessed_cube_file=(
            preprocessed_cube_file.name if SAVE_PREPROCESSED_CUBE else ""
        ),
        preprocessed_cube_metadata_file=(
            preprocessed_cube_metadata_file.name if SAVE_PREPROCESSED_CUBE else ""
        ),

        # Backward-compatible aliases: these are PREPROCESSED values.
        slope_x_amplitude_per_km=a_values,
        slope_y_amplitude_per_km=b_values,
        plane_intercept=c_values,
        slope_magnitude_amplitude_per_km=slope_values,
        increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
        fit_rmse=rmse_values,
        valid_pixel_count=valid_pixel_counts,
    )

    # ----------------------------------------------
    # Angle plot
    # ----------------------------------------------
    fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

    ax_angle.plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_angle.plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )

    ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
    ax_angle.set_xlabel("Frame")
    ax_angle.set_ylabel(
        "Direction of perturbation increase "
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks(np.arange(-180, 181, 45))
    ax_angle.grid(True, alpha=0.3)
    ax_angle.legend()

    fig_angle.tight_layout()
    angle_plot_file = out_dir / "plane_angle_vs_frame.png"
    fig_angle.savefig(angle_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_angle)

    # ----------------------------------------------
    # Slope plot
    # ----------------------------------------------
    fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

    ax_slope.plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_slope.plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        alpha=0.55,
        label="Preprocessed",
    )
    ax_slope.plot(
        frame_numbers,
        raw_running_vector_magnitude,
        linewidth=1.6,
        linestyle="--",
        label=f"Raw {RUNNING_AVERAGE_HOURS:g}-hr mean vector magnitude",
    )
    ax_slope.plot(
        frame_numbers,
        running_vector_magnitude,
        linewidth=2.0,
        label=f"Preprocessed {RUNNING_AVERAGE_HOURS:g}-hr mean vector magnitude",
    )

    ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
    ax_slope.grid(True, alpha=0.3)
    ax_slope.legend()

    fig_slope.tight_layout()
    slope_plot_file = out_dir / "plane_slope_vs_frame.png"
    fig_slope.savefig(slope_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_slope)

    # ----------------------------------------------
    # Combined angle/slope comparison
    # ----------------------------------------------
    fig_combined, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 8),
        sharex=True,
    )

    axes[0].plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[0].plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[0].set_title("Plane-fit Direction and Slope: Raw vs Preprocessed")
    axes[0].set_ylabel(
        "Increase direction\n"
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    axes[0].set_ylim(-180, 180)
    axes[0].set_yticks(np.arange(-180, 181, 45))
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[1].plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig_combined.tight_layout()
    combined_plot_file = out_dir / "plane_angle_and_slope_vs_frame.png"
    fig_combined.savefig(
        combined_plot_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_combined)

    # ----------------------------------------------
    # Physical tilt vectors: raw vs preprocessed
    # ----------------------------------------------
    raw_x_slope_per_100km = 100.0 * raw_a_values
    raw_y_slope_per_100km = 100.0 * raw_b_values

    x_slope_per_100km = 100.0 * a_values
    y_slope_per_100km = 100.0 * b_values

    raw_finite_vector = (
        np.isfinite(raw_x_slope_per_100km)
        & np.isfinite(raw_y_slope_per_100km)
        & np.isfinite(times_minutes)
    )
    preprocessed_finite_vector = (
        np.isfinite(x_slope_per_100km)
        & np.isfinite(y_slope_per_100km)
        & np.isfinite(times_minutes)
    )

    raw_tilt_plot_time_mask = (
        raw_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )
    preprocessed_tilt_plot_time_mask = (
        preprocessed_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )

    whole_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_finite_vector],
            x_slope_per_100km[preprocessed_finite_vector],
        ]
    )
    whole_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_finite_vector],
            y_slope_per_100km[preprocessed_finite_vector],
        ]
    )

    whole_vector_x_limits, whole_vector_y_limits = calculate_vector_limits(
        whole_combined_x,
        whole_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    zoom_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_tilt_plot_time_mask],
            x_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )
    zoom_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_tilt_plot_time_mask],
            y_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )

    zoom_vector_x_limits, zoom_vector_y_limits = calculate_vector_limits(
        zoom_combined_x,
        zoom_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    # ----------------------------------------------
    # 3-D tilt-vector-vs-time comparison
    # top = raw, bottom = preprocessed
    # ----------------------------------------------
    fig_tilt_3d = plt.figure(figsize=(14, 15))
    ax_raw_3d = fig_tilt_3d.add_subplot(211, projection="3d")
    ax_pre_3d = fig_tilt_3d.add_subplot(212, projection="3d")

    comparison_3d = [
        (
            ax_raw_3d,
            raw_tilt_plot_time_mask,
            raw_x_slope_per_100km,
            raw_y_slope_per_100km,
            "Raw data",
        ),
        (
            ax_pre_3d,
            preprocessed_tilt_plot_time_mask,
            x_slope_per_100km,
            y_slope_per_100km,
            "Preprocessed",
        ),
    ]

    scatter_for_3d_cbar = None

    for (
        ax_here,
        mask_here,
        x_here_all,
        y_here_all,
        panel_title,
    ) in comparison_3d:
        selected_indices = np.flatnonzero(mask_here)

        if selected_indices.size > 0:
            x_here = x_here_all[selected_indices]
            y_here = y_here_all[selected_indices]
            t_here = times_minutes[selected_indices]

            ax_here.plot(
                x_here,
                y_here,
                t_here,
                linewidth=1.0,
                alpha=0.55,
            )

            scatter_here = ax_here.scatter(
                x_here,
                y_here,
                t_here,
                c=t_here,
                cmap="viridis",
                s=18,
                depthshade=False,
            )
            scatter_for_3d_cbar = scatter_here

            add_hourly_markers_3d(
                ax=ax_here,
                selected_indices=selected_indices,
                x_values=x_here_all,
                y_values=y_here_all,
                z_values=times_minutes,
                local_datetimes=local_datetimes,
                start_datetime=start_local_datetime,
                times_minutes=times_minutes,
            )

            hour_datetimes_3d, hour_elapsed_3d = make_hour_boundaries(
                start_local_datetime,
                times_minutes[selected_indices],
            )
            if hour_elapsed_3d.size > 0:
                ax_here.set_zticks(hour_elapsed_3d)
                ax_here.set_zticklabels(
                    [
                        (
                            "Midnight\n00:00"
                            if hour_datetime.hour == 0
                            else hour_datetime.strftime(
                                LOCAL_TIME_LABEL_FORMAT
                            )
                        )
                        + f"\n({elapsed_minutes:.0f} min)"
                        for hour_datetime, elapsed_minutes in zip(
                            hour_datetimes_3d,
                            hour_elapsed_3d,
                        )
                    ]
                )

        ax_here.set_xlim(*zoom_vector_x_limits)
        ax_here.set_ylim(*zoom_vector_y_limits)
        ax_here.set_box_aspect((1, 1, 1))
        ax_here.set_title(panel_title)
        ax_here.set_xlabel(
            "X slope (amplitude per 100 km; right positive)"
        )
        ax_here.set_ylabel(
            "Y slope (amplitude per 100 km; down positive)"
        )
        ax_here.set_zlabel("Elapsed time / local time")
        ax_here.grid(True, alpha=0.3)

        handles_here, _ = ax_here.get_legend_handles_labels()
        if handles_here:
            ax_here.legend()

    if scatter_for_3d_cbar is not None:
        combined_zoom_mask = (
            raw_tilt_plot_time_mask | preprocessed_tilt_plot_time_mask
        )
        cbar_3d = fig_tilt_3d.colorbar(
            scatter_for_3d_cbar,
            ax=[ax_raw_3d, ax_pre_3d],
            pad=0.05,
            shrink=0.72,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar_3d,
            start_local_datetime,
            times_minutes[combined_zoom_mask],
        )

    fig_tilt_3d.suptitle(
        "Plane-fit Tilt Vector vs Time: Raw vs Preprocessed\n"
        "Top = raw data; bottom = preprocessed; same X/Y limits",
        fontsize=14,
    )
    fig_tilt_3d.subplots_adjust(
        left=0.04,
        right=0.88,
        bottom=0.04,
        top=0.93,
        hspace=0.18,
    )

    tilt_vector_3d_file = out_dir / "plane_tilt_vector_3d_vs_time.png"
    fig_tilt_3d.savefig(
        tilt_vector_3d_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_tilt_3d)

    # ----------------------------------------------
    # 2-D whole-time and zoomed comparisons
    # ----------------------------------------------
    tilt_vector_2d_whole_file = (
        out_dir / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
    )

    tilt_vector_2d_zoom_file = (
        out_dir
        / f"plane_tilt_vector_2d_zoom_{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min_colored_by_time.png"
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_finite_vector,
        preprocessed_time_mask=preprocessed_finite_vector,
        output_file=tilt_vector_2d_whole_file,
        title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
        x_limits=whole_vector_x_limits,
        y_limits=whole_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
        figure_size=(18, 21),
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_tilt_plot_time_mask,
        preprocessed_time_mask=preprocessed_tilt_plot_time_mask,
        output_file=tilt_vector_2d_zoom_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: "
            f"{TILT_PLOT_START_TIME_MIN:.0f}–"
            f"{TILT_PLOT_END_TIME_MIN:.0f} Minutes"
        ),
        x_limits=zoom_vector_x_limits,
        y_limits=zoom_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
        figure_size=(14, 18),
    )

    # New: hourly-averaged zoomed tilt points
    hourly_average_rows = compute_hourly_average_rows(
        selected_indices=np.flatnonzero(preprocessed_finite_vector),
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        input_file=pkl_file,
        output_folder=out_dir,
        start_local_datetime=start_local_datetime,
    )

    save_hourly_average_csv(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_csv_file,
    )

    if hourly_average_rows:
        hourly_average_x = np.array(
            [
                float(row["mean_x_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )
        hourly_average_y = np.array(
            [
                float(row["mean_y_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )

        hourly_average_x_limits, hourly_average_y_limits = (
            calculate_vector_limits(
                hourly_average_x,
                hourly_average_y,
                VECTOR_PLOT_PADDING_FRACTION,
            )
        )
    else:
        hourly_average_x_limits = (-1.0, 1.0)
        hourly_average_y_limits = (1.0, -1.0)

    save_hourly_average_tilt_plot(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_plot_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: Hourly Averages\n"
            f"{HOURLY_AVERAGE_START_HOUR:02d}:00–"
            f"{HOURLY_AVERAGE_END_HOUR:02d}:00 Local Time"
        ),
        x_limits=hourly_average_x_limits,
        y_limits=hourly_average_y_limits,
        start_local_datetime=start_local_datetime,
        figure_size=ZOOM_TIME_TILT_FIGSIZE,
    )

    # ----------------------------------------------
    # Animation: raw vs preprocessed
    # ----------------------------------------------
    saved_animation_file = ""

    if MAKE_ANIMATION:
        all_slopes_for_scale = np.concatenate(
            [
                raw_slope_values[np.isfinite(raw_slope_values)],
                slope_values[np.isfinite(slope_values)],
            ]
        )

        if all_slopes_for_scale.size > 0:
            slope_ref = np.nanpercentile(
                all_slopes_for_scale,
                ARROW_LENGTH_PERCENTILE,
            )
        else:
            slope_ref = 1.0

        if not np.isfinite(slope_ref) or slope_ref <= 0:
            slope_ref = 1.0

        max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

        raw_arrow_u = np.zeros(nt, dtype=np.float64)
        raw_arrow_v = np.zeros(nt, dtype=np.float64)
        pre_arrow_u = np.zeros(nt, dtype=np.float64)
        pre_arrow_v = np.zeros(nt, dtype=np.float64)

        finite_raw_ab = np.isfinite(raw_a_values) & np.isfinite(raw_b_values)
        finite_pre_ab = np.isfinite(a_values) & np.isfinite(b_values)

        raw_arrow_u[finite_raw_ab] = (
            raw_a_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix
        raw_arrow_v[finite_raw_ab] = (
            raw_b_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix

        pre_arrow_u[finite_pre_ab] = (
            a_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix
        pre_arrow_v[finite_pre_ab] = (
            b_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix

        vmax = np.nanpercentile(np.abs(wave_cube), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax

        preprocessed_frame_0 = preprocess_frame_for_plane_fit(
            wave_cube[0],
            fit_radius_mask,
        )

        fig_anim, (ax_raw, ax_pre) = plt.subplots(
            nrows=1,
            ncols=2,
            figsize=(13, 6),
        )

        im_raw = ax_raw.imshow(
            wave_cube[0],
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )
        im_pre = ax_pre.imshow(
            preprocessed_frame_0,
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )

        cbar_raw = fig_anim.colorbar(
            im_raw,
            ax=ax_raw,
            fraction=0.046,
            pad=0.04,
        )
        cbar_raw.set_label("Perturbation amplitude")

        cbar_pre = fig_anim.colorbar(
            im_pre,
            ax=ax_pre,
            fraction=0.046,
            pad=0.04,
        )
        cbar_pre.set_label("Perturbation amplitude")

        center_x = (nx - 1) / 2.0
        center_y = (ny - 1) / 2.0

        for ax_here in (ax_raw, ax_pre):
            fit_outline = Ellipse(
                xy=(center_x, center_y),
                width=2.0 * FIT_RADIUS_KM / DX_KM,
                height=2.0 * FIT_RADIUS_KM / DY_KM,
                fill=False,
                edgecolor="black",
                linestyle="--",
                linewidth=1.2,
                alpha=0.8,
                zorder=4,
            )
            ax_here.add_patch(fit_outline)
            ax_here.scatter(
                [center_x],
                [center_y],
                s=CENTER_MARKER_SIZE,
                c=CENTER_MARKER_COLOR,
                marker="o",
            )
            ax_here.set_xlabel("X pixel")
            ax_here.set_ylabel("Y pixel")

        raw_quiver = ax_raw.quiver(
            [center_x],
            [center_y],
            [raw_arrow_u[0]],
            [raw_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        pre_quiver = ax_pre.quiver(
            [center_x],
            [center_y],
            [pre_arrow_u[0]],
            [pre_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        ax_raw.set_title("Raw data + raw slope vector")
        ax_pre.set_title("Preprocessed + preprocessed slope vector")

        title = fig_anim.suptitle("")

        raw_info = ax_raw.text(
            0.02,
            0.98,
            "",
            transform=ax_raw.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        pre_info = ax_pre.text(
            0.02,
            0.98,
            "",
            transform=ax_pre.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        def update(frame_index):
            raw_frame = wave_cube[frame_index]
            preprocessed_frame = preprocess_frame_for_plane_fit(
                raw_frame,
                fit_radius_mask,
            )

            im_raw.set_data(raw_frame)
            im_pre.set_data(preprocessed_frame)

            raw_quiver.set_UVC(
                [raw_arrow_u[frame_index]],
                [raw_arrow_v[frame_index]],
            )
            pre_quiver.set_UVC(
                [pre_arrow_u[frame_index]],
                [pre_arrow_v[frame_index]],
            )

            local_time_text = local_datetimes[frame_index].strftime(
                LOCAL_TIME_EXACT_FORMAT
            )
            title.set_text(
                f"Raw vs Preprocessed Plane-fit Vector | "
                f"Frame {frame_index}/{nt - 1} | "
                f"t = {times_minutes[frame_index]:.1f} min | "
                f"Local time = {local_time_text}"
            )

            raw_info.set_text(
                f"Direction = {raw_angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {raw_slope_values[frame_index]:.4g} amp/km\n"
                f"a = {raw_a_values[frame_index]:.4g}\n"
                f"b = {raw_b_values[frame_index]:.4g}"
            )
            pre_info.set_text(
                f"Direction = {angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {slope_values[frame_index]:.4g} amp/km\n"
                f"a = {a_values[frame_index]:.4g}\n"
                f"b = {b_values[frame_index]:.4g}"
            )

            return (
                im_raw,
                im_pre,
                raw_quiver,
                pre_quiver,
                title,
                raw_info,
                pre_info,
            )

        ani = FuncAnimation(
            fig_anim,
            update,
            frames=nt,
            interval=1000 / FPS,
            blit=False,
        )

        fig_anim.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))

        animation_mp4_file = out_dir / "plane_fit_vector_animation.mp4"
        animation_gif_file = out_dir / "plane_fit_vector_animation.gif"

        try:
            writer = FFMpegWriter(
                fps=FPS,
                bitrate=3000,
            )
            ani.save(
                animation_mp4_file,
                writer=writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_mp4_file)
        except Exception as exc:
            print(
                "Could not save MP4 with ffmpeg. "
                "Falling back to GIF."
            )
            print(f"Reason: {exc}")

            gif_writer = PillowWriter(fps=FPS)
            ani.save(
                animation_gif_file,
                writer=gif_writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_gif_file)

        plt.close(fig_anim)

    print(f"Finished: {pkl_file.name}")
    print(f"Successful fits: {successful_fits}/{nt}")
    print(f"Output folder: {out_dir}")

    return (
        {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "processed",
            "output_folder": str(out_dir),
            "frame_count": nt,
            "successful_fits": successful_fits,
            "hourly_average_csv": str(hourly_average_csv_file),
            "preprocessed_cube_file": (str(preprocessed_cube_file) if SAVE_PREPROCESSED_CUBE else ""),
            "running_average_csv": str(running_average_csv_file),
            "message": (
                f"Animation: {saved_animation_file}"
                if MAKE_ANIMATION
                else "Animation disabled"
            ),
        },
        hourly_average_rows,
    )


# ==================================================
# Batch driver
# ==================================================
def discover_pkl_files():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f"Input folder does not exist:\n{INPUT_ROOT}"
        )

    iterator = (
        INPUT_ROOT.rglob("*.pkl")
        if RECURSIVE_SEARCH
        else INPUT_ROOT.glob("*.pkl")
    )

    files = []
    output_root_resolved = OUTPUT_ROOT.resolve()

    for file_path in iterator:
        try:
            file_resolved = file_path.resolve()
            if (
                file_resolved == output_root_resolved
                or output_root_resolved in file_resolved.parents
            ):
                continue
        except OSError:
            pass

        files.append(file_path)

    return sorted(files)


def write_batch_summary(summary_rows):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary_file = OUTPUT_ROOT / "batch_processing_summary.csv"

    fieldnames = [
        "input_file",
        "internal_start_datetime",
        "status",
        "output_folder",
        "frame_count",
        "successful_fits",
        "hourly_average_csv",
        "preprocessed_cube_file",
        "running_average_csv",
        "message",
    ]

    with open(summary_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_file


def main():
    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Input root: {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Target years: {sorted(TARGET_YEARS)}")
    print(f"Animations enabled: {MAKE_ANIMATION}")
    print(f"Save preprocessed cube: {SAVE_PREPROCESSED_CUBE}")
    print(
        "Slope running average: centered "
        f"{RUNNING_AVERAGE_HOURS:g}-hour window"
    )
    print(
        "Preprocessing diagnostics enabled: "
        f"{MAKE_PREPROCESSING_DIAGNOSTICS}"
    )

    # ==================================================
    # Single-file mode
    # ==================================================
    if PROCESS_SINGLE_FILE:
        single_file = Path(
            SINGLE_PKL_FILE
        ).expanduser()

        print("\n" + "=" * 80)
        print("RUN MODE: SINGLE FILE")
        print(f"Selected PKL: {single_file}")

        if not single_file.exists():
            raise FileNotFoundError(
                "PROCESS_SINGLE_FILE is True, but the selected PKL "
                "does not exist:\n"
                f"{single_file}\n\n"
                "Change SINGLE_PKL_FILE near the top of the script."
            )

        summary_rows = []

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                single_file
            )

            summary_rows.append(
                summary_row
            )

        except Exception as exc:
            print(f"FAILED: {single_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(single_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "preprocessed_cube_file": "",
                    "running_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            summary_file = write_batch_summary(
                summary_rows
            )
            print(
                f"Summary saved: {summary_file}"
            )

            if not CONTINUE_ON_ERROR:
                raise

            return

        finally:
            plt.close("all")
            gc.collect()

        summary_file = write_batch_summary(
            summary_rows
        )

        print("\n" + "=" * 80)
        print("Single-file processing finished.")
        print(f"Summary: {summary_file}")

        if (
            summary_rows
            and summary_rows[0]["status"] == "processed"
        ):
            print(
                "Output folder: "
                f"{summary_rows[0]['output_folder']}"
            )

        print(
            "\nWhen you are satisfied with this day's preprocessing and "
            "plane fit, set PROCESS_SINGLE_FILE = False to run the batch."
        )

        return

    # ==================================================
    # Batch mode
    # ==================================================
    print("\n" + "=" * 80)
    print("RUN MODE: BATCH")

    pkl_files = discover_pkl_files()

    print(
        f"PKL files found: {len(pkl_files)}"
    )

    if not pkl_files:
        print("No PKL files were found.")
        return

    summary_rows = []
    all_days_hourly_average_rows = []

    for (
        file_number,
        pkl_file,
    ) in enumerate(
        pkl_files,
        start=1,
    ):
        print(
            f"\nBatch file {file_number}/{len(pkl_files)}: "
            f"{pkl_file.name}"
        )

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                pkl_file
            )

            summary_rows.append(
                summary_row
            )

            all_days_hourly_average_rows.extend(
                hourly_rows
            )

        except Exception as exc:
            print(f"FAILED: {pkl_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(pkl_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "preprocessed_cube_file": "",
                    "running_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            if not CONTINUE_ON_ERROR:
                summary_file = write_batch_summary(
                    summary_rows
                )
                print(
                    f"Partial summary saved: {summary_file}"
                )
                raise

        finally:
            plt.close("all")
            gc.collect()

        write_batch_summary(
            summary_rows
        )

    summary_file = write_batch_summary(
        summary_rows
    )

    all_days_hourly_average_csv_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )

    all_days_hourly_average_plot_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    save_all_days_hourly_average_csv(
        all_days_hourly_average_rows,
        all_days_hourly_average_csv_file,
    )

    save_all_days_hourly_average_plot(
        all_days_hourly_average_rows,
        all_days_hourly_average_plot_file,
    )

    processed_count = sum(
        row["status"] == "processed"
        for row in summary_rows
    )

    skipped_count = sum(
        str(row["status"]).startswith(
            "skipped"
        )
        for row in summary_rows
    )

    failed_count = sum(
        row["status"] == "failed"
        for row in summary_rows
    )

    print("\n" + "=" * 80)
    print("Batch finished.")
    print(f"Processed: {processed_count}")
    print(f"Skipped:   {skipped_count}")
    print(f"Failed:    {failed_count}")
    print(f"Summary:   {summary_file}")

    print(
        "All-days hourly-average CSV:  "
        f"{all_days_hourly_average_csv_file}"
    )

    print(
        "All-days hourly-average plot: "
        f"{all_days_hourly_average_plot_file}"
    )

    print(
        "Hourly-average local-hour window: "
        f"{HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )


if __name__ == "__main__":
    main()


Input root: /Users/anhphan/Downloads/pkl
Output root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs
Target years: [2024, 2025, 2026]
Animations enabled: True
Save preprocessed cube: True
Slope running average: centered 2-hour window
Preprocessing diagnostics enabled: True

RUN MODE: SINGLE FILE
Selected PKL: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)
PKL interval start datetime: 2025-03-25 20:29:27
First frame-center datetime: 2025-03-25 20:30:27
Plane-fit region: radius <= 300.0 km (282792 pixels before other validity filters)
Spatial preprocessing: Gaussian-background peak replacement (sigma = 40 km, threshold = 2.5 MAD-sigma, mask dilation = 4 km)
Saving final preprocessed cube while processing frames: preprocessed_perturbation_cube.npy (float32)
Fitting planes to raw and preprocessed frames...
  Proces

## Read cube and create keograms

In [2]:
# -*- coding: utf-8 -*-
"""
Create BLO directional keograms from the saved FINAL PREPROCESSED cube.

This is a follow-up to:
    BLO_plane_fit_with_cube_and_2hr_running_average.py

Expected input files in CUBE_DIR:
    preprocessed_perturbation_cube.npy
    preprocessed_perturbation_cube_metadata.npz

The cube is loaded with mmap_mode="r", so the full time x y x x cube does not
have to be copied into RAM.

For each requested angle this script:
    1. Creates a center-crossing directional keogram.
    2. Averages across a narrow strip perpendicular to the cut.
    3. Saves the keogram image and full keogram .npy array.
    4. Plots/saves time series at selected distances along the cut.

It also optionally saves:
    - an animation of the preprocessed cube,
    - first/final frame images,
    - the first frame with directional cut lines,
    - common coordinate/time metadata arrays.

Time convention:
    The saved metadata from the preprocessing script is used directly.
    Therefore the keogram times are frame-center times. With 120-s frames,
    frame 0 is at 1 minute after the PKL interval start.

Angle convention:
    0 degrees   = E-W cut, increasing toward +X / image right
    +90 degrees = N-S cut, increasing toward +Y / image down
    -90 degrees = N-S cut, increasing toward -Y / image up
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from scipy.ndimage import map_coordinates


# ==================================================
# Input / output settings
# ==================================================
DOWNLOADS_DIR = Path.home() / "Downloads"

# This should point to the OUTPUT FOLDER produced for one PKL by the
# preprocessing / plane-fit script.
#
# Example for:
#   ~/Downloads/pkl/processed_data_full_03_25_2025.pkl
#
# the preprocessing script normally creates:
#   ~/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/
#       processed_data_full_03_25_2025/
CUBE_DIR = (
    DOWNLOADS_DIR
    / "pkl"
    / "BLO_plane_fit_batch_outputs"
    / "2025"
    / "processed_data_full_03_25_2025"
)

CUBE_FILE = CUBE_DIR / "preprocessed_perturbation_cube.npy"
METADATA_FILE = CUBE_DIR / "preprocessed_perturbation_cube_metadata.npz"

# Put the keogram products beside the preprocessing products.
OUT_DIR = CUBE_DIR / "keogram_outputs"


# ==================================================
# Output switches
# ==================================================
MAKE_ANIMATION = True
SAVE_FIRST_AND_FINAL_FRAMES = True
SAVE_FIRST_FRAME_WITH_CUTS = True

FPS = 20
DPI = 200
CMAP = "RdBu_r"

# Estimate the animation/frame color scale from a subset of frames instead of
# forming abs(cube) for the entire disk-backed cube at once.
CUBE_COLOR_PERCENTILE = 99.0
CUBE_COLOR_SAMPLE_FRAMES = 15


# ==================================================
# Keogram settings
# ==================================================
# Make keograms every 5 degrees.
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first-frame cut figure.
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Number of neighboring parallel cuts averaged together.
# Odd values are easiest to interpret because one strip is exactly centered.
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits.
KEOGRAM_TIME_MIN_MIN = 0.0
KEOGRAM_TIME_MIN_MAX = 500.0
KEOGRAM_DISTANCE_MIN = -250.0
KEOGRAM_DISTANCE_MAX = 250.0

# Distance rows to plot from each keogram.
PROFILE_DISTANCES_KM = [-250.0, 0.0, 250.0]

# Percentile used to determine the color range separately for each directional
# keogram, using only the displayed time/distance region.
KEOGRAM_COLOR_PERCENTILE = 99.0

# Fractional padding added to the automatically determined y range of each
# profile plot.
PROFILE_Y_PADDING_FRACTION = 0.05

# Interpolation order for scipy.ndimage.map_coordinates.
# 1 = bilinear interpolation.
INTERPOLATION_ORDER = 1


# ==================================================
# Small helpers
# ==================================================
def metadata_scalar(metadata, key, default=None):
    """Read a scalar/string value from an np.load metadata object."""
    if key not in metadata.files:
        return default

    value = metadata[key]

    if np.ndim(value) == 0:
        try:
            return value.item()
        except ValueError:
            return value

    if np.size(value) == 1:
        try:
            return np.ravel(value)[0].item()
        except (AttributeError, ValueError):
            return np.ravel(value)[0]

    return value


def estimate_cube_symmetric_limit(
    cube,
    percentile=99.0,
    sample_frames=15,
):
    """
    Estimate a symmetric display limit from a small set of frames.

    This avoids creating a full-size np.abs(cube) array for a large memmap.
    """
    nt = cube.shape[0]

    sample_frames = max(int(sample_frames), 1)
    count = min(sample_frames, nt)
    indices = np.unique(
        np.linspace(0, nt - 1, count, dtype=int)
    )

    finite_parts = []

    for frame_index in indices:
        frame = np.asarray(cube[frame_index], dtype=np.float32)
        finite = frame[np.isfinite(frame)]
        if finite.size > 0:
            finite_parts.append(np.abs(finite))

    if not finite_parts:
        return 1.0

    values = np.concatenate(finite_parts)
    vmax = float(np.nanpercentile(values, percentile))

    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    return vmax


def save_basic_array_metadata(
    out_dir,
    times_s,
    times_min,
    x_km,
    y_km,
):
    np.save(out_dir / "times_s.npy", times_s)
    np.save(out_dir / "times_min.npy", times_min)
    np.save(out_dir / "x_km.npy", x_km)
    np.save(out_dir / "y_km.npy", y_km)
    np.save(out_dir / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)
    np.save(out_dir / "first_frame_cut_angles_deg.npy", PLOT_CUT_ANGLES_DEG)
    np.save(
        out_dir / "profile_distances_km.npy",
        np.asarray(PROFILE_DISTANCES_KM, dtype=np.float64),
    )


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(
    wave_cube,
    angle_deg,
    dx_km,
    dy_km,
    width_pixels=3,
    interpolation_order=1,
):
    """
    Create a directional keogram through the center of an image cube.

    Parameters
    ----------
    wave_cube : array-like, shape (time, y, x)
        May be a NumPy memmap.
    angle_deg : float
        0 degrees  = +X / E-W cut
        90 degrees = +Y / N-S cut (image-down direction)
    dx_km, dy_km : float
        Physical pixel sizes.
    width_pixels : int
        Number of parallel strips averaged perpendicular to the cut.
    interpolation_order : int
        scipy.ndimage.map_coordinates interpolation order.

    Returns
    -------
    keogram : ndarray, shape (time, distance)
    distance_km : ndarray
        Signed physical distance from image center along the cut.
    """
    nt, ny, nx = wave_cube.shape

    width_pixels = max(int(width_pixels), 1)

    theta = np.deg2rad(float(angle_deg))

    # Unit vector along the cut, in pixel-coordinate components.
    ux = np.cos(theta)
    uy = np.sin(theta)

    # Unit vector perpendicular to the cut, used for width averaging.
    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (nx - 1) / 2.0
    cy = (ny - 1) / 2.0

    # This is sufficient for the square BLO cubes and preserves the sampling
    # convention used in the earlier PKL-based keogram script.
    half_length_pix = min(nx, ny) // 2

    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
        dtype=np.float64,
    )

    # Make offsets symmetric even if an even width is selected.
    offsets = (
        np.arange(width_pixels, dtype=np.float64)
        - (width_pixels - 1) / 2.0
    )

    # Physical distance corresponding to one pixel-step along this cut.
    # For dx = dy = 1 km this is exactly 1 km at every angle.
    km_per_along_step = np.hypot(
        ux * float(dx_km),
        uy * float(dy_km),
    )
    distance_km = distances_pix * km_per_along_step

    # Precompute all sample coordinates for this angle because they are the
    # same for every time frame.
    sample_x_by_offset = []
    sample_y_by_offset = []
    valid_by_offset = []

    for off in offsets:
        x_sample = cx + distances_pix * ux + off * px
        y_sample = cy + distances_pix * uy + off * py

        valid = (
            (x_sample >= 0)
            & (x_sample <= nx - 1)
            & (y_sample >= 0)
            & (y_sample <= ny - 1)
        )

        sample_x_by_offset.append(x_sample)
        sample_y_by_offset.append(y_sample)
        valid_by_offset.append(valid)

    keogram = np.full(
        (nt, distances_pix.size),
        np.nan,
        dtype=np.float32,
    )

    for time_index in range(nt):
        frame = np.asarray(wave_cube[time_index])

        sampled_slices = np.full(
            (width_pixels, distances_pix.size),
            np.nan,
            dtype=np.float32,
        )

        for offset_index in range(width_pixels):
            x_sample = sample_x_by_offset[offset_index]
            y_sample = sample_y_by_offset[offset_index]
            valid = valid_by_offset[offset_index]

            if not np.any(valid):
                continue

            sampled_slices[offset_index, valid] = map_coordinates(
                frame,
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=interpolation_order,
                mode="nearest",
                prefilter=(interpolation_order > 1),
            ).astype(np.float32, copy=False)

        finite_samples = np.isfinite(sampled_slices)
        sample_count = np.sum(finite_samples, axis=0)
        sample_sum = np.nansum(sampled_slices, axis=0)

        valid_count = sample_count > 0
        keogram[time_index, valid_count] = (
            sample_sum[valid_count] / sample_count[valid_count]
        ).astype(np.float32, copy=False)

    return keogram, distance_km


# ==================================================
# Plot helpers
# ==================================================
def save_cube_animation(
    wave_cube,
    times_min,
    x_km,
    y_km,
    out_file_mp4,
    out_file_gif,
    vmax,
):
    nt = wave_cube.shape[0]

    fig, ax = plt.subplots(figsize=(7, 6))

    im = ax.imshow(
        wave_cube[0],
        origin="upper",
        cmap=CMAP,
        vmin=-vmax,
        vmax=vmax,
        extent=[
            x_km.min(),
            x_km.max(),
            y_km.max(),
            y_km.min(),
        ],
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Preprocessed perturbation amplitude")

    title = ax.set_title("")
    ax.set_xlabel("E-W distance from image center (km)")
    ax.set_ylabel("N-S image-coordinate distance from center (km)")
    ax.set_aspect("equal")

    def update(frame_index):
        im.set_data(wave_cube[frame_index])
        title.set_text(
            "BLO Final Preprocessed Perturbation | "
            f"Frame {frame_index}/{nt - 1} | "
            f"t = {times_min[frame_index]:.1f} min"
        )
        return im, title

    animation = FuncAnimation(
        fig,
        update,
        frames=nt,
        interval=1000 / FPS,
        blit=False,
    )

    saved_file = None

    try:
        writer = FFMpegWriter(
            fps=FPS,
            bitrate=3000,
        )
        animation.save(
            out_file_mp4,
            writer=writer,
            dpi=DPI,
        )
        saved_file = out_file_mp4
    except Exception as exc:
        print("Could not save MP4 with ffmpeg; falling back to GIF.")
        print(f"Reason: {exc}")
        writer = PillowWriter(fps=FPS)
        animation.save(
            out_file_gif,
            writer=writer,
            dpi=DPI,
        )
        saved_file = out_file_gif

    plt.close(fig)
    return saved_file


def save_frame_image(
    frame,
    output_file,
    title_text,
    x_km,
    y_km,
    vmax,
):
    fig, ax = plt.subplots(figsize=(7, 6))

    image = ax.imshow(
        frame,
        origin="upper",
        cmap=CMAP,
        vmin=-vmax,
        vmax=vmax,
        extent=[
            x_km.min(),
            x_km.max(),
            y_km.max(),
            y_km.min(),
        ],
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Preprocessed perturbation amplitude")

    ax.set_title(title_text)
    ax.set_xlabel("E-W distance from image center (km)")
    ax.set_ylabel("N-S image-coordinate distance from center (km)")
    ax.set_aspect("equal")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_first_frame_with_cuts(
    frame,
    output_file,
    x_km,
    y_km,
    vmax,
):
    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(
        frame,
        origin="upper",
        cmap=CMAP,
        vmin=-vmax,
        vmax=vmax,
        extent=[
            x_km.min(),
            x_km.max(),
            y_km.max(),
            y_km.min(),
        ],
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Preprocessed perturbation amplitude")

    # The preprocessing metadata is centered: x=0, y=0 at image center.
    half_length_km = min(
        max(abs(float(x_km.min())), abs(float(x_km.max()))),
        max(abs(float(y_km.min())), abs(float(y_km.max()))),
    )

    for angle_deg in PLOT_CUT_ANGLES_DEG:
        theta = np.deg2rad(float(angle_deg))

        dx_line = half_length_km * np.cos(theta)
        dy_line = half_length_km * np.sin(theta)

        ax.plot(
            [-dx_line, +dx_line],
            [-dy_line, +dy_line],
            linewidth=1.4,
            alpha=0.85,
            label=f"{angle_deg:.0f}°",
        )

    ax.scatter([0.0], [0.0], s=22, c="black", zorder=5)

    ax.set_title(
        "First Final-Preprocessed Frame with Directional Keogram Cuts"
    )
    ax.set_xlabel("E-W distance from image center (km)")
    ax.set_ylabel("N-S image-coordinate distance from center (km)")
    ax.set_aspect("equal")

    ax.legend(
        loc="upper right",
        fontsize=7,
        ncol=2,
        framealpha=0.8,
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_keogram_figure(
    keogram,
    distance_km,
    times_min,
    angle_deg,
    output_file,
):
    visible_time_mask = (
        (times_min >= KEOGRAM_TIME_MIN_MIN)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]

    finite_values = visible_keogram[np.isfinite(visible_keogram)]

    if finite_values.size > 0:
        vmax = float(
            np.nanpercentile(
                np.abs(finite_values),
                KEOGRAM_COLOR_PERCENTILE,
            )
        )
    else:
        vmax = 1.0

    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(9, 5))

    image = ax.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=-vmax,
        vmax=vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Preprocessed perturbation amplitude")

    ax.set_title(
        f"BLO Final-Preprocessed Directional Keogram: {angle_deg:.0f}°"
    )
    ax.set_xlabel("Elapsed time from PKL interval start (min)")
    ax.set_ylabel("Distance along slice from image center (km)")

    ax.set_xlim(
        KEOGRAM_TIME_MIN_MIN,
        KEOGRAM_TIME_MIN_MAX,
    )
    ax.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = int(
            np.argmin(np.abs(distance_km - requested_distance))
        )
        actual_distance = distance_km[distance_index]

        ax.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_profile_figure_and_array(
    keogram,
    distance_km,
    times_min,
    angle_deg,
    output_png,
    output_npy,
):
    visible_time_mask = (
        (times_min >= KEOGRAM_TIME_MIN_MIN)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    selected_profiles = []
    actual_distances = []

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = int(
            np.argmin(np.abs(distance_km - requested_distance))
        )
        actual_distance = float(distance_km[distance_index])
        profile = np.asarray(keogram[:, distance_index], dtype=np.float32)

        actual_distances.append(actual_distance)
        selected_profiles.append(profile)

        ax.plot(
            times_min,
            profile,
            linewidth=1.3,
            label=f"{actual_distance:+.1f} km",
        )

    ax.axhline(
        0.0,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    requested_text = ", ".join(
        f"{distance:+g}" for distance in PROFILE_DISTANCES_KM
    )

    ax.set_title(
        f"BLO Keogram Profiles ({requested_text} km): {angle_deg:.0f}°"
    )
    ax.set_xlabel("Elapsed time from PKL interval start (min)")
    ax.set_ylabel("Preprocessed perturbation amplitude")
    ax.set_xlim(
        KEOGRAM_TIME_MIN_MIN,
        KEOGRAM_TIME_MIN_MAX,
    )

    finite_parts = []
    for profile in selected_profiles:
        valid = visible_time_mask & np.isfinite(profile)
        if np.any(valid):
            finite_parts.append(profile[valid])

    if finite_parts:
        finite_values = np.concatenate(finite_parts)
        ymin = float(np.nanmin(finite_values))
        ymax = float(np.nanmax(finite_values))
        span = ymax - ymin

        if not np.isfinite(span) or span <= 0:
            span = max(abs(ymin), abs(ymax), 1.0)

        padding = PROFILE_Y_PADDING_FRACTION * span
        ax.set_ylim(ymin - padding, ymax + padding)

    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_png, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    # Columns:
    #   0 = time_min
    #   1..N = profiles in PROFILE_DISTANCES_KM order
    profile_array = np.column_stack(
        [times_min] + selected_profiles
    )
    np.save(output_npy, profile_array)

    return actual_distances


# ==================================================
# Main
# ==================================================
def main():
    if not CUBE_FILE.exists():
        raise FileNotFoundError(
            "Could not find the saved preprocessed cube:\n"
            f"{CUBE_FILE}\n\n"
            "Set CUBE_DIR near the top of this script to the output folder "
            "created by the preprocessing / plane-fit script."
        )

    if not METADATA_FILE.exists():
        raise FileNotFoundError(
            "Could not find the saved cube metadata:\n"
            f"{METADATA_FILE}\n\n"
            "The cube and metadata should be in the same preprocessing "
            "output folder."
        )

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Disk-backed read: does not copy the whole cube into RAM.
    wave_cube = np.load(
        CUBE_FILE,
        mmap_mode="r",
    )

    with np.load(METADATA_FILE, allow_pickle=False) as metadata:
        times_s = np.asarray(metadata["time_seconds"], dtype=np.float64)
        times_min = np.asarray(metadata["time_minutes"], dtype=np.float64)
        x_km = np.asarray(metadata["x_km"], dtype=np.float64)
        y_km = np.asarray(metadata["y_km"], dtype=np.float64)

        dx_km = float(metadata_scalar(metadata, "dx_km", 1.0))
        dy_km = float(metadata_scalar(metadata, "dy_km", 1.0))

        start_local_datetime = str(
            metadata_scalar(metadata, "start_local_datetime", "")
        )
        input_file = str(
            metadata_scalar(metadata, "input_file", "")
        )
        cube_contents = str(
            metadata_scalar(
                metadata,
                "cube_contents",
                "final_preprocessed_perturbation",
            )
        )
        time_reference = str(
            metadata_scalar(metadata, "time_reference", "frame_center")
        )

        if "local_datetime" in metadata.files:
            local_datetime = np.asarray(metadata["local_datetime"], dtype=str)
        else:
            local_datetime = np.array([], dtype=str)

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected saved cube shape (time, y, x), but got "
            f"{wave_cube.shape}."
        )

    nt, ny, nx = wave_cube.shape

    if times_min.size != nt:
        raise ValueError(
            "Metadata time_minutes length does not match cube time axis: "
            f"{times_min.size} vs {nt}."
        )
    if times_s.size != nt:
        raise ValueError(
            "Metadata time_seconds length does not match cube time axis: "
            f"{times_s.size} vs {nt}."
        )
    if x_km.size != nx:
        raise ValueError(
            "Metadata x_km length does not match cube X dimension: "
            f"{x_km.size} vs {nx}."
        )
    if y_km.size != ny:
        raise ValueError(
            "Metadata y_km length does not match cube Y dimension: "
            f"{y_km.size} vs {ny}."
        )

    print("=" * 80)
    print("BLO KEOGRAMS FROM SAVED PREPROCESSED CUBE")
    print(f"Cube file:     {CUBE_FILE}")
    print(f"Metadata file: {METADATA_FILE}")
    print(f"Output folder: {OUT_DIR}")
    print(f"Cube shape:    {wave_cube.shape}")
    print(f"Cube dtype:    {wave_cube.dtype}")
    print(f"Cube contents: {cube_contents}")
    print(f"Original PKL:  {input_file}")
    print(f"Start local:   {start_local_datetime}")
    print(f"Time reference:{time_reference}")
    print(
        f"Time range:    {times_min[0]:.1f} to "
        f"{times_min[-1]:.1f} min"
    )
    print(f"Pixel size:    dx={dx_km:g} km, dy={dy_km:g} km")
    print(
        f"Keogram angles:{KEOGRAM_ANGLES_DEG[0]:g} to "
        f"{KEOGRAM_ANGLES_DEG[-1]:g} deg, "
        f"{len(KEOGRAM_ANGLES_DEG)} total"
    )
    print(f"Strip width:   {KEOGRAM_WIDTH_PIXELS} pixels")

    # --------------------------------------------------
    # Common display scale for cube/frame products
    # --------------------------------------------------
    cube_vmax = estimate_cube_symmetric_limit(
        wave_cube,
        percentile=CUBE_COLOR_PERCENTILE,
        sample_frames=CUBE_COLOR_SAMPLE_FRAMES,
    )

    # --------------------------------------------------
    # Save metadata arrays first
    # --------------------------------------------------
    save_basic_array_metadata(
        OUT_DIR,
        times_s,
        times_min,
        x_km,
        y_km,
    )

    np.savez(
        OUT_DIR / "keogram_metadata.npz",
        source_cube_file=str(CUBE_FILE),
        source_metadata_file=str(METADATA_FILE),
        original_input_file=input_file,
        cube_contents=cube_contents,
        start_local_datetime=start_local_datetime,
        local_datetime=local_datetime,
        time_reference=time_reference,
        time_seconds=times_s,
        time_minutes=times_min,
        x_km=x_km,
        y_km=y_km,
        dx_km=dx_km,
        dy_km=dy_km,
        keogram_angles_deg=KEOGRAM_ANGLES_DEG,
        first_frame_cut_angles_deg=PLOT_CUT_ANGLES_DEG,
        keogram_width_pixels=KEOGRAM_WIDTH_PIXELS,
        profile_distances_km=np.asarray(PROFILE_DISTANCES_KM),
        interpolation_order=INTERPOLATION_ORDER,
        angle_convention=(
            "0 deg = +X/right; +90 deg = +Y/image down; "
            "-90 deg = -Y/image up"
        ),
        distance_convention="0 km = image center",
    )

    with open(OUT_DIR / "Date_and_Start_Time.txt", "w") as file:
        file.write(f"Start local datetime: {start_local_datetime}\n")
        file.write(f"Original PKL: {input_file}\n")
        file.write(f"Time reference: {time_reference}\n")
        file.write(
            f"First frame-center elapsed time: {times_min[0]:.6f} min\n"
        )

    # --------------------------------------------------
    # Optional animation / frame products
    # --------------------------------------------------
    saved_animation_file = None

    if MAKE_ANIMATION:
        print("Saving preprocessed-cube animation...")
        saved_animation_file = save_cube_animation(
            wave_cube=wave_cube,
            times_min=times_min,
            x_km=x_km,
            y_km=y_km,
            out_file_mp4=OUT_DIR / "BLO_preprocessed_perturbation_animation.mp4",
            out_file_gif=OUT_DIR / "BLO_preprocessed_perturbation_animation.gif",
            vmax=cube_vmax,
        )

    if SAVE_FIRST_AND_FINAL_FRAMES:
        save_frame_image(
            frame=wave_cube[0],
            output_file=OUT_DIR / "first_preprocessed_perturbation_frame.png",
            title_text=(
                "First Preprocessed BLO Perturbation Frame\n"
                f"t = {times_min[0]:.1f} min"
            ),
            x_km=x_km,
            y_km=y_km,
            vmax=cube_vmax,
        )

        save_frame_image(
            frame=wave_cube[-1],
            output_file=OUT_DIR / "final_preprocessed_perturbation_frame.png",
            title_text=(
                "Final Preprocessed BLO Perturbation Frame\n"
                f"t = {times_min[-1]:.1f} min"
            ),
            x_km=x_km,
            y_km=y_km,
            vmax=cube_vmax,
        )

    if SAVE_FIRST_FRAME_WITH_CUTS:
        save_first_frame_with_cuts(
            frame=wave_cube[0],
            output_file=(
                OUT_DIR
                / "first_preprocessed_frame_with_keogram_cuts_every_15deg.png"
            ),
            x_km=x_km,
            y_km=y_km,
            vmax=cube_vmax,
        )

    # --------------------------------------------------
    # Make all directional keograms
    # --------------------------------------------------
    saved_keograms = []
    saved_profile_plots = []

    for angle_number, angle_deg in enumerate(KEOGRAM_ANGLES_DEG, start=1):
        print(
            f"Creating keogram {angle_number}/{len(KEOGRAM_ANGLES_DEG)}: "
            f"{angle_deg:.0f} deg"
        )

        keogram, distance_km = make_directional_keogram(
            wave_cube=wave_cube,
            angle_deg=angle_deg,
            dx_km=dx_km,
            dy_km=dy_km,
            width_pixels=KEOGRAM_WIDTH_PIXELS,
            interpolation_order=INTERPOLATION_ORDER,
        )

        # Keep the old filename structure so downstream analysis scripts are
        # easy to adapt/reuse.
        out_png = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.png"
        out_npy = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.npy"

        save_keogram_figure(
            keogram=keogram,
            distance_km=distance_km,
            times_min=times_min,
            angle_deg=angle_deg,
            output_file=out_png,
        )

        np.save(out_npy, keogram)
        saved_keograms.append(out_png)

        profile_text = "_".join(
            [
                ("minus" + f"{abs(distance):g}")
                if distance < 0
                else ("plus" + f"{distance:g}")
                if distance > 0
                else "0"
                for distance in PROFILE_DISTANCES_KM
            ]
        )

        out_profile_png = (
            OUT_DIR
            / f"BLO_keogram_profiles_{profile_text}km_"
              f"angle_{angle_deg:03.0f}.png"
        )
        out_profile_npy = (
            OUT_DIR
            / f"BLO_keogram_profiles_{profile_text}km_"
              f"angle_{angle_deg:03.0f}.npy"
        )

        actual_profile_distances = save_profile_figure_and_array(
            keogram=keogram,
            distance_km=distance_km,
            times_min=times_min,
            angle_deg=angle_deg,
            output_png=out_profile_png,
            output_npy=out_profile_npy,
        )

        saved_profile_plots.append(out_profile_png)

        # The distance axis is the same for a given angle, and can differ
        # slightly with angle only if dx != dy. Save it explicitly for every
        # angle so nothing has to be reconstructed later.
        np.save(
            OUT_DIR / f"BLO_keogram_distance_km_angle_{angle_deg:03.0f}.npy",
            distance_km,
        )

        np.savez(
            OUT_DIR / f"BLO_keogram_info_angle_{angle_deg:03.0f}.npz",
            angle_deg=float(angle_deg),
            distance_km=distance_km,
            times_min=times_min,
            requested_profile_distances_km=np.asarray(PROFILE_DISTANCES_KM),
            actual_profile_distances_km=np.asarray(actual_profile_distances),
            keogram_width_pixels=KEOGRAM_WIDTH_PIXELS,
            dx_km=dx_km,
            dy_km=dy_km,
        )

    # --------------------------------------------------
    # Print outputs
    # --------------------------------------------------
    print("\n" + "=" * 80)
    print("Finished creating keograms from the saved preprocessed cube.")
    print(f"Output folder: {OUT_DIR}")
    print(f"Number of directional keograms: {len(saved_keograms)}")
    print(f"Number of profile plots:        {len(saved_profile_plots)}")
    print(
        f"Keogram display time range:     {KEOGRAM_TIME_MIN_MIN:g} to "
        f"{KEOGRAM_TIME_MIN_MAX:g} min"
    )
    print(
        f"Keogram distance range:         {KEOGRAM_DISTANCE_MIN:g} to "
        f"{KEOGRAM_DISTANCE_MAX:g} km"
    )
    print(f"Profile distances requested:    {PROFILE_DISTANCES_KM}")
    print(
        "Time axis comes directly from the preprocessing metadata "
        f"({time_reference})."
    )

    if saved_animation_file is not None:
        print(f"Animation: {saved_animation_file}")


if __name__ == "__main__":
    main()


BLO KEOGRAMS FROM SAVED PREPROCESSED CUBE
Cube file:     /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/preprocessed_perturbation_cube.npy
Metadata file: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/preprocessed_perturbation_cube_metadata.npz
Output folder: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/keogram_outputs
Cube shape:    (280, 1000, 1000)
Cube dtype:    float32
Cube contents: final_preprocessed_perturbation
Original PKL:  /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Start local:   2025-03-25 20:29:27
Time reference:frame_center
Time range:    1.0 to 559.0 min
Pixel size:    dx=1 km, dy=1 km
Keogram angles:0 to 175 deg, 36 total
Strip width:   7 pixels
Saving preprocessed-cube animation...
Creating keogram 1/36: 0 deg
Creating keogram 2/36: 5 deg
Creating keogram 3/36: 10 deg
Creating keogram 4/36: 15 deg
Creating keo

## Correlation Analysis, 3hr-5hr windows

In [17]:
# -*- coding: utf-8 -*-
"""
Analyze BLO directional keograms created from the saved preprocessed cube.

This is a simplified follow-up to the keogram-generation script.

Main changes relative to the older correlation-analysis script:
    1. Reads the new keogram_outputs folder directly.
    2. Uses the full 23:00–05:00 SAVED LOCAL CLOCK TIME interval and runs
       every 3-, 4-, and 5-hour window that fits inside it, with window
       starts every 30 minutes. This gives 15 total windows:
           7 three-hour windows
           5 four-hour windows
           3 five-hour windows

    3. Analyzes only keogram angles from 30° through 150° in 5° steps.
    4. Uses the explicitly saved distance axis for every keogram angle.
    5. Optionally removes the planar bias estimated from the saved centered
       2-hour running-average slope vector.
    6. Creates one folder per time window plus a root-level comparison table.

           keogram_outputs/
               analysis_3_4_5hour_windows/
                   analysis_3hr_23_to_02/
                       plots/
                       data/
                       all_angle_slope_summary.csv
                       phase_speed_vs_angle.png
                       r_squared_vs_angle.png
                       processing_log.txt
                   ...
                   window_summary_table.csv
                   window_summary_table.png

No per-angle directory hierarchy is created. Angle and dataset names are
encoded directly in each filename.

Correlation method:
    - For each directional keogram, use the 0-km row as the reference.
    - Correlate rows from 0 to 250 km against the reference for time shifts.
    - Find the maximum-correlation ridge.
    - Fit shift_minutes = slope * distance_km + intercept.
    - Equivalent signed phase speed = -(1 / slope) * 1000 / 60 m/s.

2-hour bias correction:
    The preprocessing/plane-fit script saves a centered 2-hour running mean of
    the preprocessed plane-slope components in amplitude/km. For a keogram cut
    at angle theta and signed distance d:

        slope_along_cut(t) = sx(t)*cos(theta) + sy(t)*sin(theta)
        bias(t, d) = slope_along_cut(t) * d

    Subtracting this term removes the slowly varying planar component while
    retaining the faster perturbation structure.
"""

import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks


# ==================================================
# Input / output settings
# ==================================================
# Change only this directory for a different night.
KEOGRAM_DIR = (
    Path.home()
    / "Downloads"
    / "pkl"
    / "BLO_plane_fit_batch_outputs"
    / "2025"
    / "processed_data_full_03_25_2025"
    / "keogram_outputs"
)

# The plane-fit results sit one level above keogram_outputs.
PLANE_RESULTS_FILE = KEOGRAM_DIR.parent / "plane_fit_results.npz"
KEOGRAM_METADATA_FILE = KEOGRAM_DIR / "keogram_metadata.npz"

# Output structure for all 3-, 4-, and 5-hour windows inside 23:00–05:00.
ANALYSIS_ROOT = KEOGRAM_DIR / "analysis_3_4_5hour_windows"
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)


# ==================================================
# Analysis settings
# ==================================================
# Analyze only the requested angle range.
KEOGRAM_ANGLES_DEG = np.arange(30, 151, 5, dtype=int)

# Every 3-, 4-, and 5-hour local-clock window fully contained in 23:00–05:00.
# Start times are spaced by 30 minutes.
# label, start_hour, end_hour
ANALYSIS_WINDOWS = [
    # 3-hour windows
    ("3hr_23_00_to_02_00", 23.0, 2.0),
    ("3hr_23_30_to_02_30", 23.5, 2.5),
    ("3hr_00_00_to_03_00", 0.0, 3.0),
    ("3hr_00_30_to_03_30", 0.5, 3.5),
    ("3hr_01_00_to_04_00", 1.0, 4.0),
    ("3hr_01_30_to_04_30", 1.5, 4.5),
    ("3hr_02_00_to_05_00", 2.0, 5.0),

    # 4-hour windows
    ("4hr_23_00_to_03_00", 23.0, 3.0),
    ("4hr_23_30_to_03_30", 23.5, 3.5),
    ("4hr_00_00_to_04_00", 0.0, 4.0),
    ("4hr_00_30_to_04_30", 0.5, 4.5),
    ("4hr_01_00_to_05_00", 1.0, 5.0),

    # 5-hour windows
    ("5hr_23_00_to_04_00", 23.0, 4.0),
    ("5hr_23_30_to_04_30", 23.5, 4.5),
    ("5hr_00_00_to_05_00", 0.0, 5.0),
]

# Distance range included in correlation map / ridge fit.
CORRELATION_DISTANCE_MIN_KM = 0.0
CORRELATION_DISTANCE_MAX_KM = 250.0
SLOPE_FIT_DISTANCE_MIN_KM = 0.0
SLOPE_FIT_DISTANCE_MAX_KM = 250.0

# Shift range for correlation map and ridge fit.
SHIFT_MIN_MINUTES = -50.0
SHIFT_MAX_MINUTES = 50.0

# Wider shift range used only for the 0-km autocorrelation profile.
ZERO_KM_PROFILE_SHIFT_MIN_MINUTES = -200.0
ZERO_KM_PROFILE_SHIFT_MAX_MINUTES = 200.0

MIN_VALID_CORRELATION_POINTS = 20
MIN_CORRELATION_FOR_SLOPE = 0.50

# Ridge cleanup.
REMOVE_RIDGE_OUTLIERS = True
RIDGE_MAXIMUM_JUMP_MINUTES = 20.0
RIDGE_OUTLIER_SIGMA = 3.0
RIDGE_OUTLIER_MAX_ITERATIONS = 10
RIDGE_OUTLIER_MIN_POINTS = 3

# Optional NaN-aware 2-D smoothing of the selected keogram.
APPLY_2D_SMOOTHING = True
SMOOTH_SIGMA_TIME_PIXELS = 1.5
SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0
SMOOTH_MODE = "nearest"
SMOOTH_WEIGHT_TOLERANCE = 1.0e-12

# Compare the processed keogram with the same keogram after removal of the
# centered 2-hour plane-slope background.
APPLY_2HR_SLOPE_BIAS_SUBTRACTION = True

# Plot settings.
DPI = 200
KEOGRAM_CMAP = "RdBu_r"
CORRELATION_CMAP = "RdBu_r"
KEOGRAM_COLOR_PERCENTILE = 99.0
CORRELATION_VMIN = 0.70
CORRELATION_VMAX = 1.00

# Phase-speed-vs-angle display range.
PHASE_SPEED_PLOT_ANGLE_MIN_DEG = 30.0
PHASE_SPEED_PLOT_ANGLE_MAX_DEG = 150.0

# Y range is determined dynamically from the displayed phase-speed values.
PHASE_SPEED_Y_PADDING_FRACTION = 0.10
PHASE_SPEED_Y_MIN_PADDING_M_PER_S = 50.0

# Local maxima on the 0-km autocorrelation profile.
LOCAL_MAXIMUM_MIN_PROMINENCE = 0.0
LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS = 1


# ==================================================
# Filename helpers
# ==================================================
def input_angle_label(angle_deg):
    """Match filenames written by the keogram-generation script."""
    return f"{int(angle_deg):03d}"


def output_angle_label(angle_deg):
    """Unambiguous filename-safe output label."""
    angle_deg = int(angle_deg)
    if angle_deg < 0:
        return f"m{abs(angle_deg):03d}"
    return f"p{angle_deg:03d}"


# ==================================================
# Time helpers
# ==================================================
def metadata_scalar(metadata, key, default=None):
    if key not in metadata.files:
        return default

    value = metadata[key]
    if np.ndim(value) == 0:
        try:
            return value.item()
        except ValueError:
            return value

    if np.size(value) == 1:
        value = np.ravel(value)[0]
        try:
            return value.item()
        except (AttributeError, ValueError):
            return value

    return value


def parse_iso_datetime(text):
    text = str(text).strip()
    try:
        return datetime.fromisoformat(text)
    except ValueError as exc:
        raise ValueError(f"Could not parse saved local datetime: {text!r}") from exc


def build_local_datetimes(metadata, times_min):
    """Read exact saved frame-center datetimes, with a start-time fallback."""
    if "local_datetime" in metadata.files:
        local_text = np.asarray(metadata["local_datetime"], dtype=str)
        if len(local_text) == len(times_min):
            return np.array(
                [parse_iso_datetime(value) for value in local_text],
                dtype=object,
            )

    start_text = str(metadata_scalar(metadata, "start_local_datetime", ""))
    if not start_text:
        raise ValueError(
            "keogram_metadata.npz has neither a usable local_datetime array "
            "nor start_local_datetime."
        )

    start_datetime = parse_iso_datetime(start_text)
    return np.array(
        [start_datetime + timedelta(minutes=float(value)) for value in times_min],
        dtype=object,
    )


def local_clock_hours(local_datetimes):
    return np.array(
        [
            dt.hour
            + dt.minute / 60.0
            + dt.second / 3600.0
            + dt.microsecond / 3.6e9
            for dt in local_datetimes
        ],
        dtype=np.float64,
    )


def make_local_time_mask(local_hours, start_hour, end_hour):
    """Support both ordinary and across-midnight clock windows."""
    start_hour = float(start_hour) % 24.0
    end_hour = float(end_hour) % 24.0

    if np.isclose(start_hour, end_hour):
        return np.ones(local_hours.shape, dtype=bool)

    if start_hour < end_hour:
        return (local_hours >= start_hour) & (local_hours <= end_hour)

    return (local_hours >= start_hour) | (local_hours <= end_hour)


def format_clock_hour(hour_value):
    total_minutes = int(round(float(hour_value) * 60.0))
    hour = (total_minutes // 60) % 24
    minute = total_minutes % 60
    return f"{hour:02d}:{minute:02d}"


def minutes_from_window_start(local_hours, start_hour):
    """Convert local clock hours to a continuous axis beginning at 0 min."""
    local_hours = np.asarray(local_hours, dtype=np.float64)
    return ((local_hours - float(start_hour)) % 24.0) * 60.0


def build_window_plot_ticks(start_hour, end_hour):
    """Return 30-minute tick positions/labels for one requested analysis window."""
    duration_hours = (float(end_hour) - float(start_hour)) % 24.0
    if np.isclose(duration_hours, 0.0):
        duration_hours = 24.0

    duration_minutes = 60.0 * duration_hours
    tick_minutes = np.arange(0.0, duration_minutes + 0.1, 30.0)
    tick_labels = [
        format_clock_hour(float(start_hour) + minute / 60.0)
        for minute in tick_minutes
    ]
    return duration_minutes, tick_minutes, tick_labels


# ==================================================
# NaN-aware smoothing
# ==================================================
def smooth_keogram_2d(
    data,
    sigma_time,
    sigma_distance,
    mode="nearest",
    weight_tolerance=1.0e-12,
):
    data = np.asarray(data, dtype=np.float64)

    if data.ndim != 2:
        raise ValueError("Keogram must be two-dimensional.")

    if sigma_time < 0 or sigma_distance < 0:
        raise ValueError("Smoothing sigma values must be nonnegative.")

    if sigma_time == 0 and sigma_distance == 0:
        return data.copy()

    valid = np.isfinite(data)
    filled = np.where(valid, data, 0.0)
    weights = valid.astype(np.float64)

    sigma = (float(sigma_time), float(sigma_distance))

    smoothed_values = gaussian_filter(filled, sigma=sigma, mode=mode)
    smoothed_weights = gaussian_filter(weights, sigma=sigma, mode=mode)

    result = np.full(data.shape, np.nan, dtype=np.float64)
    good = smoothed_weights > weight_tolerance
    result[good] = smoothed_values[good] / smoothed_weights[good]
    return result


# ==================================================
# Load / interpolate saved 2-hour slope background
# ==================================================
def interpolate_finite(source_time, source_values, target_time):
    source_time = np.asarray(source_time, dtype=np.float64)
    source_values = np.asarray(source_values, dtype=np.float64)
    target_time = np.asarray(target_time, dtype=np.float64)

    valid = np.isfinite(source_time) & np.isfinite(source_values)
    if np.count_nonzero(valid) < 2:
        return np.full(target_time.shape, np.nan, dtype=np.float64)

    x = source_time[valid]
    y = source_values[valid]
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    return np.interp(target_time, x, y, left=np.nan, right=np.nan)


def load_two_hour_slope_background(plane_results_file, target_times_min):
    if not plane_results_file.exists():
        raise FileNotFoundError(
            "Could not find plane_fit_results.npz needed for 2-hour slope "
            f"bias subtraction:\n{plane_results_file}"
        )

    with np.load(plane_results_file, allow_pickle=False) as results:
        required = [
            "time_minutes",
            "preprocessed_running_mean_x_slope_amplitude_per_km",
            "preprocessed_running_mean_y_slope_amplitude_per_km",
        ]

        missing = [key for key in required if key not in results.files]
        if missing:
            raise KeyError(
                "plane_fit_results.npz is missing the saved 2-hour running-"
                "average slope fields: " + ", ".join(missing)
            )

        source_time = np.asarray(results["time_minutes"], dtype=np.float64)
        source_x = np.asarray(
            results["preprocessed_running_mean_x_slope_amplitude_per_km"],
            dtype=np.float64,
        )
        source_y = np.asarray(
            results["preprocessed_running_mean_y_slope_amplitude_per_km"],
            dtype=np.float64,
        )

        window_hours = float(
            metadata_scalar(results, "running_average_window_hours", 2.0)
        )

    x_target = interpolate_finite(source_time, source_x, target_times_min)
    y_target = interpolate_finite(source_time, source_y, target_times_min)

    return x_target, y_target, window_hours


def create_directional_two_hour_bias(
    slope_x_per_km,
    slope_y_per_km,
    distance_km,
    angle_deg,
):
    """Project the saved 2-hour plane slope onto one directional cut."""
    theta = np.deg2rad(float(angle_deg))

    slope_along_cut_per_km = (
        np.asarray(slope_x_per_km, dtype=np.float64) * np.cos(theta)
        + np.asarray(slope_y_per_km, dtype=np.float64) * np.sin(theta)
    )

    bias_keogram = (
        slope_along_cut_per_km[:, np.newaxis]
        * np.asarray(distance_km, dtype=np.float64)[np.newaxis, :]
    )

    return bias_keogram, slope_along_cut_per_km


# ==================================================
# Correlation helpers
# ==================================================
def nan_pearson_correlation(reference, comparison, minimum_points=20):
    reference = np.asarray(reference, dtype=np.float64)
    comparison = np.asarray(comparison, dtype=np.float64)

    valid = np.isfinite(reference) & np.isfinite(comparison)
    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    x = reference[valid]
    y = comparison[valid]

    x = x - np.mean(x)
    y = y - np.mean(y)

    denominator = np.sqrt(np.sum(x * x) * np.sum(y * y))
    if not np.isfinite(denominator) or denominator <= 0:
        return np.nan

    return float(np.clip(np.sum(x * y) / denominator, -1.0, 1.0))


def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    reference_row = np.asarray(reference_row, dtype=np.float64)
    comparison_row = np.asarray(comparison_row, dtype=np.float64)

    if reference_row.shape != comparison_row.shape:
        raise ValueError("Reference and comparison rows must have same shape.")

    ntime = len(reference_row)
    if abs(int(shift_pixels)) >= ntime:
        return np.nan

    shift_pixels = int(shift_pixels)

    if shift_pixels > 0:
        reference_overlap = reference_row[shift_pixels:]
        comparison_overlap = comparison_row[:-shift_pixels]
    elif shift_pixels < 0:
        shift_abs = abs(shift_pixels)
        reference_overlap = reference_row[:-shift_abs]
        comparison_overlap = comparison_row[shift_abs:]
    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference_overlap,
        comparison_overlap,
        minimum_points=minimum_points,
    )


def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    keogram = np.asarray(keogram, dtype=np.float64)
    distance_km = np.asarray(distance_km, dtype=np.float64)
    shifts_pixels = np.asarray(shifts_pixels, dtype=int)

    reference_index = int(
        np.argmin(np.abs(distance_km - float(reference_distance_km)))
    )
    selected_indices = np.flatnonzero(selected_distance_mask)
    selected_distances = distance_km[selected_indices]

    reference_row = keogram[:, reference_index]

    correlation_map = np.full(
        (len(selected_indices), len(shifts_pixels)),
        np.nan,
        dtype=np.float64,
    )

    for output_row, keogram_row in enumerate(selected_indices):
        comparison_row = keogram[:, keogram_row]
        for shift_index, shift_pixels in enumerate(shifts_pixels):
            correlation_map[output_row, shift_index] = (
                calculate_shifted_correlation(
                    reference_row,
                    comparison_row,
                    int(shift_pixels),
                    minimum_points=minimum_points,
                )
            )

    return correlation_map, selected_distances, reference_index


def find_best_shifts(correlation_map, shifts_minutes):
    correlation_map = np.asarray(correlation_map, dtype=np.float64)
    shifts_minutes = np.asarray(shifts_minutes, dtype=np.float64)

    best_shift = np.full(correlation_map.shape[0], np.nan, dtype=np.float64)
    maximum_correlation = np.full(
        correlation_map.shape[0], np.nan, dtype=np.float64
    )

    for row_index, row in enumerate(correlation_map):
        finite = np.isfinite(row)
        if not np.any(finite):
            continue

        finite_indices = np.flatnonzero(finite)
        full_index = finite_indices[int(np.argmax(row[finite]))]
        best_shift[row_index] = shifts_minutes[full_index]
        maximum_correlation[row_index] = row[full_index]

    return best_shift, maximum_correlation


def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
    minimum_correlation=0.5,
    distance_min_km=None,
    distance_max_km=None,
    remove_outliers=True,
    maximum_ridge_jump_minutes=20.0,
    outlier_sigma=3.0,
    outlier_max_iterations=10,
    outlier_min_points=3,
):
    distances_km = np.asarray(distances_km, dtype=np.float64)
    best_shift_minutes = np.asarray(best_shift_minutes, dtype=np.float64)
    maximum_correlation = np.asarray(maximum_correlation, dtype=np.float64)

    candidate_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (maximum_correlation >= minimum_correlation)
    )

    if distance_min_km is not None:
        candidate_mask &= distances_km >= float(distance_min_km)
    if distance_max_km is not None:
        candidate_mask &= distances_km <= float(distance_max_km)

    candidate_count = int(np.count_nonzero(candidate_mask))
    if candidate_count < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": candidate_count,
            "number_of_outliers_removed": 0,
            "fit_mask": candidate_mask,
            "outlier_mask": np.zeros_like(candidate_mask, dtype=bool),
        }

    retained_mask = candidate_mask.copy()

    # Retain a continuous ridge branch beginning at the smallest distance.
    if remove_outliers:
        candidate_indices = np.flatnonzero(candidate_mask)
        sorted_indices = candidate_indices[
            np.argsort(distances_km[candidate_indices])
        ]

        continuity_mask = np.zeros_like(candidate_mask, dtype=bool)
        first_index = sorted_indices[0]
        continuity_mask[first_index] = True
        previous_shift = best_shift_minutes[first_index]

        for index in sorted_indices[1:]:
            current_shift = best_shift_minutes[index]
            if abs(current_shift - previous_shift) <= maximum_ridge_jump_minutes:
                continuity_mask[index] = True
                previous_shift = current_shift

        if np.count_nonzero(continuity_mask) >= max(2, outlier_min_points):
            retained_mask = continuity_mask

    # Iterative robust residual rejection.
    if remove_outliers:
        for _ in range(int(outlier_max_iterations)):
            if np.count_nonzero(retained_mask) < max(3, outlier_min_points):
                break

            x = distances_km[retained_mask]
            y = best_shift_minutes[retained_mask]
            trial_slope, trial_intercept = np.polyfit(x, y, 1)
            residual = y - (trial_slope * x + trial_intercept)
            residual_median = np.median(residual)
            mad = np.median(np.abs(residual - residual_median))

            if not np.isfinite(mad) or np.isclose(mad, 0.0):
                break

            robust_sigma = 1.4826 * mad
            keep_local = (
                np.abs(residual - residual_median)
                <= float(outlier_sigma) * robust_sigma
            )

            if np.all(keep_local):
                break

            retained_indices = np.flatnonzero(retained_mask)
            proposed = retained_mask.copy()
            proposed[retained_indices[~keep_local]] = False

            if np.count_nonzero(proposed) < max(2, outlier_min_points):
                break

            retained_mask = proposed

    outlier_mask = candidate_mask & ~retained_mask
    fit_mask = retained_mask

    fit_distances = distances_km[fit_mask]
    fit_shifts = best_shift_minutes[fit_mask]

    if len(fit_distances) < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": len(fit_distances),
            "number_of_outliers_removed": int(np.count_nonzero(outlier_mask)),
            "fit_mask": fit_mask,
            "outlier_mask": outlier_mask,
        }

    slope, intercept = np.polyfit(fit_distances, fit_shifts, 1)
    predicted = slope * fit_distances + intercept

    ss_res = np.sum((fit_shifts - predicted) ** 2)
    ss_tot = np.sum((fit_shifts - np.mean(fit_shifts)) ** 2)
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    if np.isclose(slope, 0.0):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        speed_km_per_min = -1.0 / slope
        speed_m_per_s = speed_km_per_min * 1000.0 / 60.0

    return {
        "slope_min_per_km": float(slope),
        "intercept_minutes": float(intercept),
        "speed_km_per_min": float(speed_km_per_min),
        "speed_m_per_s": float(speed_m_per_s),
        "r_squared": float(r_squared),
        "number_of_fit_points": int(np.count_nonzero(fit_mask)),
        "number_of_outliers_removed": int(np.count_nonzero(outlier_mask)),
        "fit_mask": fit_mask,
        "outlier_mask": outlier_mask,
    }


def find_left_and_right_local_maxima(
    correlation_profile,
    shifts_minutes,
    minimum_prominence=0.0,
    minimum_distance_pixels=1,
):
    correlation_profile = np.asarray(correlation_profile, dtype=np.float64)
    shifts_minutes = np.asarray(shifts_minutes, dtype=np.float64)

    def find_one_side(side_mask):
        indices = np.flatnonzero(side_mask & np.isfinite(correlation_profile))
        if len(indices) == 0:
            return {
                "index": -1,
                "shift_minutes": np.nan,
                "correlation": np.nan,
                "used_fallback": False,
            }

        values = correlation_profile[indices]
        local_indices, _ = find_peaks(
            values,
            prominence=minimum_prominence,
            distance=minimum_distance_pixels,
        )

        used_fallback = False
        if len(local_indices) > 0:
            strongest = local_indices[int(np.nanargmax(values[local_indices]))]
            full_index = int(indices[strongest])
        else:
            used_fallback = True
            full_index = int(indices[int(np.nanargmax(values))])

        return {
            "index": full_index,
            "shift_minutes": float(shifts_minutes[full_index]),
            "correlation": float(correlation_profile[full_index]),
            "used_fallback": used_fallback,
        }

    return {
        "left": find_one_side(shifts_minutes < 0.0),
        "right": find_one_side(shifts_minutes > 0.0),
    }


# ==================================================
# Plot helpers
# ==================================================
def symmetric_keogram_limit(data, percentile=99.0):
    data = np.asarray(data, dtype=np.float64)
    finite = np.abs(data[np.isfinite(data)])
    if finite.size == 0:
        return 1.0

    limit = float(np.nanpercentile(finite, percentile))
    if not np.isfinite(limit) or limit <= 0:
        limit = float(np.nanmax(finite))
    if not np.isfinite(limit) or limit <= 0:
        limit = 1.0
    return limit


def plot_one_dataset_analysis(
    angle_deg,
    dataset_label,
    keogram,
    window_minutes,
    window_duration_minutes,
    window_tick_minutes,
    window_tick_labels,
    time_range_label,
    distance_km,
    selected_distance_mask,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_km_profile,
    zero_profile_shifts_minutes,
    zero_km_peaks,
    output_file,
):
    """Compact four-panel diagnostic figure for one analysis window."""
    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(11, 15.5),
        gridspec_kw={"height_ratios": [1.6, 1.1, 2.4, 1.1]},
    )

    ax_keogram, ax_profiles, ax_map, ax_autocorr = axes

    # --------------------------------------------------
    # Selected local-time keogram
    # --------------------------------------------------
    selected_keogram = keogram[:, selected_distance_mask]
    selected_distance = distance_km[selected_distance_mask]
    abs_limit = symmetric_keogram_limit(
        selected_keogram,
        percentile=KEOGRAM_COLOR_PERCENTILE,
    )

    mesh = ax_keogram.pcolormesh(
        window_minutes,
        selected_distance,
        selected_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-abs_limit,
        vmax=abs_limit,
    )
    cbar = fig.colorbar(mesh, ax=ax_keogram, fraction=0.025, pad=0.02)
    cbar.set_label("Keogram value")

    ax_keogram.set_xlim(0.0, window_duration_minutes)
    ax_keogram.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )
    ax_keogram.set_xticks(window_tick_minutes)
    ax_keogram.set_xticklabels(window_tick_labels)
    ax_keogram.set_xlabel("Local time")
    ax_keogram.set_ylabel("Distance along cut (km)")
    ax_keogram.set_title(
        f"{angle_deg:+d}° Keogram, {time_range_label} Local Time\n{dataset_label}"
    )

    # --------------------------------------------------
    # Profiles at 0, 125, 250 km
    # --------------------------------------------------
    profile_targets = [0.0, 125.0, 250.0]
    for target in profile_targets:
        index = int(np.argmin(np.abs(distance_km - target)))
        ax_profiles.plot(
            window_minutes,
            keogram[:, index],
            linewidth=1.2,
            label=f"{distance_km[index]:g} km",
        )

    ax_profiles.axhline(0.0, linewidth=0.8, linestyle="--")
    ax_profiles.set_xlim(0.0, window_duration_minutes)
    ax_profiles.set_xticks(window_tick_minutes)
    ax_profiles.set_xticklabels(window_tick_labels)
    ax_profiles.set_xlabel("Local time")
    ax_profiles.set_ylabel("Keogram value")
    ax_profiles.set_title("Profiles at 0, 125, and 250 km")
    ax_profiles.grid(True, alpha=0.3)
    ax_profiles.legend(loc="best")

    # --------------------------------------------------
    # Correlation map and ridge
    # --------------------------------------------------
    if len(shifts_minutes) > 1:
        shift_spacing = float(np.median(np.diff(shifts_minutes)))
    else:
        shift_spacing = 1.0

    if len(selected_distances) > 1:
        distance_spacing = float(np.median(np.diff(selected_distances)))
    else:
        distance_spacing = 1.0

    image = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=CORRELATION_CMAP,
        vmin=CORRELATION_VMIN,
        vmax=CORRELATION_VMAX,
        extent=[
            shifts_minutes[0] - shift_spacing / 2.0,
            shifts_minutes[-1] + shift_spacing / 2.0,
            selected_distances[0] - distance_spacing / 2.0,
            selected_distances[-1] + distance_spacing / 2.0,
        ],
    )
    cbar_map = fig.colorbar(image, ax=ax_map, fraction=0.025, pad=0.02)
    cbar_map.set_label("Pearson correlation with 0-km row")

    ax_map.axvline(0.0, linestyle="--", linewidth=1.0)

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (maximum_correlation >= MIN_CORRELATION_FOR_SLOPE)
    )

    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            linewidth=1.2,
            label="Maximum correlation",
        )
        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            s=9,
        )

    outlier_mask = slope_results.get(
        "outlier_mask", np.zeros_like(ridge_mask, dtype=bool)
    )
    if np.any(outlier_mask):
        ax_map.scatter(
            best_shift_minutes[outlier_mask],
            selected_distances[outlier_mask],
            marker="x",
            s=35,
            linewidth=1.1,
            label="Rejected ridge point",
        )

    slope = slope_results["slope_min_per_km"]
    intercept = slope_results["intercept_minutes"]
    speed = slope_results["speed_m_per_s"]
    r_squared = slope_results["r_squared"]

    if np.isfinite(slope):
        fit_distance = np.linspace(
            SLOPE_FIT_DISTANCE_MIN_KM,
            SLOPE_FIT_DISTANCE_MAX_KM,
            300,
        )
        fit_shift = slope * fit_distance + intercept
        ax_map.plot(
            fit_shift,
            fit_distance,
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        ax_map.text(
            0.02,
            0.03,
            (
                f"slope = {slope:.5f} min/km\n"
                f"speed = {speed:.1f} m/s\n"
                f"R² = {r_squared:.3f}\n"
                f"points = {slope_results['number_of_fit_points']}\n"
                f"rejected = {slope_results['number_of_outliers_removed']}"
            ),
            transform=ax_map.transAxes,
            va="bottom",
            ha="left",
            fontsize=9,
            bbox={"facecolor": "white", "edgecolor": "0.4", "alpha": 0.85},
        )

    ax_map.set_xlim(SHIFT_MIN_MINUTES, SHIFT_MAX_MINUTES)
    ax_map.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )
    ax_map.set_xlabel("Shift applied to row (min)")
    ax_map.set_ylabel("Distance along cut (km)")
    ax_map.set_title("Shifted-row correlation and maximum-correlation ridge")
    ax_map.legend(loc="upper right", fontsize=8)

    # --------------------------------------------------
    # 0-km autocorrelation
    # --------------------------------------------------
    ax_autocorr.plot(
        zero_profile_shifts_minutes,
        zero_km_profile,
        linewidth=1.3,
        label="0-km autocorrelation",
    )
    ax_autocorr.axvline(0.0, linestyle="--", linewidth=0.9)

    for side, marker in (("left", "o"), ("right", "s")):
        peak = zero_km_peaks[side]
        if np.isfinite(peak["shift_minutes"]):
            ax_autocorr.scatter(
                peak["shift_minutes"],
                peak["correlation"],
                marker=marker,
                s=50,
                zorder=4,
                label=(
                    f"{side.capitalize()} max: "
                    f"{peak['shift_minutes']:.1f} min"
                ),
            )

    ax_autocorr.set_xlim(
        ZERO_KM_PROFILE_SHIFT_MIN_MINUTES,
        ZERO_KM_PROFILE_SHIFT_MAX_MINUTES,
    )
    ax_autocorr.set_xlabel("Shift (min)")
    ax_autocorr.set_ylabel("Correlation")
    ax_autocorr.set_title("0-km autocorrelation profile")
    ax_autocorr.grid(True, alpha=0.3)
    ax_autocorr.legend(loc="best", fontsize=8)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_phase_speed_plot(combined_results, output_file, time_range_label):
    if combined_results.size == 0:
        return

    angle = combined_results[:, 0]
    processed_speed = combined_results[:, 4]
    corrected_speed = combined_results[:, 10]

    angle_mask = (
        np.isfinite(angle)
        & (angle >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    angle_plot = angle[angle_mask]
    processed_plot = processed_speed[angle_mask].copy()
    corrected_plot = corrected_speed[angle_mask].copy()

    order = np.argsort(angle_plot)
    angle_plot = angle_plot[order]
    processed_plot = processed_plot[order]
    corrected_plot = corrected_plot[order]

    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.plot(
        angle_plot,
        processed_plot,
        marker="o",
        linewidth=1.5,
        label="Processed",
    )

    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        ax.plot(
            angle_plot,
            corrected_plot,
            marker="s",
            linewidth=1.5,
            label="Processed minus 2-hr slope bias",
        )

    # Horizontal line through the minimum of each displayed curve.
    for values, label, linestyle in (
        (processed_plot, "Processed minimum", "--"),
        (corrected_plot, "Minus-bias minimum", ":"),
    ):
        if label.startswith("Minus") and not APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
            continue
        if np.any(np.isfinite(values)):
            index = int(np.nanargmin(values))
            minimum_speed = float(values[index])
            minimum_angle = float(angle_plot[index])
            ax.axhline(
                minimum_speed,
                linestyle=linestyle,
                linewidth=1.1,
                alpha=0.8,
                label=(
                    f"{label}: {minimum_speed:.1f} m/s "
                    f"at {minimum_angle:+.0f}°"
                ),
            )
            ax.scatter(minimum_angle, minimum_speed, marker="*", s=160, zorder=5)

    ax.set_xlim(
        PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
        PHASE_SPEED_PLOT_ANGLE_MAX_DEG,
    )
    displayed_values = [processed_plot[np.isfinite(processed_plot)]]
    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        displayed_values.append(corrected_plot[np.isfinite(corrected_plot)])

    displayed_values = [
        values for values in displayed_values if values.size > 0
    ]
    if displayed_values:
        all_displayed = np.concatenate(displayed_values)
        y_min = float(np.nanmin(all_displayed))
        y_max = float(np.nanmax(all_displayed))
        y_span = y_max - y_min
        padding = max(
            PHASE_SPEED_Y_MIN_PADDING_M_PER_S,
            PHASE_SPEED_Y_PADDING_FRACTION * max(y_span, abs(y_min), abs(y_max)),
        )
        if np.isclose(y_span, 0.0):
            y_min -= padding
            y_max += padding
        else:
            y_min -= padding
            y_max += padding
        ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(30, 151, 10))
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("Equivalent phase speed (m/s)")
    ax.set_title(
        f"Phase Speed versus Angle, {time_range_label} Local Time"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_r_squared_plot(combined_results, output_file, time_range_label):
    if combined_results.size == 0:
        return

    angle = combined_results[:, 0]
    processed_r2 = combined_results[:, 5]
    corrected_r2 = combined_results[:, 11]

    angle_mask = (
        np.isfinite(angle)
        & (angle >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    angle_plot = angle[angle_mask]
    processed_plot = processed_r2[angle_mask]
    corrected_plot = corrected_r2[angle_mask]

    order = np.argsort(angle_plot)
    angle_plot = angle_plot[order]
    processed_plot = processed_plot[order]
    corrected_plot = corrected_plot[order]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        angle_plot,
        processed_plot,
        marker="o",
        linewidth=1.5,
        label="Processed",
    )

    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        ax.plot(
            angle_plot,
            corrected_plot,
            marker="s",
            linewidth=1.5,
            label="Processed minus 2-hr slope bias",
        )

    ax.set_xlim(
        PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
        PHASE_SPEED_PLOT_ANGLE_MAX_DEG,
    )
    ax.set_xticks(np.arange(30, 151, 10))
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("R²")
    ax.set_title(
        f"Maximum-Correlation Ridge Fit R², {time_range_label} Local Time"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Main analysis
# ==================================================
def analyze_window(window_name, start_hour, end_hour):
    time_range_label = (
        f"{format_clock_hour(start_hour)}–{format_clock_hour(end_hour)}"
    )

    OUT_DIR = ANALYSIS_ROOT / f"analysis_{window_name}"
    PLOTS_DIR = OUT_DIR / "plots"
    DATA_DIR = OUT_DIR / "data"
    for directory in (OUT_DIR, PLOTS_DIR, DATA_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    window_duration_minutes, window_tick_minutes, window_tick_labels = (
        build_window_plot_ticks(start_hour, end_hour)
    )

    if not KEOGRAM_DIR.exists():
        raise FileNotFoundError(f"Keogram directory does not exist:\n{KEOGRAM_DIR}")

    if not KEOGRAM_METADATA_FILE.exists():
        raise FileNotFoundError(
            "Could not find keogram_metadata.npz:\n"
            f"{KEOGRAM_METADATA_FILE}\n"
            "Use the new keogram-generation script first."
        )

    # --------------------------------------------------
    # Read common time / local-time metadata
    # --------------------------------------------------
    with np.load(KEOGRAM_METADATA_FILE, allow_pickle=False) as metadata:
        times_min = np.asarray(metadata["time_minutes"], dtype=np.float64)
        local_datetimes = build_local_datetimes(metadata, times_min)

    if len(times_min) < 2:
        raise ValueError("At least two time samples are required.")

    time_differences = np.diff(times_min)
    finite_dt = time_differences[np.isfinite(time_differences)]
    if finite_dt.size == 0:
        raise ValueError("Could not determine time spacing.")

    minutes_per_pixel = float(np.median(finite_dt))
    if not np.isfinite(minutes_per_pixel) or minutes_per_pixel <= 0:
        raise ValueError("Invalid time spacing in saved metadata.")

    local_hours = local_clock_hours(local_datetimes)
    time_mask = make_local_time_mask(
        local_hours,
        start_hour,
        end_hour,
    )

    selected_indices = np.flatnonzero(time_mask)
    if len(selected_indices) < MIN_VALID_CORRELATION_POINTS:
        raise ValueError(
            f"Too few samples fall between {time_range_label} local time. "
            f"Found only {len(selected_indices)} samples."
        )

    times_selected_min = times_min[time_mask]
    local_datetimes_selected = local_datetimes[time_mask]
    local_hours_selected = local_hours[time_mask]
    window_minutes = minutes_from_window_start(local_hours_selected, start_hour)

    # Ensure chronological order in case the generic clock mask is ever used
    # with an across-midnight interval.
    order = np.argsort(times_selected_min)
    times_selected_min = times_selected_min[order]
    local_datetimes_selected = local_datetimes_selected[order]
    window_minutes = window_minutes[order]

    print("=" * 78)
    print("BLO KEOGRAM CORRELATION ANALYSIS")
    print("=" * 78)
    print(f"Keogram directory: {KEOGRAM_DIR}")
    print(f"Output directory:  {OUT_DIR}")
    print(f"Local-time interval: {time_range_label}")
    print(f"Selected samples: {len(times_selected_min)}")
    print(
        "Selected local datetimes: "
        f"{local_datetimes_selected[0]} to {local_datetimes_selected[-1]}"
    )
    print(f"One time pixel: {minutes_per_pixel:.6f} min")

    # --------------------------------------------------
    # Shift axes
    # --------------------------------------------------
    shift_min_pixels = int(np.ceil(SHIFT_MIN_MINUTES / minutes_per_pixel))
    shift_max_pixels = int(np.floor(SHIFT_MAX_MINUTES / minutes_per_pixel))
    shifts_pixels = np.arange(
        shift_min_pixels, shift_max_pixels + 1, dtype=int
    )
    shifts_minutes = shifts_pixels * minutes_per_pixel

    zero_min_pixels = int(
        np.ceil(ZERO_KM_PROFILE_SHIFT_MIN_MINUTES / minutes_per_pixel)
    )
    zero_max_pixels = int(
        np.floor(ZERO_KM_PROFILE_SHIFT_MAX_MINUTES / minutes_per_pixel)
    )
    zero_profile_shifts_pixels = np.arange(
        zero_min_pixels, zero_max_pixels + 1, dtype=int
    )
    zero_profile_shifts_minutes = (
        zero_profile_shifts_pixels * minutes_per_pixel
    )

    # --------------------------------------------------
    # Load saved 2-hour slope background once
    # --------------------------------------------------
    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        (
            slope_x_selected,
            slope_y_selected,
            running_average_window_hours,
        ) = load_two_hour_slope_background(
            PLANE_RESULTS_FILE,
            times_selected_min,
        )

        print(
            "2-hour bias correction: enabled "
            f"(saved window = {running_average_window_hours:g} hr)"
        )
    else:
        slope_x_selected = np.full(times_selected_min.shape, np.nan)
        slope_y_selected = np.full(times_selected_min.shape, np.nan)
        running_average_window_hours = np.nan
        print("2-hour bias correction: disabled")

    # Save common analysis coordinates.
    np.save(DATA_DIR / "angles_deg.npy", KEOGRAM_ANGLES_DEG)
    np.save(DATA_DIR / "shifts_minutes.npy", shifts_minutes)
    np.save(DATA_DIR / "shifts_pixels.npy", shifts_pixels)
    np.save(
        DATA_DIR / "zero_km_profile_shifts_minutes.npy",
        zero_profile_shifts_minutes,
    )
    np.save(
        DATA_DIR / "selected_elapsed_times_min.npy",
        times_selected_min,
    )
    np.save(
        DATA_DIR / "selected_window_minutes.npy",
        window_minutes,
    )

    combined_results = []
    combined_peak_results = []
    successful_angles = []
    failed_angles = []

    # --------------------------------------------------
    # Process all angles
    # --------------------------------------------------
    for angle_number, angle_deg in enumerate(KEOGRAM_ANGLES_DEG, start=1):
        print("\n" + "-" * 78)
        print(
            f"Angle {angle_deg:+d}° "
            f"({angle_number}/{len(KEOGRAM_ANGLES_DEG)})"
        )

        input_label = input_angle_label(angle_deg)
        output_label = output_angle_label(angle_deg)

        keogram_file = KEOGRAM_DIR / f"BLO_keogram_angle_{input_label}.npy"
        distance_file = (
            KEOGRAM_DIR
            / f"BLO_keogram_distance_km_angle_{input_label}.npy"
        )

        if not keogram_file.exists():
            failed_angles.append((int(angle_deg), "keogram file not found"))
            print(f"Missing: {keogram_file}")
            continue

        if not distance_file.exists():
            failed_angles.append((int(angle_deg), "distance file not found"))
            print(f"Missing: {distance_file}")
            continue

        try:
            keogram_original = np.asarray(np.load(keogram_file), dtype=np.float64)
            distance_km = np.asarray(np.load(distance_file), dtype=np.float64)

            if keogram_original.ndim != 2:
                raise ValueError(
                    f"Expected 2-D keogram, got {keogram_original.shape}."
                )

            if keogram_original.shape[0] != len(times_min):
                raise ValueError(
                    f"Keogram has {keogram_original.shape[0]} time samples, "
                    f"metadata has {len(times_min)}."
                )

            if keogram_original.shape[1] != len(distance_km):
                raise ValueError(
                    "Keogram distance dimension does not match saved "
                    "distance-axis file."
                )

            # Apply the local-time selection and chronological ordering.
            keogram_selected = keogram_original[time_mask, :][order, :]

            if APPLY_2D_SMOOTHING:
                keogram_processed = smooth_keogram_2d(
                    keogram_selected,
                    sigma_time=SMOOTH_SIGMA_TIME_PIXELS,
                    sigma_distance=SMOOTH_SIGMA_DISTANCE_PIXELS,
                    mode=SMOOTH_MODE,
                    weight_tolerance=SMOOTH_WEIGHT_TOLERANCE,
                )
                processed_label = (
                    "Processed keogram "
                    f"(Gaussian σt={SMOOTH_SIGMA_TIME_PIXELS:g}, "
                    f"σd={SMOOTH_SIGMA_DISTANCE_PIXELS:g})"
                )
            else:
                keogram_processed = keogram_selected.copy()
                processed_label = "Processed keogram (no additional smoothing)"

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                bias_keogram, slope_along_cut = create_directional_two_hour_bias(
                    slope_x_selected,
                    slope_y_selected,
                    distance_km,
                    angle_deg,
                )

                keogram_minus_bias = keogram_processed - bias_keogram
                keogram_minus_bias[~np.isfinite(keogram_processed)] = np.nan
                corrected_label = (
                    f"Processed minus {running_average_window_hours:g}-hr "
                    "running-average plane slope"
                )
            else:
                bias_keogram = np.zeros_like(keogram_processed)
                slope_along_cut = np.full(times_selected_min.shape, np.nan)
                keogram_minus_bias = keogram_processed.copy()
                corrected_label = "Bias subtraction disabled"

            selected_distance_mask = (
                (distance_km >= CORRELATION_DISTANCE_MIN_KM)
                & (distance_km <= CORRELATION_DISTANCE_MAX_KM)
            )

            if not np.any(selected_distance_mask):
                raise ValueError("No distances lie in the requested range.")

            # ------------------------------------------
            # Processed correlation map
            # ------------------------------------------
            (
                correlation_processed,
                selected_distances,
                _,
            ) = calculate_correlation_map(
                keogram_processed,
                distance_km,
                reference_distance_km=0.0,
                selected_distance_mask=selected_distance_mask,
                shifts_pixels=shifts_pixels,
                minimum_points=MIN_VALID_CORRELATION_POINTS,
            )

            best_shift_processed, max_corr_processed = find_best_shifts(
                correlation_processed,
                shifts_minutes,
            )

            slope_processed = fit_maximum_correlation_slope(
                selected_distances,
                best_shift_processed,
                max_corr_processed,
                minimum_correlation=MIN_CORRELATION_FOR_SLOPE,
                distance_min_km=SLOPE_FIT_DISTANCE_MIN_KM,
                distance_max_km=SLOPE_FIT_DISTANCE_MAX_KM,
                remove_outliers=REMOVE_RIDGE_OUTLIERS,
                maximum_ridge_jump_minutes=RIDGE_MAXIMUM_JUMP_MINUTES,
                outlier_sigma=RIDGE_OUTLIER_SIGMA,
                outlier_max_iterations=RIDGE_OUTLIER_MAX_ITERATIONS,
                outlier_min_points=RIDGE_OUTLIER_MIN_POINTS,
            )

            # ------------------------------------------
            # Minus-2hr-bias correlation map
            # ------------------------------------------
            (
                correlation_corrected,
                selected_distances_corrected,
                _,
            ) = calculate_correlation_map(
                keogram_minus_bias,
                distance_km,
                reference_distance_km=0.0,
                selected_distance_mask=selected_distance_mask,
                shifts_pixels=shifts_pixels,
                minimum_points=MIN_VALID_CORRELATION_POINTS,
            )

            best_shift_corrected, max_corr_corrected = find_best_shifts(
                correlation_corrected,
                shifts_minutes,
            )

            slope_corrected = fit_maximum_correlation_slope(
                selected_distances_corrected,
                best_shift_corrected,
                max_corr_corrected,
                minimum_correlation=MIN_CORRELATION_FOR_SLOPE,
                distance_min_km=SLOPE_FIT_DISTANCE_MIN_KM,
                distance_max_km=SLOPE_FIT_DISTANCE_MAX_KM,
                remove_outliers=REMOVE_RIDGE_OUTLIERS,
                maximum_ridge_jump_minutes=RIDGE_MAXIMUM_JUMP_MINUTES,
                outlier_sigma=RIDGE_OUTLIER_SIGMA,
                outlier_max_iterations=RIDGE_OUTLIER_MAX_ITERATIONS,
                outlier_min_points=RIDGE_OUTLIER_MIN_POINTS,
            )

            # ------------------------------------------
            # 0-km autocorrelation profiles
            # ------------------------------------------
            zero_index = int(np.argmin(np.abs(distance_km)))

            zero_processed = keogram_processed[:, zero_index]
            zero_corrected = keogram_minus_bias[:, zero_index]

            zero_profile_processed = np.array(
                [
                    calculate_shifted_correlation(
                        zero_processed,
                        zero_processed,
                        int(shift),
                        minimum_points=MIN_VALID_CORRELATION_POINTS,
                    )
                    for shift in zero_profile_shifts_pixels
                ],
                dtype=np.float64,
            )

            zero_profile_corrected = np.array(
                [
                    calculate_shifted_correlation(
                        zero_corrected,
                        zero_corrected,
                        int(shift),
                        minimum_points=MIN_VALID_CORRELATION_POINTS,
                    )
                    for shift in zero_profile_shifts_pixels
                ],
                dtype=np.float64,
            )

            peaks_processed = find_left_and_right_local_maxima(
                zero_profile_processed,
                zero_profile_shifts_minutes,
                minimum_prominence=LOCAL_MAXIMUM_MIN_PROMINENCE,
                minimum_distance_pixels=LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS,
            )

            peaks_corrected = find_left_and_right_local_maxima(
                zero_profile_corrected,
                zero_profile_shifts_minutes,
                minimum_prominence=LOCAL_MAXIMUM_MIN_PROMINENCE,
                minimum_distance_pixels=LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS,
            )

            # ------------------------------------------
            # Plots -- all flat in plots/
            # ------------------------------------------
            processed_plot_file = (
                PLOTS_DIR
                / f"angle_{output_label}_processed.png"
            )
            corrected_plot_file = (
                PLOTS_DIR
                / f"angle_{output_label}_minus_2hr_bias.png"
            )

            plot_one_dataset_analysis(
                angle_deg,
                processed_label,
                keogram_processed,
                window_minutes,
                window_duration_minutes,
                window_tick_minutes,
                window_tick_labels,
                time_range_label,
                distance_km,
                selected_distance_mask,
                correlation_processed,
                shifts_minutes,
                selected_distances,
                best_shift_processed,
                max_corr_processed,
                slope_processed,
                zero_profile_processed,
                zero_profile_shifts_minutes,
                peaks_processed,
                processed_plot_file,
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                plot_one_dataset_analysis(
                    angle_deg,
                    corrected_label,
                    keogram_minus_bias,
                    window_minutes,
                    window_duration_minutes,
                    window_tick_minutes,
                    window_tick_labels,
                    time_range_label,
                    distance_km,
                    selected_distance_mask,
                    correlation_corrected,
                    shifts_minutes,
                    selected_distances_corrected,
                    best_shift_corrected,
                    max_corr_corrected,
                    slope_corrected,
                    zero_profile_corrected,
                    zero_profile_shifts_minutes,
                    peaks_corrected,
                    corrected_plot_file,
                )

            # ------------------------------------------
            # Data products -- all flat in data/
            # ------------------------------------------
            np.save(
                DATA_DIR / f"correlation_{output_label}_processed.npy",
                correlation_processed,
            )
            np.save(
                DATA_DIR / f"best_shift_{output_label}_processed.npy",
                best_shift_processed,
            )
            np.save(
                DATA_DIR / f"max_correlation_{output_label}_processed.npy",
                max_corr_processed,
            )
            np.save(
                DATA_DIR / f"distances_{output_label}.npy",
                selected_distances,
            )
            np.save(
                DATA_DIR / f"zero_km_autocorrelation_{output_label}_processed.npy",
                zero_profile_processed,
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                np.save(
                    DATA_DIR / f"correlation_{output_label}_minus_2hr_bias.npy",
                    correlation_corrected,
                )
                np.save(
                    DATA_DIR / f"best_shift_{output_label}_minus_2hr_bias.npy",
                    best_shift_corrected,
                )
                np.save(
                    DATA_DIR / f"max_correlation_{output_label}_minus_2hr_bias.npy",
                    max_corr_corrected,
                )
                np.save(
                    DATA_DIR / f"zero_km_autocorrelation_{output_label}_minus_2hr_bias.npy",
                    zero_profile_corrected,
                )
                np.save(
                    DATA_DIR / f"two_hour_bias_keogram_{output_label}.npy",
                    bias_keogram,
                )
                np.save(
                    DATA_DIR / f"two_hour_slope_along_cut_{output_label}.npy",
                    slope_along_cut,
                )

            ridge_processed_csv = np.column_stack(
                [selected_distances, best_shift_processed, max_corr_processed]
            )
            ridge_corrected_csv = np.column_stack(
                [
                    selected_distances_corrected,
                    best_shift_corrected,
                    max_corr_corrected,
                ]
            )

            np.savetxt(
                DATA_DIR / f"ridge_{output_label}_processed.csv",
                ridge_processed_csv,
                delimiter=",",
                header="distance_km,best_shift_minutes,maximum_correlation",
                comments="",
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                np.savetxt(
                    DATA_DIR / f"ridge_{output_label}_minus_2hr_bias.csv",
                    ridge_corrected_csv,
                    delimiter=",",
                    header="distance_km,best_shift_minutes,maximum_correlation",
                    comments="",
                )

            # ------------------------------------------
            # Combined summaries
            # ------------------------------------------
            combined_results.append(
                [
                    float(angle_deg),
                    slope_processed["slope_min_per_km"],
                    slope_processed["intercept_minutes"],
                    slope_processed["speed_km_per_min"],
                    slope_processed["speed_m_per_s"],
                    slope_processed["r_squared"],
                    slope_processed["number_of_fit_points"],
                    slope_corrected["slope_min_per_km"],
                    slope_corrected["intercept_minutes"],
                    slope_corrected["speed_km_per_min"],
                    slope_corrected["speed_m_per_s"],
                    slope_corrected["r_squared"],
                    slope_corrected["number_of_fit_points"],
                ]
            )

            combined_peak_results.append(
                [
                    float(angle_deg),
                    peaks_processed["left"]["shift_minutes"],
                    peaks_processed["left"]["correlation"],
                    peaks_processed["right"]["shift_minutes"],
                    peaks_processed["right"]["correlation"],
                    peaks_corrected["left"]["shift_minutes"],
                    peaks_corrected["left"]["correlation"],
                    peaks_corrected["right"]["shift_minutes"],
                    peaks_corrected["right"]["correlation"],
                ]
            )

            successful_angles.append(int(angle_deg))

            print(
                "  Processed: "
                f"speed={slope_processed['speed_m_per_s']:.2f} m/s, "
                f"R²={slope_processed['r_squared']:.3f}"
            )
            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                print(
                    "  Minus 2-hr bias: "
                    f"speed={slope_corrected['speed_m_per_s']:.2f} m/s, "
                    f"R²={slope_corrected['r_squared']:.3f}"
                )

        except Exception as exc:
            failed_angles.append((int(angle_deg), str(exc)))
            print(f"ERROR: {exc}")

    # --------------------------------------------------
    # Save combined summaries
    # --------------------------------------------------
    combined_results = np.asarray(combined_results, dtype=np.float64)
    combined_peak_results = np.asarray(combined_peak_results, dtype=np.float64)

    slope_header = (
        "angle_deg,"
        "processed_slope_min_per_km,"
        "processed_intercept_minutes,"
        "processed_speed_km_per_min,"
        "processed_speed_m_per_s,"
        "processed_r_squared,"
        "processed_number_of_fit_points,"
        "minus_2hr_bias_slope_min_per_km,"
        "minus_2hr_bias_intercept_minutes,"
        "minus_2hr_bias_speed_km_per_min,"
        "minus_2hr_bias_speed_m_per_s,"
        "minus_2hr_bias_r_squared,"
        "minus_2hr_bias_number_of_fit_points"
    )

    if combined_results.size > 0:
        np.savetxt(
            OUT_DIR / "all_angle_slope_summary.csv",
            combined_results,
            delimiter=",",
            header=slope_header,
            comments="",
        )
        np.save(OUT_DIR / "all_angle_slope_summary.npy", combined_results)

        save_phase_speed_plot(
            combined_results,
            OUT_DIR / "phase_speed_vs_angle.png",
            time_range_label,
        )
        save_r_squared_plot(
            combined_results,
            OUT_DIR / "r_squared_vs_angle.png",
            time_range_label,
        )

    peak_header = (
        "angle_deg,"
        "processed_left_shift_minutes,processed_left_correlation,"
        "processed_right_shift_minutes,processed_right_correlation,"
        "minus_2hr_bias_left_shift_minutes,minus_2hr_bias_left_correlation,"
        "minus_2hr_bias_right_shift_minutes,minus_2hr_bias_right_correlation"
    )

    if combined_peak_results.size > 0:
        np.savetxt(
            OUT_DIR / "all_angle_zero_km_local_maxima.csv",
            combined_peak_results,
            delimiter=",",
            header=peak_header,
            comments="",
        )

    # --------------------------------------------------
    # Processing log
    # --------------------------------------------------
    with open(OUT_DIR / "processing_log.txt", "w", encoding="utf-8") as file:
        file.write(f"Keogram directory: {KEOGRAM_DIR}\n")
        file.write(f"Plane results: {PLANE_RESULTS_FILE}\n")
        file.write(f"Local-time interval: {time_range_label}\n")
        file.write(
            "Selected local datetimes: "
            f"{local_datetimes_selected[0]} to {local_datetimes_selected[-1]}\n"
        )
        file.write(f"Selected samples: {len(times_selected_min)}\n")
        file.write(f"Minutes per pixel: {minutes_per_pixel:.10f}\n")
        file.write(
            "2-hour slope bias subtraction: "
            f"{APPLY_2HR_SLOPE_BIAS_SUBTRACTION}\n"
        )
        if np.isfinite(running_average_window_hours):
            file.write(
                f"Saved running-average window: "
                f"{running_average_window_hours:g} hr\n"
            )
        file.write("\nSuccessful angles:\n")
        file.write(", ".join(str(v) for v in successful_angles) + "\n")
        file.write("\nFailed angles:\n")
        if failed_angles:
            for angle, reason in failed_angles:
                file.write(f"{angle}: {reason}\n")
        else:
            file.write("None\n")

    print("\n" + "=" * 78)
    print(f"ANALYSIS COMPLETE: {time_range_label}")
    print("=" * 78)
    print(f"Successful angles: {len(successful_angles)}")
    print(f"Failed angles:     {len(failed_angles)}")
    print(f"Output folder:     {OUT_DIR.resolve()}")
    print("\nDirectory structure:")
    print(f"  {OUT_DIR.name}/")
    print("    plots/")
    print("    data/")
    print("    all_angle_slope_summary.csv")
    print("    phase_speed_vs_angle.png")
    print("    r_squared_vs_angle.png")
    print("    processing_log.txt")

    return {
        "window_name": window_name,
        "start_hour": float(start_hour),
        "end_hour": float(end_hour),
        "duration_hours": float(window_duration_minutes / 60.0),
        "time_range_label": time_range_label,
        "combined_results": combined_results,
        "successful_angles": len(successful_angles),
        "failed_angles": len(failed_angles),
    }


# ==================================================
# Cross-window summary table
# ==================================================
def get_minimum_speed_summary(combined_results, speed_column, r2_column):
    """Return angle, minimum signed phase speed, and R² at that angle."""
    combined_results = np.asarray(combined_results, dtype=np.float64)

    if combined_results.ndim != 2 or combined_results.shape[0] == 0:
        return np.nan, np.nan, np.nan

    angle = combined_results[:, 0]
    speed = combined_results[:, speed_column]
    r_squared = combined_results[:, r2_column]

    valid = (
        np.isfinite(angle)
        & np.isfinite(speed)
        & (angle >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    if not np.any(valid):
        return np.nan, np.nan, np.nan

    valid_indices = np.flatnonzero(valid)
    index = int(valid_indices[np.argmin(speed[valid])])

    return (
        float(angle[index]),
        float(speed[index]),
        float(r_squared[index]) if np.isfinite(r_squared[index]) else np.nan,
    )


def build_window_summary_rows(window_results):
    """Create one compact comparison row for each successfully run window."""
    rows = []

    for result in window_results:
        combined_results = result["combined_results"]

        processed_angle, processed_speed, processed_r2 = (
            get_minimum_speed_summary(
                combined_results,
                speed_column=4,
                r2_column=5,
            )
        )

        corrected_angle, corrected_speed, corrected_r2 = (
            get_minimum_speed_summary(
                combined_results,
                speed_column=10,
                r2_column=11,
            )
        )

        rows.append(
            {
                "duration_hours": result["duration_hours"],
                "window": result["time_range_label"],
                "processed_min_angle_deg": processed_angle,
                "processed_min_speed_m_per_s": processed_speed,
                "processed_r_squared": processed_r2,
                "minus_2hr_bias_min_angle_deg": corrected_angle,
                "minus_2hr_bias_min_speed_m_per_s": corrected_speed,
                "minus_2hr_bias_r_squared": corrected_r2,
                "successful_angles": result["successful_angles"],
                "failed_angles": result["failed_angles"],
            }
        )

    return rows


def save_window_summary_csv(rows, output_file):
    fieldnames = [
        "duration_hours",
        "window",
        "processed_min_angle_deg",
        "processed_min_speed_m_per_s",
        "processed_r_squared",
        "minus_2hr_bias_min_angle_deg",
        "minus_2hr_bias_min_speed_m_per_s",
        "minus_2hr_bias_r_squared",
        "successful_angles",
        "failed_angles",
    ]

    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def format_table_number(value, format_string):
    if value is None:
        return ""
    try:
        value = float(value)
    except (TypeError, ValueError):
        return str(value)

    if not np.isfinite(value):
        return ""

    return format(value, format_string)


def format_table_angle(value):
    text = format_table_number(value, ".0f")
    return f"{text}°" if text else ""


def save_window_summary_table_plot(rows, output_file):
    """
    Save a more readable summary as two stacked tables:
        1. Processed
        2. Processed minus 2-hour slope bias

    Splitting the two datasets avoids an overly wide 8-column table.
    """
    if not rows:
        return

    # Sort by duration, then by the order in ANALYSIS_WINDOWS.
    window_order = {
        (
            f"{format_clock_hour(start_hour)}–"
            f"{format_clock_hour(end_hour)}"
        ): index
        for index, (_, start_hour, end_hour) in enumerate(ANALYSIS_WINDOWS)
    }
    rows = sorted(
        rows,
        key=lambda row: (
            float(row["duration_hours"]),
            window_order.get(row["window"], 999),
        ),
    )

    column_labels = [
        "Duration",
        "Local-time window",
        "Min angle (°)",
        "Min speed (m/s)",
        "R²",
    ]

    def make_table_data(prefix):
        data = []
        for row in rows:
            if prefix == "processed":
                angle_key = "processed_min_angle_deg"
                speed_key = "processed_min_speed_m_per_s"
                r2_key = "processed_r_squared"
            else:
                angle_key = "minus_2hr_bias_min_angle_deg"
                speed_key = "minus_2hr_bias_min_speed_m_per_s"
                r2_key = "minus_2hr_bias_r_squared"

            data.append(
                [
                    f"{format_table_number(row['duration_hours'], '.0f')} hr",
                    row["window"],
                    format_table_number(row[angle_key], ".0f"),
                    format_table_number(row[speed_key], ".1f"),
                    format_table_number(row[r2_key], ".3f"),
                ]
            )
        return data

    fig_height = max(8.5, 0.66 * len(rows) * 2 + 3.2)
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11.5, fig_height),
        gridspec_kw={"hspace": 0.24},
    )

    table_specs = [
        (
            axes[0],
            make_table_data("processed"),
            "Processed",
        ),
        (
            axes[1],
            make_table_data("corrected"),
            "Processed minus 2-hr slope bias",
        ),
    ]

    for ax, table_data, subtitle in table_specs:
        ax.axis("off")

        table = ax.table(
            cellText=table_data,
            colLabels=column_labels,
            cellLoc="center",
            colLoc="center",
            loc="center",
            colWidths=[0.12, 0.27, 0.16, 0.20, 0.12],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.0, 1.55)

        # Make the header bold and visually separate the 3-, 4-, and 5-hour groups.
        for column in range(len(column_labels)):
            cell = table[(0, column)]
            cell.set_text_props(weight="bold")
            cell.set_facecolor("0.88")

        previous_duration = None
        for row_index, row in enumerate(rows, start=1):
            duration = int(round(float(row["duration_hours"])))

            # Alternate very light shading by duration group.
            shade = "0.98" if duration % 2 else "0.94"
            for column in range(len(column_labels)):
                table[(row_index, column)].set_facecolor(shade)

            # Thicker line at the start of each new duration group.
            if previous_duration is not None and duration != previous_duration:
                for column in range(len(column_labels)):
                    table[(row_index, column)].set_linewidth(1.5)

            previous_duration = duration

        ax.set_title(subtitle, fontsize=12, weight="bold", pad=8)

    fig.suptitle(
        "BLO Phase-Speed Minimum Summary\n"
        "3-, 4-, and 5-hour windows from 23:00–05:00 local time, "
        "using 30-minute start increments\n"
        "R² = coefficient of determination of the linear "
        "shift-versus-distance ridge fit",
        fontsize=13,
        y=0.995,
    )

    fig.subplots_adjust(top=0.90, bottom=0.03, left=0.03, right=0.97)
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def main():
    if not KEOGRAM_DIR.exists():
        raise FileNotFoundError(f"Keogram directory does not exist:\n{KEOGRAM_DIR}")

    print("=" * 78)
    print("RUNNING BLO KEOGRAM ANALYSES FROM 23:00 TO 05:00")
    print("=" * 78)
    print("Durations: 3, 4, and 5 hours; starts every 30 minutes")
    print(f"Number of windows: {len(ANALYSIS_WINDOWS)}")
    print("Angle range: 30°–150° in 5° steps")
    print(f"Analysis root: {ANALYSIS_ROOT}")

    print("\nRequested windows:")
    for window_name, start_hour, end_hour in ANALYSIS_WINDOWS:
        duration_hours = (float(end_hour) - float(start_hour)) % 24.0
        print(
            f"  {duration_hours:.0f} hr: "
            f"{format_clock_hour(start_hour)}–{format_clock_hour(end_hour)}"
        )

    failed_windows = []
    window_results = []

    for window_name, start_hour, end_hour in ANALYSIS_WINDOWS:
        duration_hours = (float(end_hour) - float(start_hour)) % 24.0

        print("\n" + "#" * 78)
        print(
            f"{duration_hours:.0f}-HOUR WINDOW "
            f"{format_clock_hour(start_hour)}–"
            f"{format_clock_hour(end_hour)}"
        )
        print("#" * 78)

        try:
            result = analyze_window(window_name, start_hour, end_hour)
            window_results.append(result)
        except Exception as exc:
            failed_windows.append((window_name, str(exc)))
            print(f"WINDOW FAILED: {exc}")

    # --------------------------------------------------
    # Root-level comparison table
    # --------------------------------------------------
    summary_rows = build_window_summary_rows(window_results)

    if summary_rows:
        summary_csv = ANALYSIS_ROOT / "window_summary_table.csv"
        summary_png = ANALYSIS_ROOT / "window_summary_table.png"

        save_window_summary_csv(summary_rows, summary_csv)
        save_window_summary_table_plot(summary_rows, summary_png)

        print("\nCross-window summary table:")
        print(f"  {summary_csv}")
        print(f"  {summary_png}")

    print("\n" + "=" * 78)
    print("ALL REQUESTED WINDOWS FINISHED")
    print("=" * 78)
    print(f"Analysis root: {ANALYSIS_ROOT.resolve()}")
    print(f"Successful windows: {len(window_results)}")
    print(f"Failed windows:     {len(failed_windows)}")

    if failed_windows:
        print("\nFailed windows:")
        for window_name, reason in failed_windows:
            print(f"  {window_name}: {reason}")
    else:
        print(f"All {len(ANALYSIS_WINDOWS)} windows completed successfully.")


if __name__ == "__main__":
    main()


RUNNING BLO KEOGRAM ANALYSES FROM 23:00 TO 05:00
Durations: 3, 4, and 5 hours; starts every 30 minutes
Number of windows: 15
Angle range: 30°–150° in 5° steps
Analysis root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/keogram_outputs/analysis_3_4_5hour_windows

Requested windows:
  3 hr: 23:00–02:00
  3 hr: 23:30–02:30
  3 hr: 00:00–03:00
  3 hr: 00:30–03:30
  3 hr: 01:00–04:00
  3 hr: 01:30–04:30
  3 hr: 02:00–05:00
  4 hr: 23:00–03:00
  4 hr: 23:30–03:30
  4 hr: 00:00–04:00
  4 hr: 00:30–04:30
  4 hr: 01:00–05:00
  5 hr: 23:00–04:00
  5 hr: 23:30–04:30
  5 hr: 00:00–05:00

##############################################################################
3-HOUR WINDOW 23:00–02:00
##############################################################################
BLO KEOGRAM CORRELATION ANALYSIS
Keogram directory: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/keogram_outputs
Output directory:  /User

## Weighed average? 

In [20]:
# -*- coding: utf-8 -*-
"""
Analyze BLO directional keograms created from the saved preprocessed cube.

This is a simplified follow-up to the keogram-generation script.

Main changes relative to the older correlation-analysis script:
    1. Reads the new keogram_outputs folder directly.
    2. Uses the full 23:00–05:00 SAVED LOCAL CLOCK TIME interval and runs
       every 3-, 4-, and 5-hour window that fits inside it, with window
       starts every 30 minutes. This gives 15 total windows:
           7 three-hour windows
           5 four-hour windows
           3 five-hour windows

    3. Analyzes only keogram angles from 30° through 150° in 5° steps.
    4. Uses the explicitly saved distance axis for every keogram angle.
    5. Optionally removes the planar bias estimated from the saved centered
       2-hour running-average slope vector.
    6. Replaces the single minimum-speed angle with a weighted ANGLE estimate.
       Lower phase-speed magnitude receives higher weight, and R² can also
       downweight poorly fitted correlation ridges. Phase speed itself is NOT
       averaged; it is interpolated from the phase-speed curve at the weighted
       angle.
    7. Creates one folder per time window plus a root-level comparison table.
       The root summary is one merged, large-font table containing both the
       processed and minus-2-hour-bias results.

           keogram_outputs/
               analysis_3_4_5hour_windows/
                   analysis_3hr_23_00_to_02_00/
                       plots/
                       data/
                       all_angle_slope_summary.csv
                       phase_speed_vs_angle.png
                       r_squared_vs_angle.png
                       processing_log.txt
                   ...
                   window_summary_table.csv
                   window_summary_table.png

No per-angle directory hierarchy is created. Angle and dataset names are
encoded directly in each filename.

Correlation method:
    - For each directional keogram, use the 0-km row as the reference.
    - Correlate rows from 0 to 250 km against the reference for time shifts.
    - Find the maximum-correlation ridge.
    - Fit shift_minutes = slope * distance_km + intercept.
    - Equivalent signed phase speed = -(1 / slope) * 1000 / 60 m/s.

2-hour bias correction:
    The preprocessing/plane-fit script saves a centered 2-hour running mean of
    the preprocessed plane-slope components in amplitude/km. For a keogram cut
    at angle theta and signed distance d:

        slope_along_cut(t) = sx(t)*cos(theta) + sy(t)*sin(theta)
        bias(t, d) = slope_along_cut(t) * d

    Subtracting this term removes the slowly varying planar component while
    retaining the faster perturbation structure.
"""

import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks


# ==================================================
# Input / output settings
# ==================================================
# Change only this directory for a different night.
KEOGRAM_DIR = (
    Path.home()
    / "Downloads"
    / "pkl"
    / "BLO_plane_fit_batch_outputs"
    / "2025"
    / "processed_data_full_03_25_2025"
    / "keogram_outputs"
)

# The plane-fit results sit one level above keogram_outputs.
PLANE_RESULTS_FILE = KEOGRAM_DIR.parent / "plane_fit_results.npz"
KEOGRAM_METADATA_FILE = KEOGRAM_DIR / "keogram_metadata.npz"

# Output structure for all 3-, 4-, and 5-hour windows inside 23:00–05:00.
ANALYSIS_ROOT = KEOGRAM_DIR / "analysis_3_4_5hour_windows"
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)


# ==================================================
# Analysis settings
# ==================================================
# Analyze only the requested angle range.
KEOGRAM_ANGLES_DEG = np.arange(30, 151, 5, dtype=int)

# Every 3-, 4-, and 5-hour local-clock window fully contained in 23:00–05:00.
# Start times are spaced by 30 minutes.
# label, start_hour, end_hour
ANALYSIS_WINDOWS = [
    # 3-hour windows
    ("3hr_23_00_to_02_00", 23.0, 2.0),
    ("3hr_23_30_to_02_30", 23.5, 2.5),
    ("3hr_00_00_to_03_00", 0.0, 3.0),
    ("3hr_00_30_to_03_30", 0.5, 3.5),
    ("3hr_01_00_to_04_00", 1.0, 4.0),
    ("3hr_01_30_to_04_30", 1.5, 4.5),
    ("3hr_02_00_to_05_00", 2.0, 5.0),

    # 4-hour windows
    ("4hr_23_00_to_03_00", 23.0, 3.0),
    ("4hr_23_30_to_03_30", 23.5, 3.5),
    ("4hr_00_00_to_04_00", 0.0, 4.0),
    ("4hr_00_30_to_04_30", 0.5, 4.5),
    ("4hr_01_00_to_05_00", 1.0, 5.0),

    # 5-hour windows
    ("5hr_23_00_to_04_00", 23.0, 4.0),
    ("5hr_23_30_to_04_30", 23.5, 4.5),
    ("5hr_00_00_to_05_00", 0.0, 5.0),
]

# Distance range included in correlation map / ridge fit.
CORRELATION_DISTANCE_MIN_KM = 0.0
CORRELATION_DISTANCE_MAX_KM = 250.0
SLOPE_FIT_DISTANCE_MIN_KM = 0.0
SLOPE_FIT_DISTANCE_MAX_KM = 250.0

# Shift range for correlation map and ridge fit.
SHIFT_MIN_MINUTES = -50.0
SHIFT_MAX_MINUTES = 50.0

# Wider shift range used only for the 0-km autocorrelation profile.
ZERO_KM_PROFILE_SHIFT_MIN_MINUTES = -200.0
ZERO_KM_PROFILE_SHIFT_MAX_MINUTES = 200.0

MIN_VALID_CORRELATION_POINTS = 20
MIN_CORRELATION_FOR_SLOPE = 0.50

# Ridge cleanup.
REMOVE_RIDGE_OUTLIERS = True
RIDGE_MAXIMUM_JUMP_MINUTES = 20.0
RIDGE_OUTLIER_SIGMA = 3.0
RIDGE_OUTLIER_MAX_ITERATIONS = 10
RIDGE_OUTLIER_MIN_POINTS = 3

# Optional NaN-aware 2-D smoothing of the selected keogram.
APPLY_2D_SMOOTHING = True
SMOOTH_SIGMA_TIME_PIXELS = 1.5
SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0
SMOOTH_MODE = "nearest"
SMOOTH_WEIGHT_TOLERANCE = 1.0e-12

# Compare the processed keogram with the same keogram after removal of the
# centered 2-hour plane-slope background.
APPLY_2HR_SLOPE_BIAS_SUBTRACTION = True

# Plot settings.
DPI = 200
KEOGRAM_CMAP = "RdBu_r"
CORRELATION_CMAP = "RdBu_r"
KEOGRAM_COLOR_PERCENTILE = 99.0
CORRELATION_VMIN = 0.70
CORRELATION_VMAX = 1.00

# Phase-speed-vs-angle display range.
PHASE_SPEED_PLOT_ANGLE_MIN_DEG = 30.0
PHASE_SPEED_PLOT_ANGLE_MAX_DEG = 150.0

# Weighted-angle summary.
#
# ONLY THE ANGLE IS AVERAGED. Phase speed is not averaged.
#
# Lower phase-speed magnitude receives higher weight:
#
#     speed_weight = (minimum_speed_magnitude / speed_magnitude) ** power
#
# By default, R² also contributes linearly to the angle weight so that a
# low-speed point with a poor ridge fit does not dominate the angle estimate:
#
#     final_weight = speed_weight * R² ** WEIGHTED_R2_POWER
#
# Only points with speed magnitude <= factor * minimum speed magnitude are
# included. After the weighted angle is found, phase speed and R² are
# interpolated from their angle curves at that weighted angle.
WEIGHTED_SPEED_POWER = 2.0
WEIGHTED_SPEED_MAX_FACTOR = 1.5
WEIGHT_BY_R_SQUARED = True
WEIGHTED_R2_POWER = 1.0

# Summary-table highlighting. A merged row is shaded light green when either
# the processed or minus-2hr-bias R² at the weighted angle exceeds this value.
SUMMARY_R2_HIGHLIGHT_THRESHOLD = 0.85
SUMMARY_R2_HIGHLIGHT_COLOR = "#d9f2d9"

# Y range is determined dynamically from the displayed phase-speed values.
PHASE_SPEED_Y_PADDING_FRACTION = 0.10
PHASE_SPEED_Y_MIN_PADDING_M_PER_S = 50.0

# Local maxima on the 0-km autocorrelation profile.
LOCAL_MAXIMUM_MIN_PROMINENCE = 0.0
LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS = 1


# ==================================================
# Filename helpers
# ==================================================
def input_angle_label(angle_deg):
    """Match filenames written by the keogram-generation script."""
    return f"{int(angle_deg):03d}"


def output_angle_label(angle_deg):
    """Unambiguous filename-safe output label."""
    angle_deg = int(angle_deg)
    if angle_deg < 0:
        return f"m{abs(angle_deg):03d}"
    return f"p{angle_deg:03d}"


# ==================================================
# Time helpers
# ==================================================
def metadata_scalar(metadata, key, default=None):
    if key not in metadata.files:
        return default

    value = metadata[key]
    if np.ndim(value) == 0:
        try:
            return value.item()
        except ValueError:
            return value

    if np.size(value) == 1:
        value = np.ravel(value)[0]
        try:
            return value.item()
        except (AttributeError, ValueError):
            return value

    return value


def parse_iso_datetime(text):
    text = str(text).strip()
    try:
        return datetime.fromisoformat(text)
    except ValueError as exc:
        raise ValueError(f"Could not parse saved local datetime: {text!r}") from exc


def build_local_datetimes(metadata, times_min):
    """Read exact saved frame-center datetimes, with a start-time fallback."""
    if "local_datetime" in metadata.files:
        local_text = np.asarray(metadata["local_datetime"], dtype=str)
        if len(local_text) == len(times_min):
            return np.array(
                [parse_iso_datetime(value) for value in local_text],
                dtype=object,
            )

    start_text = str(metadata_scalar(metadata, "start_local_datetime", ""))
    if not start_text:
        raise ValueError(
            "keogram_metadata.npz has neither a usable local_datetime array "
            "nor start_local_datetime."
        )

    start_datetime = parse_iso_datetime(start_text)
    return np.array(
        [start_datetime + timedelta(minutes=float(value)) for value in times_min],
        dtype=object,
    )


def local_clock_hours(local_datetimes):
    return np.array(
        [
            dt.hour
            + dt.minute / 60.0
            + dt.second / 3600.0
            + dt.microsecond / 3.6e9
            for dt in local_datetimes
        ],
        dtype=np.float64,
    )


def make_local_time_mask(local_hours, start_hour, end_hour):
    """Support both ordinary and across-midnight clock windows."""
    start_hour = float(start_hour) % 24.0
    end_hour = float(end_hour) % 24.0

    if np.isclose(start_hour, end_hour):
        return np.ones(local_hours.shape, dtype=bool)

    if start_hour < end_hour:
        return (local_hours >= start_hour) & (local_hours <= end_hour)

    return (local_hours >= start_hour) | (local_hours <= end_hour)


def format_clock_hour(hour_value):
    total_minutes = int(round(float(hour_value) * 60.0))
    hour = (total_minutes // 60) % 24
    minute = total_minutes % 60
    return f"{hour:02d}:{minute:02d}"


def minutes_from_window_start(local_hours, start_hour):
    """Convert local clock hours to a continuous axis beginning at 0 min."""
    local_hours = np.asarray(local_hours, dtype=np.float64)
    return ((local_hours - float(start_hour)) % 24.0) * 60.0


def build_window_plot_ticks(start_hour, end_hour):
    """Return 30-minute tick positions/labels for one requested analysis window."""
    duration_hours = (float(end_hour) - float(start_hour)) % 24.0
    if np.isclose(duration_hours, 0.0):
        duration_hours = 24.0

    duration_minutes = 60.0 * duration_hours
    tick_minutes = np.arange(0.0, duration_minutes + 0.1, 30.0)
    tick_labels = [
        format_clock_hour(float(start_hour) + minute / 60.0)
        for minute in tick_minutes
    ]
    return duration_minutes, tick_minutes, tick_labels


# ==================================================
# NaN-aware smoothing
# ==================================================
def smooth_keogram_2d(
    data,
    sigma_time,
    sigma_distance,
    mode="nearest",
    weight_tolerance=1.0e-12,
):
    data = np.asarray(data, dtype=np.float64)

    if data.ndim != 2:
        raise ValueError("Keogram must be two-dimensional.")

    if sigma_time < 0 or sigma_distance < 0:
        raise ValueError("Smoothing sigma values must be nonnegative.")

    if sigma_time == 0 and sigma_distance == 0:
        return data.copy()

    valid = np.isfinite(data)
    filled = np.where(valid, data, 0.0)
    weights = valid.astype(np.float64)

    sigma = (float(sigma_time), float(sigma_distance))

    smoothed_values = gaussian_filter(filled, sigma=sigma, mode=mode)
    smoothed_weights = gaussian_filter(weights, sigma=sigma, mode=mode)

    result = np.full(data.shape, np.nan, dtype=np.float64)
    good = smoothed_weights > weight_tolerance
    result[good] = smoothed_values[good] / smoothed_weights[good]
    return result


# ==================================================
# Load / interpolate saved 2-hour slope background
# ==================================================
def interpolate_finite(source_time, source_values, target_time):
    source_time = np.asarray(source_time, dtype=np.float64)
    source_values = np.asarray(source_values, dtype=np.float64)
    target_time = np.asarray(target_time, dtype=np.float64)

    valid = np.isfinite(source_time) & np.isfinite(source_values)
    if np.count_nonzero(valid) < 2:
        return np.full(target_time.shape, np.nan, dtype=np.float64)

    x = source_time[valid]
    y = source_values[valid]
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    return np.interp(target_time, x, y, left=np.nan, right=np.nan)


def load_two_hour_slope_background(plane_results_file, target_times_min):
    if not plane_results_file.exists():
        raise FileNotFoundError(
            "Could not find plane_fit_results.npz needed for 2-hour slope "
            f"bias subtraction:\n{plane_results_file}"
        )

    with np.load(plane_results_file, allow_pickle=False) as results:
        required = [
            "time_minutes",
            "preprocessed_running_mean_x_slope_amplitude_per_km",
            "preprocessed_running_mean_y_slope_amplitude_per_km",
        ]

        missing = [key for key in required if key not in results.files]
        if missing:
            raise KeyError(
                "plane_fit_results.npz is missing the saved 2-hour running-"
                "average slope fields: " + ", ".join(missing)
            )

        source_time = np.asarray(results["time_minutes"], dtype=np.float64)
        source_x = np.asarray(
            results["preprocessed_running_mean_x_slope_amplitude_per_km"],
            dtype=np.float64,
        )
        source_y = np.asarray(
            results["preprocessed_running_mean_y_slope_amplitude_per_km"],
            dtype=np.float64,
        )

        window_hours = float(
            metadata_scalar(results, "running_average_window_hours", 2.0)
        )

    x_target = interpolate_finite(source_time, source_x, target_times_min)
    y_target = interpolate_finite(source_time, source_y, target_times_min)

    return x_target, y_target, window_hours


def create_directional_two_hour_bias(
    slope_x_per_km,
    slope_y_per_km,
    distance_km,
    angle_deg,
):
    """Project the saved 2-hour plane slope onto one directional cut."""
    theta = np.deg2rad(float(angle_deg))

    slope_along_cut_per_km = (
        np.asarray(slope_x_per_km, dtype=np.float64) * np.cos(theta)
        + np.asarray(slope_y_per_km, dtype=np.float64) * np.sin(theta)
    )

    bias_keogram = (
        slope_along_cut_per_km[:, np.newaxis]
        * np.asarray(distance_km, dtype=np.float64)[np.newaxis, :]
    )

    return bias_keogram, slope_along_cut_per_km


# ==================================================
# Correlation helpers
# ==================================================
def nan_pearson_correlation(reference, comparison, minimum_points=20):
    reference = np.asarray(reference, dtype=np.float64)
    comparison = np.asarray(comparison, dtype=np.float64)

    valid = np.isfinite(reference) & np.isfinite(comparison)
    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    x = reference[valid]
    y = comparison[valid]

    x = x - np.mean(x)
    y = y - np.mean(y)

    denominator = np.sqrt(np.sum(x * x) * np.sum(y * y))
    if not np.isfinite(denominator) or denominator <= 0:
        return np.nan

    return float(np.clip(np.sum(x * y) / denominator, -1.0, 1.0))


def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    reference_row = np.asarray(reference_row, dtype=np.float64)
    comparison_row = np.asarray(comparison_row, dtype=np.float64)

    if reference_row.shape != comparison_row.shape:
        raise ValueError("Reference and comparison rows must have same shape.")

    ntime = len(reference_row)
    if abs(int(shift_pixels)) >= ntime:
        return np.nan

    shift_pixels = int(shift_pixels)

    if shift_pixels > 0:
        reference_overlap = reference_row[shift_pixels:]
        comparison_overlap = comparison_row[:-shift_pixels]
    elif shift_pixels < 0:
        shift_abs = abs(shift_pixels)
        reference_overlap = reference_row[:-shift_abs]
        comparison_overlap = comparison_row[shift_abs:]
    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference_overlap,
        comparison_overlap,
        minimum_points=minimum_points,
    )


def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    keogram = np.asarray(keogram, dtype=np.float64)
    distance_km = np.asarray(distance_km, dtype=np.float64)
    shifts_pixels = np.asarray(shifts_pixels, dtype=int)

    reference_index = int(
        np.argmin(np.abs(distance_km - float(reference_distance_km)))
    )
    selected_indices = np.flatnonzero(selected_distance_mask)
    selected_distances = distance_km[selected_indices]

    reference_row = keogram[:, reference_index]

    correlation_map = np.full(
        (len(selected_indices), len(shifts_pixels)),
        np.nan,
        dtype=np.float64,
    )

    for output_row, keogram_row in enumerate(selected_indices):
        comparison_row = keogram[:, keogram_row]
        for shift_index, shift_pixels in enumerate(shifts_pixels):
            correlation_map[output_row, shift_index] = (
                calculate_shifted_correlation(
                    reference_row,
                    comparison_row,
                    int(shift_pixels),
                    minimum_points=minimum_points,
                )
            )

    return correlation_map, selected_distances, reference_index


def find_best_shifts(correlation_map, shifts_minutes):
    correlation_map = np.asarray(correlation_map, dtype=np.float64)
    shifts_minutes = np.asarray(shifts_minutes, dtype=np.float64)

    best_shift = np.full(correlation_map.shape[0], np.nan, dtype=np.float64)
    maximum_correlation = np.full(
        correlation_map.shape[0], np.nan, dtype=np.float64
    )

    for row_index, row in enumerate(correlation_map):
        finite = np.isfinite(row)
        if not np.any(finite):
            continue

        finite_indices = np.flatnonzero(finite)
        full_index = finite_indices[int(np.argmax(row[finite]))]
        best_shift[row_index] = shifts_minutes[full_index]
        maximum_correlation[row_index] = row[full_index]

    return best_shift, maximum_correlation


def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
    minimum_correlation=0.5,
    distance_min_km=None,
    distance_max_km=None,
    remove_outliers=True,
    maximum_ridge_jump_minutes=20.0,
    outlier_sigma=3.0,
    outlier_max_iterations=10,
    outlier_min_points=3,
):
    distances_km = np.asarray(distances_km, dtype=np.float64)
    best_shift_minutes = np.asarray(best_shift_minutes, dtype=np.float64)
    maximum_correlation = np.asarray(maximum_correlation, dtype=np.float64)

    candidate_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (maximum_correlation >= minimum_correlation)
    )

    if distance_min_km is not None:
        candidate_mask &= distances_km >= float(distance_min_km)
    if distance_max_km is not None:
        candidate_mask &= distances_km <= float(distance_max_km)

    candidate_count = int(np.count_nonzero(candidate_mask))
    if candidate_count < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": candidate_count,
            "number_of_outliers_removed": 0,
            "fit_mask": candidate_mask,
            "outlier_mask": np.zeros_like(candidate_mask, dtype=bool),
        }

    retained_mask = candidate_mask.copy()

    # Retain a continuous ridge branch beginning at the smallest distance.
    if remove_outliers:
        candidate_indices = np.flatnonzero(candidate_mask)
        sorted_indices = candidate_indices[
            np.argsort(distances_km[candidate_indices])
        ]

        continuity_mask = np.zeros_like(candidate_mask, dtype=bool)
        first_index = sorted_indices[0]
        continuity_mask[first_index] = True
        previous_shift = best_shift_minutes[first_index]

        for index in sorted_indices[1:]:
            current_shift = best_shift_minutes[index]
            if abs(current_shift - previous_shift) <= maximum_ridge_jump_minutes:
                continuity_mask[index] = True
                previous_shift = current_shift

        if np.count_nonzero(continuity_mask) >= max(2, outlier_min_points):
            retained_mask = continuity_mask

    # Iterative robust residual rejection.
    if remove_outliers:
        for _ in range(int(outlier_max_iterations)):
            if np.count_nonzero(retained_mask) < max(3, outlier_min_points):
                break

            x = distances_km[retained_mask]
            y = best_shift_minutes[retained_mask]
            trial_slope, trial_intercept = np.polyfit(x, y, 1)
            residual = y - (trial_slope * x + trial_intercept)
            residual_median = np.median(residual)
            mad = np.median(np.abs(residual - residual_median))

            if not np.isfinite(mad) or np.isclose(mad, 0.0):
                break

            robust_sigma = 1.4826 * mad
            keep_local = (
                np.abs(residual - residual_median)
                <= float(outlier_sigma) * robust_sigma
            )

            if np.all(keep_local):
                break

            retained_indices = np.flatnonzero(retained_mask)
            proposed = retained_mask.copy()
            proposed[retained_indices[~keep_local]] = False

            if np.count_nonzero(proposed) < max(2, outlier_min_points):
                break

            retained_mask = proposed

    outlier_mask = candidate_mask & ~retained_mask
    fit_mask = retained_mask

    fit_distances = distances_km[fit_mask]
    fit_shifts = best_shift_minutes[fit_mask]

    if len(fit_distances) < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": len(fit_distances),
            "number_of_outliers_removed": int(np.count_nonzero(outlier_mask)),
            "fit_mask": fit_mask,
            "outlier_mask": outlier_mask,
        }

    slope, intercept = np.polyfit(fit_distances, fit_shifts, 1)
    predicted = slope * fit_distances + intercept

    ss_res = np.sum((fit_shifts - predicted) ** 2)
    ss_tot = np.sum((fit_shifts - np.mean(fit_shifts)) ** 2)
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    if np.isclose(slope, 0.0):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        speed_km_per_min = -1.0 / slope
        speed_m_per_s = speed_km_per_min * 1000.0 / 60.0

    return {
        "slope_min_per_km": float(slope),
        "intercept_minutes": float(intercept),
        "speed_km_per_min": float(speed_km_per_min),
        "speed_m_per_s": float(speed_m_per_s),
        "r_squared": float(r_squared),
        "number_of_fit_points": int(np.count_nonzero(fit_mask)),
        "number_of_outliers_removed": int(np.count_nonzero(outlier_mask)),
        "fit_mask": fit_mask,
        "outlier_mask": outlier_mask,
    }


def find_left_and_right_local_maxima(
    correlation_profile,
    shifts_minutes,
    minimum_prominence=0.0,
    minimum_distance_pixels=1,
):
    correlation_profile = np.asarray(correlation_profile, dtype=np.float64)
    shifts_minutes = np.asarray(shifts_minutes, dtype=np.float64)

    def find_one_side(side_mask):
        indices = np.flatnonzero(side_mask & np.isfinite(correlation_profile))
        if len(indices) == 0:
            return {
                "index": -1,
                "shift_minutes": np.nan,
                "correlation": np.nan,
                "used_fallback": False,
            }

        values = correlation_profile[indices]
        local_indices, _ = find_peaks(
            values,
            prominence=minimum_prominence,
            distance=minimum_distance_pixels,
        )

        used_fallback = False
        if len(local_indices) > 0:
            strongest = local_indices[int(np.nanargmax(values[local_indices]))]
            full_index = int(indices[strongest])
        else:
            used_fallback = True
            full_index = int(indices[int(np.nanargmax(values))])

        return {
            "index": full_index,
            "shift_minutes": float(shifts_minutes[full_index]),
            "correlation": float(correlation_profile[full_index]),
            "used_fallback": used_fallback,
        }

    return {
        "left": find_one_side(shifts_minutes < 0.0),
        "right": find_one_side(shifts_minutes > 0.0),
    }


# ==================================================
# Plot helpers
# ==================================================
def symmetric_keogram_limit(data, percentile=99.0):
    data = np.asarray(data, dtype=np.float64)
    finite = np.abs(data[np.isfinite(data)])
    if finite.size == 0:
        return 1.0

    limit = float(np.nanpercentile(finite, percentile))
    if not np.isfinite(limit) or limit <= 0:
        limit = float(np.nanmax(finite))
    if not np.isfinite(limit) or limit <= 0:
        limit = 1.0
    return limit


def plot_one_dataset_analysis(
    angle_deg,
    dataset_label,
    keogram,
    window_minutes,
    window_duration_minutes,
    window_tick_minutes,
    window_tick_labels,
    time_range_label,
    distance_km,
    selected_distance_mask,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_km_profile,
    zero_profile_shifts_minutes,
    zero_km_peaks,
    output_file,
):
    """Compact four-panel diagnostic figure for one analysis window."""
    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(11, 15.5),
        gridspec_kw={"height_ratios": [1.6, 1.1, 2.4, 1.1]},
    )

    ax_keogram, ax_profiles, ax_map, ax_autocorr = axes

    # --------------------------------------------------
    # Selected local-time keogram
    # --------------------------------------------------
    selected_keogram = keogram[:, selected_distance_mask]
    selected_distance = distance_km[selected_distance_mask]
    abs_limit = symmetric_keogram_limit(
        selected_keogram,
        percentile=KEOGRAM_COLOR_PERCENTILE,
    )

    mesh = ax_keogram.pcolormesh(
        window_minutes,
        selected_distance,
        selected_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-abs_limit,
        vmax=abs_limit,
    )
    cbar = fig.colorbar(mesh, ax=ax_keogram, fraction=0.025, pad=0.02)
    cbar.set_label("Keogram value")

    ax_keogram.set_xlim(0.0, window_duration_minutes)
    ax_keogram.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )
    ax_keogram.set_xticks(window_tick_minutes)
    ax_keogram.set_xticklabels(window_tick_labels)
    ax_keogram.set_xlabel("Local time")
    ax_keogram.set_ylabel("Distance along cut (km)")
    ax_keogram.set_title(
        f"{angle_deg:+d}° Keogram, {time_range_label} Local Time\n{dataset_label}"
    )

    # --------------------------------------------------
    # Profiles at 0, 125, 250 km
    # --------------------------------------------------
    profile_targets = [0.0, 125.0, 250.0]
    for target in profile_targets:
        index = int(np.argmin(np.abs(distance_km - target)))
        ax_profiles.plot(
            window_minutes,
            keogram[:, index],
            linewidth=1.2,
            label=f"{distance_km[index]:g} km",
        )

    ax_profiles.axhline(0.0, linewidth=0.8, linestyle="--")
    ax_profiles.set_xlim(0.0, window_duration_minutes)
    ax_profiles.set_xticks(window_tick_minutes)
    ax_profiles.set_xticklabels(window_tick_labels)
    ax_profiles.set_xlabel("Local time")
    ax_profiles.set_ylabel("Keogram value")
    ax_profiles.set_title("Profiles at 0, 125, and 250 km")
    ax_profiles.grid(True, alpha=0.3)
    ax_profiles.legend(loc="best")

    # --------------------------------------------------
    # Correlation map and ridge
    # --------------------------------------------------
    if len(shifts_minutes) > 1:
        shift_spacing = float(np.median(np.diff(shifts_minutes)))
    else:
        shift_spacing = 1.0

    if len(selected_distances) > 1:
        distance_spacing = float(np.median(np.diff(selected_distances)))
    else:
        distance_spacing = 1.0

    image = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=CORRELATION_CMAP,
        vmin=CORRELATION_VMIN,
        vmax=CORRELATION_VMAX,
        extent=[
            shifts_minutes[0] - shift_spacing / 2.0,
            shifts_minutes[-1] + shift_spacing / 2.0,
            selected_distances[0] - distance_spacing / 2.0,
            selected_distances[-1] + distance_spacing / 2.0,
        ],
    )
    cbar_map = fig.colorbar(image, ax=ax_map, fraction=0.025, pad=0.02)
    cbar_map.set_label("Pearson correlation with 0-km row")

    ax_map.axvline(0.0, linestyle="--", linewidth=1.0)

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (maximum_correlation >= MIN_CORRELATION_FOR_SLOPE)
    )

    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            linewidth=1.2,
            label="Maximum correlation",
        )
        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            s=9,
        )

    outlier_mask = slope_results.get(
        "outlier_mask", np.zeros_like(ridge_mask, dtype=bool)
    )
    if np.any(outlier_mask):
        ax_map.scatter(
            best_shift_minutes[outlier_mask],
            selected_distances[outlier_mask],
            marker="x",
            s=35,
            linewidth=1.1,
            label="Rejected ridge point",
        )

    slope = slope_results["slope_min_per_km"]
    intercept = slope_results["intercept_minutes"]
    speed = slope_results["speed_m_per_s"]
    r_squared = slope_results["r_squared"]

    if np.isfinite(slope):
        fit_distance = np.linspace(
            SLOPE_FIT_DISTANCE_MIN_KM,
            SLOPE_FIT_DISTANCE_MAX_KM,
            300,
        )
        fit_shift = slope * fit_distance + intercept
        ax_map.plot(
            fit_shift,
            fit_distance,
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        ax_map.text(
            0.02,
            0.03,
            (
                f"slope = {slope:.5f} min/km\n"
                f"speed = {speed:.1f} m/s\n"
                f"R² = {r_squared:.3f}\n"
                f"points = {slope_results['number_of_fit_points']}\n"
                f"rejected = {slope_results['number_of_outliers_removed']}"
            ),
            transform=ax_map.transAxes,
            va="bottom",
            ha="left",
            fontsize=9,
            bbox={"facecolor": "white", "edgecolor": "0.4", "alpha": 0.85},
        )

    ax_map.set_xlim(SHIFT_MIN_MINUTES, SHIFT_MAX_MINUTES)
    ax_map.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )
    ax_map.set_xlabel("Shift applied to row (min)")
    ax_map.set_ylabel("Distance along cut (km)")
    ax_map.set_title("Shifted-row correlation and maximum-correlation ridge")
    ax_map.legend(loc="upper right", fontsize=8)

    # --------------------------------------------------
    # 0-km autocorrelation
    # --------------------------------------------------
    ax_autocorr.plot(
        zero_profile_shifts_minutes,
        zero_km_profile,
        linewidth=1.3,
        label="0-km autocorrelation",
    )
    ax_autocorr.axvline(0.0, linestyle="--", linewidth=0.9)

    for side, marker in (("left", "o"), ("right", "s")):
        peak = zero_km_peaks[side]
        if np.isfinite(peak["shift_minutes"]):
            ax_autocorr.scatter(
                peak["shift_minutes"],
                peak["correlation"],
                marker=marker,
                s=50,
                zorder=4,
                label=(
                    f"{side.capitalize()} max: "
                    f"{peak['shift_minutes']:.1f} min"
                ),
            )

    ax_autocorr.set_xlim(
        ZERO_KM_PROFILE_SHIFT_MIN_MINUTES,
        ZERO_KM_PROFILE_SHIFT_MAX_MINUTES,
    )
    ax_autocorr.set_xlabel("Shift (min)")
    ax_autocorr.set_ylabel("Correlation")
    ax_autocorr.set_title("0-km autocorrelation profile")
    ax_autocorr.grid(True, alpha=0.3)
    ax_autocorr.legend(loc="best", fontsize=8)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_phase_speed_plot(combined_results, output_file, time_range_label):
    if combined_results.size == 0:
        return

    angle = combined_results[:, 0]
    processed_speed = combined_results[:, 4]
    corrected_speed = combined_results[:, 10]

    angle_mask = (
        np.isfinite(angle)
        & (angle >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    angle_plot = angle[angle_mask]
    processed_plot = processed_speed[angle_mask].copy()
    corrected_plot = corrected_speed[angle_mask].copy()

    order = np.argsort(angle_plot)
    angle_plot = angle_plot[order]
    processed_plot = processed_plot[order]
    corrected_plot = corrected_plot[order]

    fig, ax = plt.subplots(figsize=(10.5, 5.8))

    ax.plot(
        angle_plot,
        processed_plot,
        marker="o",
        linewidth=1.5,
        label="Processed",
    )

    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        ax.plot(
            angle_plot,
            corrected_plot,
            marker="s",
            linewidth=1.5,
            label="Processed minus 2-hr slope bias",
        )

    # Mark the weighted-angle estimate for each curve.
    weighted_specs = [
        (
            4,
            5,
            "Processed weighted angle",
            "*",
        ),
    ]
    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        weighted_specs.append(
            (
                10,
                11,
                "Minus-bias weighted angle",
                "*",
            )
        )

    for speed_column, r2_column, label, marker in weighted_specs:
        summary = get_weighted_speed_summary(
            combined_results,
            speed_column=speed_column,
            r2_column=r2_column,
        )

        weighted_angle = summary["weighted_angle_deg"]
        weighted_speed = summary["speed_at_weighted_angle_m_per_s"]
        weighted_r2 = summary["r_squared_at_weighted_angle"]

        if np.isfinite(weighted_angle) and np.isfinite(weighted_speed):
            ax.scatter(
                weighted_angle,
                weighted_speed,
                marker=marker,
                s=180,
                zorder=6,
                label=(
                    f"{label}: {weighted_speed:.1f} m/s at "
                    f"{weighted_angle:.1f}° (R²={weighted_r2:.3f})"
                ),
            )
            ax.axvline(
                weighted_angle,
                linestyle="--",
                linewidth=0.9,
                alpha=0.55,
            )

    ax.set_xlim(
        PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
        PHASE_SPEED_PLOT_ANGLE_MAX_DEG,
    )
    displayed_values = [processed_plot[np.isfinite(processed_plot)]]
    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        displayed_values.append(corrected_plot[np.isfinite(corrected_plot)])

    displayed_values = [
        values for values in displayed_values if values.size > 0
    ]
    if displayed_values:
        all_displayed = np.concatenate(displayed_values)
        y_min = float(np.nanmin(all_displayed))
        y_max = float(np.nanmax(all_displayed))
        y_span = y_max - y_min
        padding = max(
            PHASE_SPEED_Y_MIN_PADDING_M_PER_S,
            PHASE_SPEED_Y_PADDING_FRACTION * max(y_span, abs(y_min), abs(y_max)),
        )
        if np.isclose(y_span, 0.0):
            y_min -= padding
            y_max += padding
        else:
            y_min -= padding
            y_max += padding
        ax.set_ylim(y_min, y_max)

    ax.set_xticks(np.arange(30, 151, 10))
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("Equivalent phase speed (m/s)")
    ax.set_title(
        f"Phase Speed versus Angle, {time_range_label} Local Time\n"
        "Stars mark the phase speed evaluated at the weighted angle"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def save_r_squared_plot(combined_results, output_file, time_range_label):
    if combined_results.size == 0:
        return

    angle = combined_results[:, 0]
    processed_r2 = combined_results[:, 5]
    corrected_r2 = combined_results[:, 11]

    angle_mask = (
        np.isfinite(angle)
        & (angle >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    angle_plot = angle[angle_mask]
    processed_plot = processed_r2[angle_mask]
    corrected_plot = corrected_r2[angle_mask]

    order = np.argsort(angle_plot)
    angle_plot = angle_plot[order]
    processed_plot = processed_plot[order]
    corrected_plot = corrected_plot[order]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        angle_plot,
        processed_plot,
        marker="o",
        linewidth=1.5,
        label="Processed",
    )

    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        ax.plot(
            angle_plot,
            corrected_plot,
            marker="s",
            linewidth=1.5,
            label="Processed minus 2-hr slope bias",
        )

    ax.set_xlim(
        PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
        PHASE_SPEED_PLOT_ANGLE_MAX_DEG,
    )
    ax.set_xticks(np.arange(30, 151, 10))
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("R²")
    ax.set_title(
        f"Maximum-Correlation Ridge Fit R², {time_range_label} Local Time"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Main analysis
# ==================================================
def analyze_window(window_name, start_hour, end_hour):
    time_range_label = (
        f"{format_clock_hour(start_hour)}–{format_clock_hour(end_hour)}"
    )

    OUT_DIR = ANALYSIS_ROOT / f"analysis_{window_name}"
    PLOTS_DIR = OUT_DIR / "plots"
    DATA_DIR = OUT_DIR / "data"
    for directory in (OUT_DIR, PLOTS_DIR, DATA_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    window_duration_minutes, window_tick_minutes, window_tick_labels = (
        build_window_plot_ticks(start_hour, end_hour)
    )

    if not KEOGRAM_DIR.exists():
        raise FileNotFoundError(f"Keogram directory does not exist:\n{KEOGRAM_DIR}")

    if not KEOGRAM_METADATA_FILE.exists():
        raise FileNotFoundError(
            "Could not find keogram_metadata.npz:\n"
            f"{KEOGRAM_METADATA_FILE}\n"
            "Use the new keogram-generation script first."
        )

    # --------------------------------------------------
    # Read common time / local-time metadata
    # --------------------------------------------------
    with np.load(KEOGRAM_METADATA_FILE, allow_pickle=False) as metadata:
        times_min = np.asarray(metadata["time_minutes"], dtype=np.float64)
        local_datetimes = build_local_datetimes(metadata, times_min)

    if len(times_min) < 2:
        raise ValueError("At least two time samples are required.")

    time_differences = np.diff(times_min)
    finite_dt = time_differences[np.isfinite(time_differences)]
    if finite_dt.size == 0:
        raise ValueError("Could not determine time spacing.")

    minutes_per_pixel = float(np.median(finite_dt))
    if not np.isfinite(minutes_per_pixel) or minutes_per_pixel <= 0:
        raise ValueError("Invalid time spacing in saved metadata.")

    local_hours = local_clock_hours(local_datetimes)
    time_mask = make_local_time_mask(
        local_hours,
        start_hour,
        end_hour,
    )

    selected_indices = np.flatnonzero(time_mask)
    if len(selected_indices) < MIN_VALID_CORRELATION_POINTS:
        raise ValueError(
            f"Too few samples fall between {time_range_label} local time. "
            f"Found only {len(selected_indices)} samples."
        )

    times_selected_min = times_min[time_mask]
    local_datetimes_selected = local_datetimes[time_mask]
    local_hours_selected = local_hours[time_mask]
    window_minutes = minutes_from_window_start(local_hours_selected, start_hour)

    # Ensure chronological order in case the generic clock mask is ever used
    # with an across-midnight interval.
    order = np.argsort(times_selected_min)
    times_selected_min = times_selected_min[order]
    local_datetimes_selected = local_datetimes_selected[order]
    window_minutes = window_minutes[order]

    print("=" * 78)
    print("BLO KEOGRAM CORRELATION ANALYSIS")
    print("=" * 78)
    print(f"Keogram directory: {KEOGRAM_DIR}")
    print(f"Output directory:  {OUT_DIR}")
    print(f"Local-time interval: {time_range_label}")
    print(f"Selected samples: {len(times_selected_min)}")
    print(
        "Selected local datetimes: "
        f"{local_datetimes_selected[0]} to {local_datetimes_selected[-1]}"
    )
    print(f"One time pixel: {minutes_per_pixel:.6f} min")

    # --------------------------------------------------
    # Shift axes
    # --------------------------------------------------
    shift_min_pixels = int(np.ceil(SHIFT_MIN_MINUTES / minutes_per_pixel))
    shift_max_pixels = int(np.floor(SHIFT_MAX_MINUTES / minutes_per_pixel))
    shifts_pixels = np.arange(
        shift_min_pixels, shift_max_pixels + 1, dtype=int
    )
    shifts_minutes = shifts_pixels * minutes_per_pixel

    zero_min_pixels = int(
        np.ceil(ZERO_KM_PROFILE_SHIFT_MIN_MINUTES / minutes_per_pixel)
    )
    zero_max_pixels = int(
        np.floor(ZERO_KM_PROFILE_SHIFT_MAX_MINUTES / minutes_per_pixel)
    )
    zero_profile_shifts_pixels = np.arange(
        zero_min_pixels, zero_max_pixels + 1, dtype=int
    )
    zero_profile_shifts_minutes = (
        zero_profile_shifts_pixels * minutes_per_pixel
    )

    # --------------------------------------------------
    # Load saved 2-hour slope background once
    # --------------------------------------------------
    if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
        (
            slope_x_selected,
            slope_y_selected,
            running_average_window_hours,
        ) = load_two_hour_slope_background(
            PLANE_RESULTS_FILE,
            times_selected_min,
        )

        print(
            "2-hour bias correction: enabled "
            f"(saved window = {running_average_window_hours:g} hr)"
        )
    else:
        slope_x_selected = np.full(times_selected_min.shape, np.nan)
        slope_y_selected = np.full(times_selected_min.shape, np.nan)
        running_average_window_hours = np.nan
        print("2-hour bias correction: disabled")

    # Save common analysis coordinates.
    np.save(DATA_DIR / "angles_deg.npy", KEOGRAM_ANGLES_DEG)
    np.save(DATA_DIR / "shifts_minutes.npy", shifts_minutes)
    np.save(DATA_DIR / "shifts_pixels.npy", shifts_pixels)
    np.save(
        DATA_DIR / "zero_km_profile_shifts_minutes.npy",
        zero_profile_shifts_minutes,
    )
    np.save(
        DATA_DIR / "selected_elapsed_times_min.npy",
        times_selected_min,
    )
    np.save(
        DATA_DIR / "selected_window_minutes.npy",
        window_minutes,
    )

    combined_results = []
    combined_peak_results = []
    successful_angles = []
    failed_angles = []

    # --------------------------------------------------
    # Process all angles
    # --------------------------------------------------
    for angle_number, angle_deg in enumerate(KEOGRAM_ANGLES_DEG, start=1):
        print("\n" + "-" * 78)
        print(
            f"Angle {angle_deg:+d}° "
            f"({angle_number}/{len(KEOGRAM_ANGLES_DEG)})"
        )

        input_label = input_angle_label(angle_deg)
        output_label = output_angle_label(angle_deg)

        keogram_file = KEOGRAM_DIR / f"BLO_keogram_angle_{input_label}.npy"
        distance_file = (
            KEOGRAM_DIR
            / f"BLO_keogram_distance_km_angle_{input_label}.npy"
        )

        if not keogram_file.exists():
            failed_angles.append((int(angle_deg), "keogram file not found"))
            print(f"Missing: {keogram_file}")
            continue

        if not distance_file.exists():
            failed_angles.append((int(angle_deg), "distance file not found"))
            print(f"Missing: {distance_file}")
            continue

        try:
            keogram_original = np.asarray(np.load(keogram_file), dtype=np.float64)
            distance_km = np.asarray(np.load(distance_file), dtype=np.float64)

            if keogram_original.ndim != 2:
                raise ValueError(
                    f"Expected 2-D keogram, got {keogram_original.shape}."
                )

            if keogram_original.shape[0] != len(times_min):
                raise ValueError(
                    f"Keogram has {keogram_original.shape[0]} time samples, "
                    f"metadata has {len(times_min)}."
                )

            if keogram_original.shape[1] != len(distance_km):
                raise ValueError(
                    "Keogram distance dimension does not match saved "
                    "distance-axis file."
                )

            # Apply the local-time selection and chronological ordering.
            keogram_selected = keogram_original[time_mask, :][order, :]

            if APPLY_2D_SMOOTHING:
                keogram_processed = smooth_keogram_2d(
                    keogram_selected,
                    sigma_time=SMOOTH_SIGMA_TIME_PIXELS,
                    sigma_distance=SMOOTH_SIGMA_DISTANCE_PIXELS,
                    mode=SMOOTH_MODE,
                    weight_tolerance=SMOOTH_WEIGHT_TOLERANCE,
                )
                processed_label = (
                    "Processed keogram "
                    f"(Gaussian σt={SMOOTH_SIGMA_TIME_PIXELS:g}, "
                    f"σd={SMOOTH_SIGMA_DISTANCE_PIXELS:g})"
                )
            else:
                keogram_processed = keogram_selected.copy()
                processed_label = "Processed keogram (no additional smoothing)"

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                bias_keogram, slope_along_cut = create_directional_two_hour_bias(
                    slope_x_selected,
                    slope_y_selected,
                    distance_km,
                    angle_deg,
                )

                keogram_minus_bias = keogram_processed - bias_keogram
                keogram_minus_bias[~np.isfinite(keogram_processed)] = np.nan
                corrected_label = (
                    f"Processed minus {running_average_window_hours:g}-hr "
                    "running-average plane slope"
                )
            else:
                bias_keogram = np.zeros_like(keogram_processed)
                slope_along_cut = np.full(times_selected_min.shape, np.nan)
                keogram_minus_bias = keogram_processed.copy()
                corrected_label = "Bias subtraction disabled"

            selected_distance_mask = (
                (distance_km >= CORRELATION_DISTANCE_MIN_KM)
                & (distance_km <= CORRELATION_DISTANCE_MAX_KM)
            )

            if not np.any(selected_distance_mask):
                raise ValueError("No distances lie in the requested range.")

            # ------------------------------------------
            # Processed correlation map
            # ------------------------------------------
            (
                correlation_processed,
                selected_distances,
                _,
            ) = calculate_correlation_map(
                keogram_processed,
                distance_km,
                reference_distance_km=0.0,
                selected_distance_mask=selected_distance_mask,
                shifts_pixels=shifts_pixels,
                minimum_points=MIN_VALID_CORRELATION_POINTS,
            )

            best_shift_processed, max_corr_processed = find_best_shifts(
                correlation_processed,
                shifts_minutes,
            )

            slope_processed = fit_maximum_correlation_slope(
                selected_distances,
                best_shift_processed,
                max_corr_processed,
                minimum_correlation=MIN_CORRELATION_FOR_SLOPE,
                distance_min_km=SLOPE_FIT_DISTANCE_MIN_KM,
                distance_max_km=SLOPE_FIT_DISTANCE_MAX_KM,
                remove_outliers=REMOVE_RIDGE_OUTLIERS,
                maximum_ridge_jump_minutes=RIDGE_MAXIMUM_JUMP_MINUTES,
                outlier_sigma=RIDGE_OUTLIER_SIGMA,
                outlier_max_iterations=RIDGE_OUTLIER_MAX_ITERATIONS,
                outlier_min_points=RIDGE_OUTLIER_MIN_POINTS,
            )

            # ------------------------------------------
            # Minus-2hr-bias correlation map
            # ------------------------------------------
            (
                correlation_corrected,
                selected_distances_corrected,
                _,
            ) = calculate_correlation_map(
                keogram_minus_bias,
                distance_km,
                reference_distance_km=0.0,
                selected_distance_mask=selected_distance_mask,
                shifts_pixels=shifts_pixels,
                minimum_points=MIN_VALID_CORRELATION_POINTS,
            )

            best_shift_corrected, max_corr_corrected = find_best_shifts(
                correlation_corrected,
                shifts_minutes,
            )

            slope_corrected = fit_maximum_correlation_slope(
                selected_distances_corrected,
                best_shift_corrected,
                max_corr_corrected,
                minimum_correlation=MIN_CORRELATION_FOR_SLOPE,
                distance_min_km=SLOPE_FIT_DISTANCE_MIN_KM,
                distance_max_km=SLOPE_FIT_DISTANCE_MAX_KM,
                remove_outliers=REMOVE_RIDGE_OUTLIERS,
                maximum_ridge_jump_minutes=RIDGE_MAXIMUM_JUMP_MINUTES,
                outlier_sigma=RIDGE_OUTLIER_SIGMA,
                outlier_max_iterations=RIDGE_OUTLIER_MAX_ITERATIONS,
                outlier_min_points=RIDGE_OUTLIER_MIN_POINTS,
            )

            # ------------------------------------------
            # 0-km autocorrelation profiles
            # ------------------------------------------
            zero_index = int(np.argmin(np.abs(distance_km)))

            zero_processed = keogram_processed[:, zero_index]
            zero_corrected = keogram_minus_bias[:, zero_index]

            zero_profile_processed = np.array(
                [
                    calculate_shifted_correlation(
                        zero_processed,
                        zero_processed,
                        int(shift),
                        minimum_points=MIN_VALID_CORRELATION_POINTS,
                    )
                    for shift in zero_profile_shifts_pixels
                ],
                dtype=np.float64,
            )

            zero_profile_corrected = np.array(
                [
                    calculate_shifted_correlation(
                        zero_corrected,
                        zero_corrected,
                        int(shift),
                        minimum_points=MIN_VALID_CORRELATION_POINTS,
                    )
                    for shift in zero_profile_shifts_pixels
                ],
                dtype=np.float64,
            )

            peaks_processed = find_left_and_right_local_maxima(
                zero_profile_processed,
                zero_profile_shifts_minutes,
                minimum_prominence=LOCAL_MAXIMUM_MIN_PROMINENCE,
                minimum_distance_pixels=LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS,
            )

            peaks_corrected = find_left_and_right_local_maxima(
                zero_profile_corrected,
                zero_profile_shifts_minutes,
                minimum_prominence=LOCAL_MAXIMUM_MIN_PROMINENCE,
                minimum_distance_pixels=LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS,
            )

            # ------------------------------------------
            # Plots -- all flat in plots/
            # ------------------------------------------
            processed_plot_file = (
                PLOTS_DIR
                / f"angle_{output_label}_processed.png"
            )
            corrected_plot_file = (
                PLOTS_DIR
                / f"angle_{output_label}_minus_2hr_bias.png"
            )

            plot_one_dataset_analysis(
                angle_deg,
                processed_label,
                keogram_processed,
                window_minutes,
                window_duration_minutes,
                window_tick_minutes,
                window_tick_labels,
                time_range_label,
                distance_km,
                selected_distance_mask,
                correlation_processed,
                shifts_minutes,
                selected_distances,
                best_shift_processed,
                max_corr_processed,
                slope_processed,
                zero_profile_processed,
                zero_profile_shifts_minutes,
                peaks_processed,
                processed_plot_file,
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                plot_one_dataset_analysis(
                    angle_deg,
                    corrected_label,
                    keogram_minus_bias,
                    window_minutes,
                    window_duration_minutes,
                    window_tick_minutes,
                    window_tick_labels,
                    time_range_label,
                    distance_km,
                    selected_distance_mask,
                    correlation_corrected,
                    shifts_minutes,
                    selected_distances_corrected,
                    best_shift_corrected,
                    max_corr_corrected,
                    slope_corrected,
                    zero_profile_corrected,
                    zero_profile_shifts_minutes,
                    peaks_corrected,
                    corrected_plot_file,
                )

            # ------------------------------------------
            # Data products -- all flat in data/
            # ------------------------------------------
            np.save(
                DATA_DIR / f"correlation_{output_label}_processed.npy",
                correlation_processed,
            )
            np.save(
                DATA_DIR / f"best_shift_{output_label}_processed.npy",
                best_shift_processed,
            )
            np.save(
                DATA_DIR / f"max_correlation_{output_label}_processed.npy",
                max_corr_processed,
            )
            np.save(
                DATA_DIR / f"distances_{output_label}.npy",
                selected_distances,
            )
            np.save(
                DATA_DIR / f"zero_km_autocorrelation_{output_label}_processed.npy",
                zero_profile_processed,
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                np.save(
                    DATA_DIR / f"correlation_{output_label}_minus_2hr_bias.npy",
                    correlation_corrected,
                )
                np.save(
                    DATA_DIR / f"best_shift_{output_label}_minus_2hr_bias.npy",
                    best_shift_corrected,
                )
                np.save(
                    DATA_DIR / f"max_correlation_{output_label}_minus_2hr_bias.npy",
                    max_corr_corrected,
                )
                np.save(
                    DATA_DIR / f"zero_km_autocorrelation_{output_label}_minus_2hr_bias.npy",
                    zero_profile_corrected,
                )
                np.save(
                    DATA_DIR / f"two_hour_bias_keogram_{output_label}.npy",
                    bias_keogram,
                )
                np.save(
                    DATA_DIR / f"two_hour_slope_along_cut_{output_label}.npy",
                    slope_along_cut,
                )

            ridge_processed_csv = np.column_stack(
                [selected_distances, best_shift_processed, max_corr_processed]
            )
            ridge_corrected_csv = np.column_stack(
                [
                    selected_distances_corrected,
                    best_shift_corrected,
                    max_corr_corrected,
                ]
            )

            np.savetxt(
                DATA_DIR / f"ridge_{output_label}_processed.csv",
                ridge_processed_csv,
                delimiter=",",
                header="distance_km,best_shift_minutes,maximum_correlation",
                comments="",
            )

            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                np.savetxt(
                    DATA_DIR / f"ridge_{output_label}_minus_2hr_bias.csv",
                    ridge_corrected_csv,
                    delimiter=",",
                    header="distance_km,best_shift_minutes,maximum_correlation",
                    comments="",
                )

            # ------------------------------------------
            # Combined summaries
            # ------------------------------------------
            combined_results.append(
                [
                    float(angle_deg),
                    slope_processed["slope_min_per_km"],
                    slope_processed["intercept_minutes"],
                    slope_processed["speed_km_per_min"],
                    slope_processed["speed_m_per_s"],
                    slope_processed["r_squared"],
                    slope_processed["number_of_fit_points"],
                    slope_corrected["slope_min_per_km"],
                    slope_corrected["intercept_minutes"],
                    slope_corrected["speed_km_per_min"],
                    slope_corrected["speed_m_per_s"],
                    slope_corrected["r_squared"],
                    slope_corrected["number_of_fit_points"],
                ]
            )

            combined_peak_results.append(
                [
                    float(angle_deg),
                    peaks_processed["left"]["shift_minutes"],
                    peaks_processed["left"]["correlation"],
                    peaks_processed["right"]["shift_minutes"],
                    peaks_processed["right"]["correlation"],
                    peaks_corrected["left"]["shift_minutes"],
                    peaks_corrected["left"]["correlation"],
                    peaks_corrected["right"]["shift_minutes"],
                    peaks_corrected["right"]["correlation"],
                ]
            )

            successful_angles.append(int(angle_deg))

            print(
                "  Processed: "
                f"speed={slope_processed['speed_m_per_s']:.2f} m/s, "
                f"R²={slope_processed['r_squared']:.3f}"
            )
            if APPLY_2HR_SLOPE_BIAS_SUBTRACTION:
                print(
                    "  Minus 2-hr bias: "
                    f"speed={slope_corrected['speed_m_per_s']:.2f} m/s, "
                    f"R²={slope_corrected['r_squared']:.3f}"
                )

        except Exception as exc:
            failed_angles.append((int(angle_deg), str(exc)))
            print(f"ERROR: {exc}")

    # --------------------------------------------------
    # Save combined summaries
    # --------------------------------------------------
    combined_results = np.asarray(combined_results, dtype=np.float64)
    combined_peak_results = np.asarray(combined_peak_results, dtype=np.float64)

    slope_header = (
        "angle_deg,"
        "processed_slope_min_per_km,"
        "processed_intercept_minutes,"
        "processed_speed_km_per_min,"
        "processed_speed_m_per_s,"
        "processed_r_squared,"
        "processed_number_of_fit_points,"
        "minus_2hr_bias_slope_min_per_km,"
        "minus_2hr_bias_intercept_minutes,"
        "minus_2hr_bias_speed_km_per_min,"
        "minus_2hr_bias_speed_m_per_s,"
        "minus_2hr_bias_r_squared,"
        "minus_2hr_bias_number_of_fit_points"
    )

    if combined_results.size > 0:
        np.savetxt(
            OUT_DIR / "all_angle_slope_summary.csv",
            combined_results,
            delimiter=",",
            header=slope_header,
            comments="",
        )
        np.save(OUT_DIR / "all_angle_slope_summary.npy", combined_results)

        save_phase_speed_plot(
            combined_results,
            OUT_DIR / "phase_speed_vs_angle.png",
            time_range_label,
        )
        save_r_squared_plot(
            combined_results,
            OUT_DIR / "r_squared_vs_angle.png",
            time_range_label,
        )

    peak_header = (
        "angle_deg,"
        "processed_left_shift_minutes,processed_left_correlation,"
        "processed_right_shift_minutes,processed_right_correlation,"
        "minus_2hr_bias_left_shift_minutes,minus_2hr_bias_left_correlation,"
        "minus_2hr_bias_right_shift_minutes,minus_2hr_bias_right_correlation"
    )

    if combined_peak_results.size > 0:
        np.savetxt(
            OUT_DIR / "all_angle_zero_km_local_maxima.csv",
            combined_peak_results,
            delimiter=",",
            header=peak_header,
            comments="",
        )

    # --------------------------------------------------
    # Processing log
    # --------------------------------------------------
    with open(OUT_DIR / "processing_log.txt", "w", encoding="utf-8") as file:
        file.write(f"Keogram directory: {KEOGRAM_DIR}\n")
        file.write(f"Plane results: {PLANE_RESULTS_FILE}\n")
        file.write(f"Local-time interval: {time_range_label}\n")
        file.write(
            "Selected local datetimes: "
            f"{local_datetimes_selected[0]} to {local_datetimes_selected[-1]}\n"
        )
        file.write(f"Selected samples: {len(times_selected_min)}\n")
        file.write(f"Minutes per pixel: {minutes_per_pixel:.10f}\n")
        file.write(
            "2-hour slope bias subtraction: "
            f"{APPLY_2HR_SLOPE_BIAS_SUBTRACTION}\n"
        )
        if np.isfinite(running_average_window_hours):
            file.write(
                f"Saved running-average window: "
                f"{running_average_window_hours:g} hr\n"
            )
        file.write("\nSuccessful angles:\n")
        file.write(", ".join(str(v) for v in successful_angles) + "\n")
        file.write("\nFailed angles:\n")
        if failed_angles:
            for angle, reason in failed_angles:
                file.write(f"{angle}: {reason}\n")
        else:
            file.write("None\n")

    print("\n" + "=" * 78)
    print(f"ANALYSIS COMPLETE: {time_range_label}")
    print("=" * 78)
    print(f"Successful angles: {len(successful_angles)}")
    print(f"Failed angles:     {len(failed_angles)}")
    print(f"Output folder:     {OUT_DIR.resolve()}")
    print("\nDirectory structure:")
    print(f"  {OUT_DIR.name}/")
    print("    plots/")
    print("    data/")
    print("    all_angle_slope_summary.csv")
    print("    phase_speed_vs_angle.png")
    print("    r_squared_vs_angle.png")
    print("    processing_log.txt")

    return {
        "window_name": window_name,
        "start_hour": float(start_hour),
        "end_hour": float(end_hour),
        "duration_hours": float(window_duration_minutes / 60.0),
        "time_range_label": time_range_label,
        "combined_results": combined_results,
        "successful_angles": len(successful_angles),
        "failed_angles": len(failed_angles),
    }


# ==================================================
# Cross-window weighted summary table
# ==================================================
def get_weighted_speed_summary(combined_results, speed_column, r2_column):
    """
    Return a weighted angle plus phase speed and R² evaluated at that angle.

    IMPORTANT:
        Only the ANGLE is averaged.

    Weighting philosophy:
        - lower |phase speed| => larger angle weight
        - optionally, larger R² => larger angle weight
        - only the low-speed trough is included

    After the weighted angle is calculated, the reported phase speed and R²
    are obtained by linear interpolation of the original angle curves. Thus,
    phase speed and R² are not averaged across angles.
    """
    combined_results = np.asarray(combined_results, dtype=np.float64)

    empty = {
        "weighted_angle_deg": np.nan,
        "speed_at_weighted_angle_m_per_s": np.nan,
        "r_squared_at_weighted_angle": np.nan,
        "minimum_speed_magnitude_m_per_s": np.nan,
        "number_of_weighted_angles": 0,
        "weight_sum": np.nan,
    }

    if combined_results.ndim != 2 or combined_results.shape[0] == 0:
        return empty

    angle_all = combined_results[:, 0]
    signed_speed_all = combined_results[:, speed_column]
    r_squared_all = combined_results[:, r2_column]
    speed_magnitude_all = np.abs(signed_speed_all)

    # Valid points for the underlying phase-speed curve. R² does not need to
    # be finite here because speed interpolation can still be performed.
    curve_valid = (
        np.isfinite(angle_all)
        & np.isfinite(signed_speed_all)
        & np.isfinite(speed_magnitude_all)
        & (speed_magnitude_all > 0.0)
        & (angle_all >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (angle_all <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    if not np.any(curve_valid):
        return empty

    curve_angle = angle_all[curve_valid]
    curve_speed = signed_speed_all[curve_valid]
    curve_r2 = r_squared_all[curve_valid]
    curve_speed_magnitude = speed_magnitude_all[curve_valid]

    order = np.argsort(curve_angle)
    curve_angle = curve_angle[order]
    curve_speed = curve_speed[order]
    curve_r2 = curve_r2[order]
    curve_speed_magnitude = curve_speed_magnitude[order]

    # Points used to calculate the weighted ANGLE.
    weight_valid = np.ones(curve_angle.shape, dtype=bool)
    if WEIGHT_BY_R_SQUARED:
        weight_valid &= np.isfinite(curve_r2)

    if not np.any(weight_valid):
        return empty

    weight_angle = curve_angle[weight_valid]
    weight_speed_magnitude = curve_speed_magnitude[weight_valid]
    weight_r2 = curve_r2[weight_valid]

    minimum_speed_magnitude = float(np.min(weight_speed_magnitude))
    trough_mask = (
        weight_speed_magnitude
        <= float(WEIGHTED_SPEED_MAX_FACTOR) * minimum_speed_magnitude
    )

    weight_angle = weight_angle[trough_mask]
    weight_speed_magnitude = weight_speed_magnitude[trough_mask]
    weight_r2 = weight_r2[trough_mask]

    if weight_angle.size == 0:
        return empty

    speed_weights = (
        minimum_speed_magnitude / weight_speed_magnitude
    ) ** float(WEIGHTED_SPEED_POWER)

    if WEIGHT_BY_R_SQUARED:
        quality = np.clip(weight_r2, 0.0, 1.0)
        quality_weights = quality ** float(WEIGHTED_R2_POWER)
        weights = speed_weights * quality_weights
    else:
        weights = speed_weights

    finite_weight = np.isfinite(weights) & (weights > 0.0)
    if not np.any(finite_weight):
        # If every quality weight is zero, fall back to speed-only weighting.
        weights = speed_weights
        finite_weight = np.isfinite(weights) & (weights > 0.0)

    weight_angle = weight_angle[finite_weight]
    weights = weights[finite_weight]

    if weight_angle.size == 0:
        return empty

    weight_sum = float(np.sum(weights))
    if not np.isfinite(weight_sum) or weight_sum <= 0.0:
        return empty

    # ONLY THIS QUANTITY IS AVERAGED.
    weighted_angle = float(np.sum(weights * weight_angle) / weight_sum)

    # Do NOT average phase speed. Evaluate the phase-speed curve at the
    # weighted angle instead. np.interp is appropriate because the weighted
    # angle must lie inside the retained angle range.
    speed_at_weighted_angle = float(
        np.interp(weighted_angle, curve_angle, curve_speed)
    )

    # Likewise, report the ridge-fit R² at the weighted angle by interpolation,
    # not by averaging R² over all contributing angles.
    finite_r2 = np.isfinite(curve_r2)
    if np.count_nonzero(finite_r2) >= 2:
        r_squared_at_weighted_angle = float(
            np.interp(
                weighted_angle,
                curve_angle[finite_r2],
                curve_r2[finite_r2],
            )
        )
    elif np.count_nonzero(finite_r2) == 1:
        r_squared_at_weighted_angle = float(curve_r2[finite_r2][0])
    else:
        r_squared_at_weighted_angle = np.nan

    return {
        "weighted_angle_deg": weighted_angle,
        "speed_at_weighted_angle_m_per_s": speed_at_weighted_angle,
        "r_squared_at_weighted_angle": r_squared_at_weighted_angle,
        "minimum_speed_magnitude_m_per_s": minimum_speed_magnitude,
        "number_of_weighted_angles": int(weight_angle.size),
        "weight_sum": weight_sum,
    }


def build_window_summary_rows(window_results):
    """Create one weighted comparison row for each successfully run window."""
    rows = []

    for result in window_results:
        combined_results = result["combined_results"]

        processed = get_weighted_speed_summary(
            combined_results,
            speed_column=4,
            r2_column=5,
        )

        corrected = get_weighted_speed_summary(
            combined_results,
            speed_column=10,
            r2_column=11,
        )

        rows.append(
            {
                "duration_hours": result["duration_hours"],
                "window": result["time_range_label"],
                "processed_weighted_angle_deg": processed[
                    "weighted_angle_deg"
                ],
                "processed_speed_at_weighted_angle_m_per_s": processed[
                    "speed_at_weighted_angle_m_per_s"
                ],
                "processed_r_squared_at_weighted_angle": processed[
                    "r_squared_at_weighted_angle"
                ],
                "processed_number_of_weighted_angles": processed[
                    "number_of_weighted_angles"
                ],
                "minus_2hr_bias_weighted_angle_deg": corrected[
                    "weighted_angle_deg"
                ],
                "minus_2hr_bias_speed_at_weighted_angle_m_per_s": corrected[
                    "speed_at_weighted_angle_m_per_s"
                ],
                "minus_2hr_bias_r_squared_at_weighted_angle": corrected[
                    "r_squared_at_weighted_angle"
                ],
                "minus_2hr_bias_number_of_weighted_angles": corrected[
                    "number_of_weighted_angles"
                ],
                "successful_angles": result["successful_angles"],
                "failed_angles": result["failed_angles"],
            }
        )

    return rows


def save_window_summary_csv(rows, output_file):
    fieldnames = [
        "duration_hours",
        "window",
        "processed_weighted_angle_deg",
        "processed_speed_at_weighted_angle_m_per_s",
        "processed_r_squared_at_weighted_angle",
        "processed_number_of_weighted_angles",
        "minus_2hr_bias_weighted_angle_deg",
        "minus_2hr_bias_speed_at_weighted_angle_m_per_s",
        "minus_2hr_bias_r_squared_at_weighted_angle",
        "minus_2hr_bias_number_of_weighted_angles",
        "successful_angles",
        "failed_angles",
    ]

    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def format_table_number(value, format_string):
    if value is None:
        return ""
    try:
        value = float(value)
    except (TypeError, ValueError):
        return str(value)

    if not np.isfinite(value):
        return ""

    return format(value, format_string)


def save_window_summary_table_plot(rows, output_file):
    """
    Save one merged large-font table containing both processed datasets.

    Columns:
        Duration | Window |
        Processed weighted angle | speed at that angle | R² at that angle |
        Minus-2hr-bias weighted angle | speed at that angle | R² at that angle
    """
    if not rows:
        return

    # Sort by duration, then by the requested window order.
    window_order = {
        (
            f"{format_clock_hour(start_hour)}–"
            f"{format_clock_hour(end_hour)}"
        ): index
        for index, (_, start_hour, end_hour) in enumerate(ANALYSIS_WINDOWS)
    }
    rows = sorted(
        rows,
        key=lambda row: (
            float(row["duration_hours"]),
            window_order.get(row["window"], 999),
        ),
    )

    column_labels = [
        "Duration",
        "Local-time\nwindow",
        "Processed\nangle (°)",
        "Processed\nspeed (m/s)",
        "Processed\nR² at angle",
        "Minus 2-hr bias\nangle (°)",
        "Minus 2-hr bias\nspeed (m/s)",
        "Minus 2-hr bias\nR² at angle",
    ]

    table_data = []
    for row in rows:
        table_data.append(
            [
                f"{format_table_number(row['duration_hours'], '.0f')} hr",
                row["window"],
                format_table_number(
                    row["processed_weighted_angle_deg"], ".1f"
                ),
                format_table_number(
                    row["processed_speed_at_weighted_angle_m_per_s"], ".1f"
                ),
                format_table_number(
                    row["processed_r_squared_at_weighted_angle"], ".3f"
                ),
                format_table_number(
                    row["minus_2hr_bias_weighted_angle_deg"], ".1f"
                ),
                format_table_number(
                    row["minus_2hr_bias_speed_at_weighted_angle_m_per_s"], ".1f"
                ),
                format_table_number(
                    row["minus_2hr_bias_r_squared_at_weighted_angle"], ".3f"
                ),
            ]
        )

    # Wide figure so the merged table can use a genuinely larger font.
    fig_height = max(10.0, 0.72 * len(rows) + 3.4)
    fig, ax = plt.subplots(figsize=(19.0, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=table_data,
        colLabels=column_labels,
        cellLoc="center",
        colLoc="center",
        loc="center",
        colWidths=[0.08, 0.14, 0.11, 0.13, 0.09, 0.13, 0.14, 0.09],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.0, 1.85)

    # Header formatting.
    for column in range(len(column_labels)):
        cell = table[(0, column)]
        cell.set_text_props(weight="bold", fontsize=12)
        cell.set_facecolor("0.86")
        cell.set_linewidth(1.2)

    # Lightly separate duration groups. If either reported R² exceeds the
    # requested threshold, highlight the entire merged row in light green.
    previous_duration = None
    group_number = -1
    for row_index, row in enumerate(rows, start=1):
        duration = int(round(float(row["duration_hours"])))

        if duration != previous_duration:
            group_number += 1

        processed_r2 = row["processed_r_squared_at_weighted_angle"]
        corrected_r2 = row["minus_2hr_bias_r_squared_at_weighted_angle"]

        row_has_high_r2 = (
            (np.isfinite(processed_r2) and
             processed_r2 > SUMMARY_R2_HIGHLIGHT_THRESHOLD)
            or
            (np.isfinite(corrected_r2) and
             corrected_r2 > SUMMARY_R2_HIGHLIGHT_THRESHOLD)
        )

        if row_has_high_r2:
            shade = SUMMARY_R2_HIGHLIGHT_COLOR
        else:
            shade = "0.98" if group_number % 2 == 0 else "0.94"

        for column in range(len(column_labels)):
            table[(row_index, column)].set_facecolor(shade)
            table[(row_index, column)].set_linewidth(0.7)

        if previous_duration is not None and duration != previous_duration:
            for column in range(len(column_labels)):
                table[(row_index, column)].set_linewidth(1.8)

        previous_duration = duration

    weighting_text = (
        f"Weights ∝ 1/|speed|^{WEIGHTED_SPEED_POWER:g}"
    )
    if WEIGHT_BY_R_SQUARED:
        weighting_text += f" × R²^{WEIGHTED_R2_POWER:g}"
    weighting_text += (
        f"; average angle only; use angles with |speed| ≤ "
        f"{WEIGHTED_SPEED_MAX_FACTOR:g} × minimum |speed|; "
        f"green row: either R² > {SUMMARY_R2_HIGHLIGHT_THRESHOLD:.2f}"
    )

    ax.set_title(
        "BLO Weighted-Angle Phase-Speed Summary\n"
        "3-, 4-, and 5-hour windows from 23:00–05:00 local time; "
        "30-minute start increments\n"
        + weighting_text,
        fontsize=16,
        weight="bold",
        pad=22,
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def main():
    if not KEOGRAM_DIR.exists():
        raise FileNotFoundError(f"Keogram directory does not exist:\n{KEOGRAM_DIR}")

    print("=" * 78)
    print("RUNNING BLO KEOGRAM ANALYSES FROM 23:00 TO 05:00")
    print("=" * 78)
    print("Durations: 3, 4, and 5 hours; starts every 30 minutes")
    print(f"Number of windows: {len(ANALYSIS_WINDOWS)}")
    print("Angle range: 30°–150° in 5° steps")
    print(
        "Weighted-angle summary: lower |speed| gets higher angle weight; "
        f"speed power={WEIGHTED_SPEED_POWER:g}, "
        f"trough factor={WEIGHTED_SPEED_MAX_FACTOR:g}, "
        f"R² weighting={WEIGHT_BY_R_SQUARED}"
    )
    print(f"Analysis root: {ANALYSIS_ROOT}")

    print("\nRequested windows:")
    for window_name, start_hour, end_hour in ANALYSIS_WINDOWS:
        duration_hours = (float(end_hour) - float(start_hour)) % 24.0
        print(
            f"  {duration_hours:.0f} hr: "
            f"{format_clock_hour(start_hour)}–{format_clock_hour(end_hour)}"
        )

    failed_windows = []
    window_results = []

    for window_name, start_hour, end_hour in ANALYSIS_WINDOWS:
        duration_hours = (float(end_hour) - float(start_hour)) % 24.0

        print("\n" + "#" * 78)
        print(
            f"{duration_hours:.0f}-HOUR WINDOW "
            f"{format_clock_hour(start_hour)}–"
            f"{format_clock_hour(end_hour)}"
        )
        print("#" * 78)

        try:
            result = analyze_window(window_name, start_hour, end_hour)
            window_results.append(result)
        except Exception as exc:
            failed_windows.append((window_name, str(exc)))
            print(f"WINDOW FAILED: {exc}")

    # --------------------------------------------------
    # Root-level weighted comparison table
    # --------------------------------------------------
    summary_rows = build_window_summary_rows(window_results)

    if summary_rows:
        summary_csv = ANALYSIS_ROOT / "window_summary_table.csv"
        summary_png = ANALYSIS_ROOT / "window_summary_table.png"

        save_window_summary_csv(summary_rows, summary_csv)
        save_window_summary_table_plot(summary_rows, summary_png)

        print("\nCross-window summary table:")
        print(f"  {summary_csv}")
        print(f"  {summary_png}")

    print("\n" + "=" * 78)
    print("ALL REQUESTED WINDOWS FINISHED")
    print("=" * 78)
    print(f"Analysis root: {ANALYSIS_ROOT.resolve()}")
    print(f"Successful windows: {len(window_results)}")
    print(f"Failed windows:     {len(failed_windows)}")

    if failed_windows:
        print("\nFailed windows:")
        for window_name, reason in failed_windows:
            print(f"  {window_name}: {reason}")
    else:
        print(f"All {len(ANALYSIS_WINDOWS)} windows completed successfully.")


if __name__ == "__main__":
    main()


RUNNING BLO KEOGRAM ANALYSES FROM 23:00 TO 05:00
Durations: 3, 4, and 5 hours; starts every 30 minutes
Number of windows: 15
Angle range: 30°–150° in 5° steps
Weighted-angle summary: lower |speed| gets higher angle weight; speed power=2, trough factor=1.5, R² weighting=True
Analysis root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/keogram_outputs/analysis_3_4_5hour_windows

Requested windows:
  3 hr: 23:00–02:00
  3 hr: 23:30–02:30
  3 hr: 00:00–03:00
  3 hr: 00:30–03:30
  3 hr: 01:00–04:00
  3 hr: 01:30–04:30
  3 hr: 02:00–05:00
  4 hr: 23:00–03:00
  4 hr: 23:30–03:30
  4 hr: 00:00–04:00
  4 hr: 00:30–04:30
  4 hr: 01:00–05:00
  5 hr: 23:00–04:00
  5 hr: 23:30–04:30
  5 hr: 00:00–05:00

##############################################################################
3-HOUR WINDOW 23:00–02:00
##############################################################################
BLO KEOGRAM CORRELATION ANALYSIS
Keogram directory: /Users/anhphan/Do

## 2D correlation (failed)

In [16]:
# -*- coding: utf-8 -*-
"""
Estimate BLO horizontal phase speed directly from the saved preprocessed cube
using 2-D frame-to-frame spatial correlation.

Method
------
1. Read:
       preprocessed_perturbation_cube.npy
       preprocessed_perturbation_cube_metadata.npz

2. Repeat the analysis for SAVED LOCAL FRAME-CENTER times 21:10, 21:20, 21:30, 21:40, and 21:50.

3. For each target, use the nearest frame as the spatial reference and compare it with four frames
   before and four frames after it (9 frames total including the reference).

4. For each neighboring frame, calculate a NaN-aware normalized 2-D Pearson
   cross-correlation map as a function of spatial X/Y shift.

5. Find the correlation maximum and refine the peak to sub-pixel precision
   with a quadratic interpolation in X and Y.

6. Fit measured displacement versus time:
       dx(t) = vx * dt + bx
       dy(t) = vy * dt + by

   The phase-speed vector is:
       (vx, vy)

   and the horizontal phase-speed magnitude is:
       sqrt(vx^2 + vy^2)

Coordinate convention
---------------------
The script preserves the same convention as the cube/keogram workflow:

    0 deg   = +X / image right
    +90 deg = +Y / image down

Therefore the reported propagation direction is in the SAVED CUBE IMAGE
COORDINATE SYSTEM, not automatically geographic east/north.

Notes
-----
- The cube is memory-mapped, so the complete cube is not copied into RAM.
- Only the selected 9 frames and the chosen correlation region are read.
- This is a translation-based phase-speed estimate. A nearly perfect infinite
  plane wave can produce an elongated correlation ridge along its wave crest;
  finite wave packets / spatial structure make the 2-D displacement better
  constrained.
"""

import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.signal import fftconvolve


# ==================================================
# Input / output settings
# ==================================================
DOWNLOADS_DIR = Path.home() / "Downloads"

CUBE_DIR = (
    DOWNLOADS_DIR
    / "pkl"
    / "BLO_plane_fit_batch_outputs"
    / "2025"
    / "processed_data_full_03_25_2025"
)

CUBE_FILE = CUBE_DIR / "preprocessed_perturbation_cube.npy"
METADATA_FILE = CUBE_DIR / "preprocessed_perturbation_cube_metadata.npz"

ANALYSIS_ROOT = CUBE_DIR / "analysis_2d_correlation_2110_to_2150"

# Run one independent ±4-frame 2-D correlation analysis at each target time.
TARGET_LOCAL_TIMES = [
    (21, 10),
    (21, 20),
    (21, 30),
    (21, 40),
    (21, 50),
]


# ==================================================
# Frame-selection settings
# ==================================================

# Four frames before + reference + four frames after.
NEIGHBOR_FRAMES_EACH_SIDE = 4

# Safety check for accidentally selecting the wrong clock time.
MAX_ALLOWED_TARGET_TIME_ERROR_MIN = 5.0


# ==================================================
# Spatial-correlation settings
# ==================================================
# Analyze the central part of the saved cube.
# Set to None to use the complete X/Y extent.
CORRELATION_HALF_WIDTH_KM = 450.0

# Maximum allowed 2-D displacement magnitude searched in the correlation map.
# Four 2-minute frames on either side gives |dt| <= about 8 min, so 450 km
# permits speeds up to about 940 m/s at the outermost frames.
MAX_SHIFT_MAGNITUDE_KM = 450.0

# Require this fraction of the maximum possible valid overlap at each shift.
MIN_OVERLAP_FRACTION = 0.30

# Absolute lower limit on valid paired pixels.
MIN_OVERLAP_PIXELS = 5000

# Optional additional smoothing before spatial correlation.
# The saved cube is already preprocessed, so this is disabled by default.
APPLY_GAUSSIAN_SMOOTHING = False
GAUSSIAN_SIGMA_PIXELS = 2.0

# Peak correlation threshold for including a displacement in the speed fit.
MIN_PEAK_CORRELATION_FOR_FIT = 0.50

# Robust vector-residual rejection from the displacement-vs-time fit.
REMOVE_FIT_OUTLIERS = True
FIT_OUTLIER_SIGMA = 3.0
FIT_OUTLIER_MAX_ITERATIONS = 8
FIT_OUTLIER_MIN_NONREFERENCE_POINTS = 4

# Include the exact reference point (dt=0, dx=0, dy=0) in the fit.
INCLUDE_REFERENCE_POINT_IN_FIT = True

# Save the cropped correlation-search map for every neighboring frame.
SAVE_CORRELATION_MAPS = True


# ==================================================
# Plot settings
# ==================================================
DPI = 200
CMAP = "RdBu_r"
FRAME_COLOR_PERCENTILE = 99.0

CORRELATION_PLOT_VMIN = 0.50
CORRELATION_PLOT_VMAX = 1.00


# ==================================================
# Metadata / time helpers
# ==================================================
def metadata_scalar(metadata, key, default=None):
    if key not in metadata.files:
        return default

    value = metadata[key]

    if np.ndim(value) == 0:
        try:
            return value.item()
        except ValueError:
            return value

    if np.size(value) == 1:
        value = np.ravel(value)[0]
        try:
            return value.item()
        except (AttributeError, ValueError):
            return value

    return value


def parse_iso_datetime(text):
    text = str(text).strip()
    if not text:
        raise ValueError("Empty datetime string.")
    return datetime.fromisoformat(text)


def build_local_datetimes(metadata, times_min):
    """
    Prefer explicitly saved frame-center local datetimes.
    Fall back to start_local_datetime + time_minutes.
    """
    if "local_datetime" in metadata.files:
        values = np.asarray(metadata["local_datetime"], dtype=str)
        if len(values) == len(times_min):
            return np.array(
                [parse_iso_datetime(value) for value in values],
                dtype=object,
            )

    start_text = str(metadata_scalar(metadata, "start_local_datetime", ""))
    if not start_text:
        raise ValueError(
            "Metadata contains neither a usable local_datetime array nor "
            "start_local_datetime."
        )

    start_datetime = parse_iso_datetime(start_text)

    return np.array(
        [
            start_datetime + timedelta(minutes=float(value))
            for value in times_min
        ],
        dtype=object,
    )


def clock_minutes(dt):
    return (
        dt.hour * 60.0
        + dt.minute
        + dt.second / 60.0
        + dt.microsecond / 60.0e6
    )


def signed_clock_difference_minutes(clock_value, target_value):
    """
    Shortest signed clock-time difference in minutes, in [-720, +720).
    """
    return ((clock_value - target_value + 720.0) % 1440.0) - 720.0


def find_reference_frame(local_datetimes, target_hour, target_minute):
    target_clock = float(target_hour) * 60.0 + float(target_minute)

    differences = np.array(
        [
            signed_clock_difference_minutes(clock_minutes(dt), target_clock)
            for dt in local_datetimes
        ],
        dtype=np.float64,
    )

    reference_index = int(np.argmin(np.abs(differences)))
    time_error_min = float(differences[reference_index])

    return reference_index, time_error_min


# ==================================================
# Frame preparation
# ==================================================
def nan_gaussian_filter_2d(data, sigma):
    data = np.asarray(data, dtype=np.float64)

    valid = np.isfinite(data)
    filled = np.where(valid, data, 0.0)
    weights = valid.astype(np.float64)

    smoothed_values = gaussian_filter(
        filled,
        sigma=float(sigma),
        mode="nearest",
    )
    smoothed_weights = gaussian_filter(
        weights,
        sigma=float(sigma),
        mode="nearest",
    )

    result = np.full(data.shape, np.nan, dtype=np.float64)
    good = smoothed_weights > 1.0e-12
    result[good] = smoothed_values[good] / smoothed_weights[good]

    return result


def select_correlation_slices(x_km, y_km, half_width_km):
    """
    Return contiguous Y/X slices spanning the requested centered region.
    """
    x_km = np.asarray(x_km, dtype=np.float64)
    y_km = np.asarray(y_km, dtype=np.float64)

    if half_width_km is None:
        return slice(0, len(y_km)), slice(0, len(x_km))

    x_good = np.flatnonzero(np.abs(x_km) <= float(half_width_km))
    y_good = np.flatnonzero(np.abs(y_km) <= float(half_width_km))

    if x_good.size < 3 or y_good.size < 3:
        raise ValueError(
            "Requested CORRELATION_HALF_WIDTH_KM leaves too few pixels."
        )

    x_slice = slice(int(x_good[0]), int(x_good[-1]) + 1)
    y_slice = slice(int(y_good[0]), int(y_good[-1]) + 1)

    return y_slice, x_slice


def prepare_frame(frame):
    frame = np.asarray(frame, dtype=np.float64)

    if APPLY_GAUSSIAN_SMOOTHING:
        return nan_gaussian_filter_2d(
            frame,
            sigma=GAUSSIAN_SIGMA_PIXELS,
        )

    return frame.copy()


# ==================================================
# NaN-aware normalized 2-D cross-correlation
# ==================================================
def normalized_cross_correlation_2d(
    reference,
    comparison,
    min_overlap_fraction=0.30,
    min_overlap_pixels=5000,
):
    """
    Calculate a full NaN-aware Pearson correlation map.

    Positive output shift means the comparison-frame structure is displaced
    toward increasing array index relative to the reference frame.

    Returns
    -------
    correlation : ndarray, shape (2*ny-1, 2*nx-1)
    overlap_count : ndarray
    shift_y_pixels : ndarray, length 2*ny-1
    shift_x_pixels : ndarray, length 2*nx-1
    """
    reference = np.asarray(reference, dtype=np.float64)
    comparison = np.asarray(comparison, dtype=np.float64)

    if reference.shape != comparison.shape:
        raise ValueError("Reference and comparison frames must match in shape.")

    if reference.ndim != 2:
        raise ValueError("Frames must be two-dimensional.")

    ny, nx = reference.shape

    valid_a = np.isfinite(reference)
    valid_b = np.isfinite(comparison)

    a = np.where(valid_a, reference, 0.0)
    b = np.where(valid_b, comparison, 0.0)

    ma = valid_a.astype(np.float64)
    mb = valid_b.astype(np.float64)

    # fftconvolve(B, reversed(A)) has positive peak shifts when B is shifted
    # toward increasing indices relative to A.
    a_rev = a[::-1, ::-1]
    ma_rev = ma[::-1, ::-1]

    sum_ab = fftconvolve(b, a_rev, mode="full")
    count = fftconvolve(mb, ma_rev, mode="full")

    sum_a = fftconvolve(mb, a_rev, mode="full")
    sum_b = fftconvolve(b, ma_rev, mode="full")

    sum_a2 = fftconvolve(mb, (a * a)[::-1, ::-1], mode="full")
    sum_b2 = fftconvolve(b * b, ma_rev, mode="full")

    correlation = np.full(sum_ab.shape, np.nan, dtype=np.float64)

    max_possible_overlap = float(
        min(np.count_nonzero(valid_a), np.count_nonzero(valid_b))
    )

    required_overlap = max(
        float(min_overlap_pixels),
        float(min_overlap_fraction) * max_possible_overlap,
    )

    good_count = count >= required_overlap

    safe_count = np.where(count > 0.0, count, np.nan)

    covariance = sum_ab - (sum_a * sum_b) / safe_count
    variance_a = sum_a2 - (sum_a * sum_a) / safe_count
    variance_b = sum_b2 - (sum_b * sum_b) / safe_count

    # Tiny negative values can occur from FFT floating-point roundoff.
    variance_a = np.maximum(variance_a, 0.0)
    variance_b = np.maximum(variance_b, 0.0)

    denominator = np.sqrt(variance_a * variance_b)

    good = (
        good_count
        & np.isfinite(covariance)
        & np.isfinite(denominator)
        & (denominator > 0.0)
    )

    correlation[good] = covariance[good] / denominator[good]
    correlation[good] = np.clip(correlation[good], -1.0, 1.0)

    shift_y_pixels = np.arange(
        -(ny - 1),
        ny,
        dtype=np.float64,
    )
    shift_x_pixels = np.arange(
        -(nx - 1),
        nx,
        dtype=np.float64,
    )

    return correlation, count, shift_y_pixels, shift_x_pixels


def make_shift_search_mask(
    shift_y_pixels,
    shift_x_pixels,
    y_step_km,
    x_step_km,
    maximum_shift_magnitude_km,
):
    shift_y_km = shift_y_pixels * float(y_step_km)
    shift_x_km = shift_x_pixels * float(x_step_km)

    grid_x_km, grid_y_km = np.meshgrid(
        shift_x_km,
        shift_y_km,
    )

    magnitude_km = np.hypot(grid_x_km, grid_y_km)

    return (
        magnitude_km <= float(maximum_shift_magnitude_km),
        shift_y_km,
        shift_x_km,
    )


def parabolic_peak_offset(value_minus, value_center, value_plus):
    """
    1-D quadratic peak correction in pixel units.
    """
    values = [value_minus, value_center, value_plus]

    if not all(np.isfinite(value) for value in values):
        return 0.0

    denominator = value_minus - 2.0 * value_center + value_plus

    if not np.isfinite(denominator) or np.isclose(denominator, 0.0):
        return 0.0

    offset = 0.5 * (value_minus - value_plus) / denominator

    if not np.isfinite(offset):
        return 0.0

    # The local quadratic refinement is only meaningful close to the center
    # sample. Clipping also protects against pathological low-curvature cases.
    return float(np.clip(offset, -1.0, 1.0))


def find_subpixel_correlation_peak(
    correlation,
    search_mask,
    shift_y_pixels,
    shift_x_pixels,
):
    """
    Find maximum correlation inside the search region and refine it to
    sub-pixel X/Y position using local parabolic interpolation.
    """
    allowed = search_mask & np.isfinite(correlation)

    if not np.any(allowed):
        raise ValueError("No finite correlation values inside shift search.")

    candidate = np.where(allowed, correlation, -np.inf)

    peak_flat = int(np.argmax(candidate))
    peak_iy, peak_ix = np.unravel_index(peak_flat, candidate.shape)

    peak_correlation = float(correlation[peak_iy, peak_ix])

    delta_x = 0.0
    delta_y = 0.0

    if 0 < peak_ix < correlation.shape[1] - 1:
        delta_x = parabolic_peak_offset(
            correlation[peak_iy, peak_ix - 1],
            correlation[peak_iy, peak_ix],
            correlation[peak_iy, peak_ix + 1],
        )

    if 0 < peak_iy < correlation.shape[0] - 1:
        delta_y = parabolic_peak_offset(
            correlation[peak_iy - 1, peak_ix],
            correlation[peak_iy, peak_ix],
            correlation[peak_iy + 1, peak_ix],
        )

    peak_shift_x_pixels = float(shift_x_pixels[peak_ix] + delta_x)
    peak_shift_y_pixels = float(shift_y_pixels[peak_iy] + delta_y)

    return {
        "peak_iy": int(peak_iy),
        "peak_ix": int(peak_ix),
        "peak_correlation": peak_correlation,
        "shift_x_pixels": peak_shift_x_pixels,
        "shift_y_pixels": peak_shift_y_pixels,
        "subpixel_dx": float(delta_x),
        "subpixel_dy": float(delta_y),
    }


def crop_correlation_for_search(
    correlation,
    shift_y_km,
    shift_x_km,
    maximum_shift_magnitude_km,
):
    """
    Crop the full correlation map to a square containing the circular search.
    """
    x_indices = np.flatnonzero(
        np.abs(shift_x_km) <= float(maximum_shift_magnitude_km)
    )
    y_indices = np.flatnonzero(
        np.abs(shift_y_km) <= float(maximum_shift_magnitude_km)
    )

    if x_indices.size == 0 or y_indices.size == 0:
        raise ValueError("No correlation shifts lie inside requested range.")

    x_slice = slice(int(x_indices[0]), int(x_indices[-1]) + 1)
    y_slice = slice(int(y_indices[0]), int(y_indices[-1]) + 1)

    return (
        np.asarray(correlation[y_slice, x_slice], dtype=np.float32),
        np.asarray(shift_y_km[y_slice], dtype=np.float64),
        np.asarray(shift_x_km[x_slice], dtype=np.float64),
    )


# ==================================================
# Displacement / velocity fit
# ==================================================
def linear_fit_with_r_squared(time_s, displacement_km):
    time_s = np.asarray(time_s, dtype=np.float64)
    displacement_km = np.asarray(displacement_km, dtype=np.float64)

    if len(time_s) < 2:
        return np.nan, np.nan, np.nan

    slope_km_per_s, intercept_km = np.polyfit(
        time_s,
        displacement_km,
        1,
    )

    predicted = slope_km_per_s * time_s + intercept_km

    ss_res = np.sum((displacement_km - predicted) ** 2)
    ss_tot = np.sum(
        (displacement_km - np.mean(displacement_km)) ** 2
    )

    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return (
        float(slope_km_per_s),
        float(intercept_km),
        float(r_squared),
    )


def combined_vector_r_squared(
    time_s,
    x_km,
    y_km,
    vx_km_per_s,
    bx_km,
    vy_km_per_s,
    by_km,
):
    time_s = np.asarray(time_s, dtype=np.float64)
    x_km = np.asarray(x_km, dtype=np.float64)
    y_km = np.asarray(y_km, dtype=np.float64)

    x_pred = vx_km_per_s * time_s + bx_km
    y_pred = vy_km_per_s * time_s + by_km

    ss_res = np.sum((x_km - x_pred) ** 2 + (y_km - y_pred) ** 2)

    x_mean = np.mean(x_km)
    y_mean = np.mean(y_km)

    ss_tot = np.sum(
        (x_km - x_mean) ** 2 + (y_km - y_mean) ** 2
    )

    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan


def robust_velocity_fit(
    delta_time_s,
    shift_x_km,
    shift_y_km,
    peak_correlation,
    reference_mask,
):
    delta_time_s = np.asarray(delta_time_s, dtype=np.float64)
    shift_x_km = np.asarray(shift_x_km, dtype=np.float64)
    shift_y_km = np.asarray(shift_y_km, dtype=np.float64)
    peak_correlation = np.asarray(peak_correlation, dtype=np.float64)
    reference_mask = np.asarray(reference_mask, dtype=bool)

    fit_mask = (
        np.isfinite(delta_time_s)
        & np.isfinite(shift_x_km)
        & np.isfinite(shift_y_km)
        & np.isfinite(peak_correlation)
        & (peak_correlation >= MIN_PEAK_CORRELATION_FOR_FIT)
    )

    if INCLUDE_REFERENCE_POINT_IN_FIT:
        fit_mask |= reference_mask
    else:
        fit_mask &= ~reference_mask

    minimum_points = 3

    if np.count_nonzero(fit_mask) < minimum_points:
        raise ValueError(
            "Too few displacement points pass the correlation threshold "
            f"{MIN_PEAK_CORRELATION_FOR_FIT:.2f}."
        )

    if REMOVE_FIT_OUTLIERS:
        for _ in range(int(FIT_OUTLIER_MAX_ITERATIONS)):
            indices = np.flatnonzero(fit_mask)

            nonref_count = np.count_nonzero(
                fit_mask & ~reference_mask
            )

            if (
                len(indices) < minimum_points
                or nonref_count < FIT_OUTLIER_MIN_NONREFERENCE_POINTS
            ):
                break

            t = delta_time_s[indices]
            x = shift_x_km[indices]
            y = shift_y_km[indices]

            vx, bx, _ = linear_fit_with_r_squared(t, x)
            vy, by, _ = linear_fit_with_r_squared(t, y)

            x_pred = vx * t + bx
            y_pred = vy * t + by

            residual = np.hypot(
                x - x_pred,
                y - y_pred,
            )

            residual_median = float(np.median(residual))
            mad = float(
                np.median(np.abs(residual - residual_median))
            )

            if not np.isfinite(mad) or np.isclose(mad, 0.0):
                break

            robust_sigma = 1.4826 * mad
            keep_local = (
                np.abs(residual - residual_median)
                <= float(FIT_OUTLIER_SIGMA) * robust_sigma
            )

            # Never reject the exact reference point solely because it has
            # zero displacement by construction.
            local_reference = reference_mask[indices]
            keep_local[local_reference] = True

            if np.all(keep_local):
                break

            proposed = fit_mask.copy()
            proposed[indices[~keep_local]] = False

            proposed_nonref = np.count_nonzero(
                proposed & ~reference_mask
            )

            if proposed_nonref < FIT_OUTLIER_MIN_NONREFERENCE_POINTS:
                break

            fit_mask = proposed

    indices = np.flatnonzero(fit_mask)

    t = delta_time_s[indices]
    x = shift_x_km[indices]
    y = shift_y_km[indices]

    vx_km_per_s, bx_km, r2_x = linear_fit_with_r_squared(t, x)
    vy_km_per_s, by_km, r2_y = linear_fit_with_r_squared(t, y)

    vector_r2 = combined_vector_r_squared(
        t,
        x,
        y,
        vx_km_per_s,
        bx_km,
        vy_km_per_s,
        by_km,
    )

    speed_km_per_s = float(
        np.hypot(vx_km_per_s, vy_km_per_s)
    )
    speed_m_per_s = speed_km_per_s * 1000.0

    direction_deg = float(
        np.degrees(np.arctan2(vy_km_per_s, vx_km_per_s))
    )

    # Wrap to [-180, +180).
    direction_deg = (
        (direction_deg + 180.0) % 360.0
    ) - 180.0

    direction_0_360_deg = direction_deg % 360.0

    return {
        "fit_mask": fit_mask,
        "vx_km_per_s": vx_km_per_s,
        "vy_km_per_s": vy_km_per_s,
        "vx_m_per_s": vx_km_per_s * 1000.0,
        "vy_m_per_s": vy_km_per_s * 1000.0,
        "x_intercept_km": bx_km,
        "y_intercept_km": by_km,
        "r_squared_x": r2_x,
        "r_squared_y": r2_y,
        "r_squared_vector": vector_r2,
        "speed_km_per_s": speed_km_per_s,
        "speed_m_per_s": speed_m_per_s,
        "direction_deg": direction_deg,
        "direction_0_360_deg": direction_0_360_deg,
        "number_of_fit_points": int(np.count_nonzero(fit_mask)),
        "number_rejected": int(
            np.count_nonzero(
                np.isfinite(shift_x_km)
                & ~reference_mask
                & ~fit_mask
            )
        ),
    }


# ==================================================
# Plot helpers
# ==================================================
def symmetric_frame_limit(frames, percentile=99.0):
    finite_parts = []

    for frame in frames:
        values = np.asarray(frame, dtype=np.float64)
        finite = values[np.isfinite(values)]

        if finite.size > 0:
            finite_parts.append(np.abs(finite))

    if not finite_parts:
        return 1.0

    combined = np.concatenate(finite_parts)

    vmax = float(np.nanpercentile(combined, percentile))

    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    return vmax


def save_selected_frames_plot(
    frames,
    selected_indices,
    local_datetimes,
    reference_index,
    x_roi_km,
    y_roi_km,
    output_file,
):
    vmax = symmetric_frame_limit(
        frames,
        percentile=FRAME_COLOR_PERCENTILE,
    )

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(13, 12),
        sharex=True,
        sharey=True,
    )

    image = None

    for ax, frame, frame_index in zip(
        axes.ravel(),
        frames,
        selected_indices,
    ):
        image = ax.imshow(
            frame,
            origin="upper",
            cmap=CMAP,
            vmin=-vmax,
            vmax=vmax,
            extent=[
                x_roi_km.min(),
                x_roi_km.max(),
                y_roi_km.max(),
                y_roi_km.min(),
            ],
        )

        marker = " [REFERENCE]" if frame_index == reference_index else ""

        ax.set_title(
            f"Frame {frame_index} | "
            f"{local_datetimes[frame_index].strftime('%H:%M:%S')}"
            f"{marker}",
            fontsize=10,
        )

        ax.set_aspect("equal")

    for ax in axes[-1, :]:
        ax.set_xlabel("X distance (km)")

    for ax in axes[:, 0]:
        ax.set_ylabel("Y image-coordinate distance (km)")

    if image is not None:
        cbar = fig.colorbar(
            image,
            ax=axes.ravel().tolist(),
            fraction=0.025,
            pad=0.02,
        )
        cbar.set_label("Preprocessed perturbation amplitude")

    fig.suptitle(
        "Frames used for 2-D correlation around each requested local time",
        y=0.995,
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.90,
        bottom=0.06,
        top=0.94,
        wspace=0.18,
        hspace=0.22,
    )

    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_correlation_maps_plot(
    correlation_products,
    output_file,
):
    """
    Plot the eight neighbor-frame correlation maps.
    """
    fig, axes = plt.subplots(
        2,
        4,
        figsize=(16, 8.5),
        sharex=True,
        sharey=True,
    )

    image = None

    for ax, product in zip(
        axes.ravel(),
        correlation_products,
    ):
        corr = product["correlation_crop"]
        sx = product["shift_x_crop_km"]
        sy = product["shift_y_crop_km"]

        image = ax.imshow(
            corr,
            origin="lower",
            aspect="equal",
            cmap="viridis",
            vmin=CORRELATION_PLOT_VMIN,
            vmax=CORRELATION_PLOT_VMAX,
            extent=[
                sx[0],
                sx[-1],
                sy[0],
                sy[-1],
            ],
        )

        # Draw the circular search limit.
        theta = np.linspace(0.0, 2.0 * np.pi, 400)
        ax.plot(
            MAX_SHIFT_MAGNITUDE_KM * np.cos(theta),
            MAX_SHIFT_MAGNITUDE_KM * np.sin(theta),
            linestyle="--",
            linewidth=0.8,
        )

        ax.scatter(
            product["shift_x_km"],
            product["shift_y_km"],
            marker="x",
            s=55,
            linewidth=1.5,
        )

        ax.axhline(0.0, linewidth=0.6, alpha=0.5)
        ax.axvline(0.0, linewidth=0.6, alpha=0.5)

        ax.set_title(
            f"Δt={product['delta_time_min']:+.1f} min\n"
            f"r={product['peak_correlation']:.3f}, "
            f"dx={product['shift_x_km']:+.1f}, "
            f"dy={product['shift_y_km']:+.1f} km",
            fontsize=9,
        )

        ax.set_aspect("equal")

    for ax in axes[-1, :]:
        ax.set_xlabel("X shift (km)")

    for ax in axes[:, 0]:
        ax.set_ylabel("Y shift (km)")

    if image is not None:
        cbar = fig.colorbar(
            image,
            ax=axes.ravel().tolist(),
            fraction=0.025,
            pad=0.02,
        )
        cbar.set_label("Normalized spatial correlation")

    fig.suptitle(
        "2-D normalized cross-correlation with the 21:30 reference frame",
        y=0.995,
    )

    fig.subplots_adjust(
        left=0.06,
        right=0.91,
        bottom=0.07,
        top=0.91,
        wspace=0.18,
        hspace=0.28,
    )

    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_displacement_fit_plot(
    delta_time_min,
    shift_x_km,
    shift_y_km,
    peak_correlation,
    reference_mask,
    fit_results,
    output_file,
):
    fit_mask = fit_results["fit_mask"]

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(10.5, 10.5),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.0, 0.75]},
    )

    ax_x, ax_y, ax_corr = axes

    finite_nonref = (
        np.isfinite(shift_x_km)
        & np.isfinite(shift_y_km)
        & ~reference_mask
    )

    rejected = finite_nonref & ~fit_mask
    used = fit_mask

    # X displacement.
    ax_x.scatter(
        delta_time_min[used],
        shift_x_km[used],
        s=45,
        label="Used in fit",
    )

    if np.any(rejected):
        ax_x.scatter(
            delta_time_min[rejected],
            shift_x_km[rejected],
            marker="x",
            s=55,
            label="Rejected / below threshold",
        )

    fit_time_min = np.linspace(
        np.nanmin(delta_time_min),
        np.nanmax(delta_time_min),
        300,
    )
    fit_time_s = fit_time_min * 60.0

    x_fit = (
        fit_results["vx_km_per_s"] * fit_time_s
        + fit_results["x_intercept_km"]
    )

    ax_x.plot(
        fit_time_min,
        x_fit,
        linestyle="--",
        linewidth=1.7,
        label=(
            f"vx = {fit_results['vx_m_per_s']:.1f} m/s, "
            f"R² = {fit_results['r_squared_x']:.3f}"
        ),
    )

    ax_x.axhline(0.0, linewidth=0.7, alpha=0.5)
    ax_x.set_ylabel("X displacement (km)")
    ax_x.grid(True, alpha=0.3)
    ax_x.legend(loc="best")

    # Y displacement.
    ax_y.scatter(
        delta_time_min[used],
        shift_y_km[used],
        s=45,
        label="Used in fit",
    )

    if np.any(rejected):
        ax_y.scatter(
            delta_time_min[rejected],
            shift_y_km[rejected],
            marker="x",
            s=55,
            label="Rejected / below threshold",
        )

    y_fit = (
        fit_results["vy_km_per_s"] * fit_time_s
        + fit_results["y_intercept_km"]
    )

    ax_y.plot(
        fit_time_min,
        y_fit,
        linestyle="--",
        linewidth=1.7,
        label=(
            f"vy = {fit_results['vy_m_per_s']:.1f} m/s, "
            f"R² = {fit_results['r_squared_y']:.3f}"
        ),
    )

    ax_y.axhline(0.0, linewidth=0.7, alpha=0.5)
    ax_y.set_ylabel("Y displacement (km)")
    ax_y.grid(True, alpha=0.3)
    ax_y.legend(loc="best")

    # Correlation strength.
    ax_corr.plot(
        delta_time_min[finite_nonref],
        peak_correlation[finite_nonref],
        marker="o",
        linewidth=1.2,
    )
    ax_corr.axhline(
        MIN_PEAK_CORRELATION_FOR_FIT,
        linestyle="--",
        linewidth=1.0,
        label=f"Fit threshold = {MIN_PEAK_CORRELATION_FOR_FIT:.2f}",
    )
    ax_corr.set_xlabel("Time from reference frame (min)")
    ax_corr.set_ylabel("Peak correlation")
    ax_corr.set_ylim(-0.05, 1.05)
    ax_corr.grid(True, alpha=0.3)
    ax_corr.legend(loc="best")

    fig.suptitle(
        (
            "2-D correlation displacement fit\n"
            f"Phase speed = {fit_results['speed_m_per_s']:.1f} m/s, "
            f"direction = {fit_results['direction_deg']:+.1f}°, "
            f"vector R² = {fit_results['r_squared_vector']:.3f}"
        ),
        y=0.995,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_xy_trajectory_plot(
    delta_time_min,
    shift_x_km,
    shift_y_km,
    reference_mask,
    fit_results,
    output_file,
):
    fit_mask = fit_results["fit_mask"]

    fig, ax = plt.subplots(figsize=(7.5, 7.0))

    used_nonref = fit_mask & ~reference_mask

    scatter = ax.scatter(
        shift_x_km[used_nonref],
        shift_y_km[used_nonref],
        c=delta_time_min[used_nonref],
        s=65,
        zorder=4,
    )

    ax.scatter(
        [0.0],
        [0.0],
        marker="*",
        s=180,
        label="Reference",
        zorder=5,
    )

    rejected = (
        np.isfinite(shift_x_km)
        & np.isfinite(shift_y_km)
        & ~reference_mask
        & ~fit_mask
    )

    if np.any(rejected):
        ax.scatter(
            shift_x_km[rejected],
            shift_y_km[rejected],
            marker="x",
            s=60,
            label="Rejected / below threshold",
            zorder=5,
        )

    fit_time_min = np.linspace(
        np.nanmin(delta_time_min),
        np.nanmax(delta_time_min),
        300,
    )
    fit_time_s = fit_time_min * 60.0

    x_fit = (
        fit_results["vx_km_per_s"] * fit_time_s
        + fit_results["x_intercept_km"]
    )
    y_fit = (
        fit_results["vy_km_per_s"] * fit_time_s
        + fit_results["y_intercept_km"]
    )

    ax.plot(
        x_fit,
        y_fit,
        linestyle="--",
        linewidth=1.8,
        label="Fitted displacement trajectory",
    )

    cbar = fig.colorbar(
        scatter,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Time from reference (min)")

    ax.axhline(0.0, linewidth=0.7, alpha=0.5)
    ax.axvline(0.0, linewidth=0.7, alpha=0.5)

    ax.set_xlabel("X displacement (km)")
    ax.set_ylabel("Y displacement (km)")
    ax.set_title(
        (
            "Measured 2-D displacement trajectory\n"
            f"speed = {fit_results['speed_m_per_s']:.1f} m/s | "
            f"direction = {fit_results['direction_deg']:+.1f}°"
        )
    )

    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    ax.set_aspect("equal", adjustable="datalim")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Save tables
# ==================================================
def save_measurement_csv(
    output_file,
    selected_indices,
    local_datetimes,
    delta_time_min,
    shift_x_pixels,
    shift_y_pixels,
    shift_x_km,
    shift_y_km,
    peak_correlation,
    reference_mask,
    fit_mask,
):
    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "frame_index",
                "local_datetime",
                "delta_time_minutes",
                "shift_x_pixels",
                "shift_y_pixels",
                "shift_x_km",
                "shift_y_km",
                "shift_magnitude_km",
                "peak_correlation",
                "is_reference",
                "used_in_fit",
            ]
        )

        for i in range(len(selected_indices)):
            magnitude = (
                np.hypot(shift_x_km[i], shift_y_km[i])
                if np.isfinite(shift_x_km[i])
                and np.isfinite(shift_y_km[i])
                else np.nan
            )

            writer.writerow(
                [
                    int(selected_indices[i]),
                    local_datetimes[selected_indices[i]].isoformat(),
                    float(delta_time_min[i]),
                    float(shift_x_pixels[i])
                    if np.isfinite(shift_x_pixels[i])
                    else np.nan,
                    float(shift_y_pixels[i])
                    if np.isfinite(shift_y_pixels[i])
                    else np.nan,
                    float(shift_x_km[i])
                    if np.isfinite(shift_x_km[i])
                    else np.nan,
                    float(shift_y_km[i])
                    if np.isfinite(shift_y_km[i])
                    else np.nan,
                    float(magnitude)
                    if np.isfinite(magnitude)
                    else np.nan,
                    float(peak_correlation[i])
                    if np.isfinite(peak_correlation[i])
                    else np.nan,
                    bool(reference_mask[i]),
                    bool(fit_mask[i]),
                ]
            )


def save_phase_speed_summary_csv(
    output_file,
    reference_index,
    reference_datetime,
    fit_results,
):
    fieldnames = [
        "reference_frame_index",
        "reference_local_datetime",
        "vx_m_per_s",
        "vy_m_per_s",
        "phase_speed_m_per_s",
        "direction_deg_minus180_to_180",
        "direction_deg_0_to_360",
        "x_intercept_km",
        "y_intercept_km",
        "r_squared_x",
        "r_squared_y",
        "r_squared_vector",
        "number_of_fit_points",
        "number_rejected",
    ]

    row = {
        "reference_frame_index": int(reference_index),
        "reference_local_datetime": reference_datetime.isoformat(),
        "vx_m_per_s": fit_results["vx_m_per_s"],
        "vy_m_per_s": fit_results["vy_m_per_s"],
        "phase_speed_m_per_s": fit_results["speed_m_per_s"],
        "direction_deg_minus180_to_180": fit_results["direction_deg"],
        "direction_deg_0_to_360": fit_results["direction_0_360_deg"],
        "x_intercept_km": fit_results["x_intercept_km"],
        "y_intercept_km": fit_results["y_intercept_km"],
        "r_squared_x": fit_results["r_squared_x"],
        "r_squared_y": fit_results["r_squared_y"],
        "r_squared_vector": fit_results["r_squared_vector"],
        "number_of_fit_points": fit_results["number_of_fit_points"],
        "number_rejected": fit_results["number_rejected"],
    }

    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(row)


# ==================================================
# Per-target analysis
# ==================================================
def analyze_target_time(target_hour, target_minute):
    target_label = f"{int(target_hour):02d}{int(target_minute):02d}"
    OUT_DIR = ANALYSIS_ROOT / f"analysis_{target_label}"
    PLOTS_DIR = OUT_DIR / "plots"
    DATA_DIR = OUT_DIR / "data"

    for directory in (ANALYSIS_ROOT, OUT_DIR, PLOTS_DIR, DATA_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    if not CUBE_FILE.exists():
        raise FileNotFoundError(
            "Could not find saved preprocessed cube:\n"
            f"{CUBE_FILE}"
        )

    if not METADATA_FILE.exists():
        raise FileNotFoundError(
            "Could not find saved cube metadata:\n"
            f"{METADATA_FILE}"
        )

    # --------------------------------------------------
    # Load cube + metadata
    # --------------------------------------------------
    cube = np.load(
        CUBE_FILE,
        mmap_mode="r",
    )

    if cube.ndim != 3:
        raise ValueError(
            f"Expected cube shape (time, y, x), got {cube.shape}."
        )

    with np.load(METADATA_FILE, allow_pickle=False) as metadata:
        times_s = np.asarray(
            metadata["time_seconds"],
            dtype=np.float64,
        )
        times_min = np.asarray(
            metadata["time_minutes"],
            dtype=np.float64,
        )
        x_km = np.asarray(
            metadata["x_km"],
            dtype=np.float64,
        )
        y_km = np.asarray(
            metadata["y_km"],
            dtype=np.float64,
        )

        local_datetimes = build_local_datetimes(
            metadata,
            times_min,
        )

        input_file = str(
            metadata_scalar(metadata, "input_file", "")
        )
        cube_contents = str(
            metadata_scalar(
                metadata,
                "cube_contents",
                "final_preprocessed_perturbation",
            )
        )
        time_reference = str(
            metadata_scalar(metadata, "time_reference", "frame_center")
        )

    nt, ny, nx = cube.shape

    if len(times_s) != nt or len(times_min) != nt:
        raise ValueError(
            "Metadata time axis does not match cube time dimension."
        )

    if len(x_km) != nx or len(y_km) != ny:
        raise ValueError(
            "Metadata spatial axes do not match cube dimensions."
        )

    if nx < 3 or ny < 3:
        raise ValueError("Cube spatial dimensions are too small.")

    x_step_km = float(np.median(np.diff(x_km)))
    y_step_km = float(np.median(np.diff(y_km)))

    if not np.isfinite(x_step_km) or np.isclose(x_step_km, 0.0):
        raise ValueError("Could not determine X pixel size.")

    if not np.isfinite(y_step_km) or np.isclose(y_step_km, 0.0):
        raise ValueError("Could not determine Y pixel size.")

    # --------------------------------------------------
    # Find 21:30 reference frame
    # --------------------------------------------------
    reference_index, reference_time_error_min = find_reference_frame(
        local_datetimes,
        target_hour,
        target_minute,
    )

    if abs(reference_time_error_min) > MAX_ALLOWED_TARGET_TIME_ERROR_MIN:
        raise ValueError(
            "Nearest frame is too far from requested local time "
            f"{target_hour:02d}:{target_minute:02d}. "
            f"Nearest is {local_datetimes[reference_index]} "
            f"({reference_time_error_min:+.2f} min)."
        )

    first_index = reference_index - NEIGHBOR_FRAMES_EACH_SIDE
    last_index = reference_index + NEIGHBOR_FRAMES_EACH_SIDE

    if first_index < 0 or last_index >= nt:
        raise ValueError(
            "Not enough frames around the selected reference to use "
            f"±{NEIGHBOR_FRAMES_EACH_SIDE} frames."
        )

    selected_indices = np.arange(
        first_index,
        last_index + 1,
        dtype=int,
    )

    reference_position = int(
        np.where(selected_indices == reference_index)[0][0]
    )

    delta_time_s = (
        times_s[selected_indices]
        - times_s[reference_index]
    )
    delta_time_min = delta_time_s / 60.0

    # --------------------------------------------------
    # Correlation ROI
    # --------------------------------------------------
    y_slice, x_slice = select_correlation_slices(
        x_km,
        y_km,
        CORRELATION_HALF_WIDTH_KM,
    )

    x_roi_km = x_km[x_slice]
    y_roi_km = y_km[y_slice]

    frames = [
        prepare_frame(
            cube[index, y_slice, x_slice]
        )
        for index in selected_indices
    ]

    reference_frame = frames[reference_position]

    print("=" * 80)
    print("BLO 2-D CUBE CORRELATION PHASE-SPEED ANALYSIS")
    print("=" * 80)
    print(f"Cube:              {CUBE_FILE}")
    print(f"Metadata:          {METADATA_FILE}")
    print(f"Original input:    {input_file}")
    print(f"Cube contents:     {cube_contents}")
    print(f"Time reference:    {time_reference}")
    print(f"Cube shape:        {cube.shape}")
    print(
        f"Pixel steps:       dx={x_step_km:g} km, "
        f"dy={y_step_km:g} km"
    )
    print(
        f"Correlation ROI:   x={x_roi_km.min():.1f}.."
        f"{x_roi_km.max():.1f} km, "
        f"y={y_roi_km.min():.1f}.."
        f"{y_roi_km.max():.1f} km"
    )
    print(
        f"Requested time:    "
        f"{target_hour:02d}:{target_minute:02d}"
    )
    print(
        f"Reference frame:   {reference_index} | "
        f"{local_datetimes[reference_index]}"
    )
    print(
        f"Clock-time error:  {reference_time_error_min:+.3f} min"
    )
    print(
        f"Selected frames:   {selected_indices[0]}.."
        f"{selected_indices[-1]}"
    )
    print(
        f"Time offsets:      {delta_time_min[0]:+.1f}.."
        f"{delta_time_min[-1]:+.1f} min"
    )
    print(
        f"Max shift search:  {MAX_SHIFT_MAGNITUDE_KM:g} km"
    )

    # Save selected frame image.
    save_selected_frames_plot(
        frames=frames,
        selected_indices=selected_indices,
        local_datetimes=local_datetimes,
        reference_index=reference_index,
        x_roi_km=x_roi_km,
        y_roi_km=y_roi_km,
        output_file=PLOTS_DIR / "selected_frames_around_reference.png",
    )

    # --------------------------------------------------
    # Measure displacement relative to reference
    # --------------------------------------------------
    n_selected = len(selected_indices)

    shift_x_pixels = np.full(
        n_selected,
        np.nan,
        dtype=np.float64,
    )
    shift_y_pixels = np.full(
        n_selected,
        np.nan,
        dtype=np.float64,
    )
    shift_x_km = np.full(
        n_selected,
        np.nan,
        dtype=np.float64,
    )
    shift_y_km = np.full(
        n_selected,
        np.nan,
        dtype=np.float64,
    )
    peak_correlation = np.full(
        n_selected,
        np.nan,
        dtype=np.float64,
    )

    reference_mask = np.zeros(
        n_selected,
        dtype=bool,
    )
    reference_mask[reference_position] = True

    # Exact self-reference.
    shift_x_pixels[reference_position] = 0.0
    shift_y_pixels[reference_position] = 0.0
    shift_x_km[reference_position] = 0.0
    shift_y_km[reference_position] = 0.0
    peak_correlation[reference_position] = 1.0

    correlation_products = []

    for selected_position, frame_index in enumerate(selected_indices):
        if frame_index == reference_index:
            continue

        comparison_frame = frames[selected_position]

        print(
            "\nCorrelating frame "
            f"{frame_index} "
            f"({local_datetimes[frame_index].strftime('%H:%M:%S')}, "
            f"Δt={delta_time_min[selected_position]:+.1f} min)"
        )

        (
            correlation,
            overlap_count,
            shift_y_axis_pixels,
            shift_x_axis_pixels,
        ) = normalized_cross_correlation_2d(
            reference=reference_frame,
            comparison=comparison_frame,
            min_overlap_fraction=MIN_OVERLAP_FRACTION,
            min_overlap_pixels=MIN_OVERLAP_PIXELS,
        )

        (
            search_mask,
            shift_y_axis_km,
            shift_x_axis_km,
        ) = make_shift_search_mask(
            shift_y_pixels=shift_y_axis_pixels,
            shift_x_pixels=shift_x_axis_pixels,
            y_step_km=y_step_km,
            x_step_km=x_step_km,
            maximum_shift_magnitude_km=MAX_SHIFT_MAGNITUDE_KM,
        )

        peak = find_subpixel_correlation_peak(
            correlation=correlation,
            search_mask=search_mask,
            shift_y_pixels=shift_y_axis_pixels,
            shift_x_pixels=shift_x_axis_pixels,
        )

        measured_x_pixels = peak["shift_x_pixels"]
        measured_y_pixels = peak["shift_y_pixels"]

        measured_x_km = measured_x_pixels * x_step_km
        measured_y_km = measured_y_pixels * y_step_km

        shift_x_pixels[selected_position] = measured_x_pixels
        shift_y_pixels[selected_position] = measured_y_pixels
        shift_x_km[selected_position] = measured_x_km
        shift_y_km[selected_position] = measured_y_km
        peak_correlation[selected_position] = peak["peak_correlation"]

        (
            correlation_crop,
            shift_y_crop_km,
            shift_x_crop_km,
        ) = crop_correlation_for_search(
            correlation=correlation,
            shift_y_km=shift_y_axis_km,
            shift_x_km=shift_x_axis_km,
            maximum_shift_magnitude_km=MAX_SHIFT_MAGNITUDE_KM,
        )

        product = {
            "frame_index": int(frame_index),
            "delta_time_min": float(delta_time_min[selected_position]),
            "peak_correlation": float(peak["peak_correlation"]),
            "shift_x_pixels": float(measured_x_pixels),
            "shift_y_pixels": float(measured_y_pixels),
            "shift_x_km": float(measured_x_km),
            "shift_y_km": float(measured_y_km),
            "correlation_crop": correlation_crop,
            "shift_y_crop_km": shift_y_crop_km,
            "shift_x_crop_km": shift_x_crop_km,
        }

        correlation_products.append(product)

        if SAVE_CORRELATION_MAPS:
            np.savez_compressed(
                DATA_DIR
                / (
                    f"correlation_frame_{frame_index:04d}_"
                    f"dt_{delta_time_min[selected_position]:+05.1f}min.npz"
                ),
                frame_index=int(frame_index),
                local_datetime=local_datetimes[frame_index].isoformat(),
                delta_time_min=float(delta_time_min[selected_position]),
                correlation=correlation_crop,
                shift_y_km=shift_y_crop_km,
                shift_x_km=shift_x_crop_km,
                peak_correlation=float(peak["peak_correlation"]),
                peak_shift_x_pixels=float(measured_x_pixels),
                peak_shift_y_pixels=float(measured_y_pixels),
                peak_shift_x_km=float(measured_x_km),
                peak_shift_y_km=float(measured_y_km),
            )

        print(
            f"  Peak r = {peak['peak_correlation']:.4f}"
        )
        print(
            f"  Shift = "
            f"({measured_x_pixels:+.3f}, "
            f"{measured_y_pixels:+.3f}) pixels"
        )
        print(
            f"        = "
            f"({measured_x_km:+.3f}, "
            f"{measured_y_km:+.3f}) km"
        )

    # --------------------------------------------------
    # Fit velocity / phase speed
    # --------------------------------------------------
    fit_results = robust_velocity_fit(
        delta_time_s=delta_time_s,
        shift_x_km=shift_x_km,
        shift_y_km=shift_y_km,
        peak_correlation=peak_correlation,
        reference_mask=reference_mask,
    )

    # --------------------------------------------------
    # Save plots
    # --------------------------------------------------
    save_correlation_maps_plot(
        correlation_products=correlation_products,
        output_file=PLOTS_DIR / "correlation_maps_vs_reference.png",
    )

    save_displacement_fit_plot(
        delta_time_min=delta_time_min,
        shift_x_km=shift_x_km,
        shift_y_km=shift_y_km,
        peak_correlation=peak_correlation,
        reference_mask=reference_mask,
        fit_results=fit_results,
        output_file=PLOTS_DIR / "displacement_vs_time_fit.png",
    )

    save_xy_trajectory_plot(
        delta_time_min=delta_time_min,
        shift_x_km=shift_x_km,
        shift_y_km=shift_y_km,
        reference_mask=reference_mask,
        fit_results=fit_results,
        output_file=PLOTS_DIR / "xy_displacement_trajectory.png",
    )

    # --------------------------------------------------
    # Save data products
    # --------------------------------------------------
    save_measurement_csv(
        output_file=OUT_DIR / "frame_displacement_measurements.csv",
        selected_indices=selected_indices,
        local_datetimes=local_datetimes,
        delta_time_min=delta_time_min,
        shift_x_pixels=shift_x_pixels,
        shift_y_pixels=shift_y_pixels,
        shift_x_km=shift_x_km,
        shift_y_km=shift_y_km,
        peak_correlation=peak_correlation,
        reference_mask=reference_mask,
        fit_mask=fit_results["fit_mask"],
    )

    save_phase_speed_summary_csv(
        output_file=OUT_DIR / "phase_speed_summary.csv",
        reference_index=reference_index,
        reference_datetime=local_datetimes[reference_index],
        fit_results=fit_results,
    )

    np.savez(
        DATA_DIR / "displacement_fit_data.npz",
        selected_frame_indices=selected_indices,
        selected_local_datetime=np.asarray(
            [
                local_datetimes[index].isoformat()
                for index in selected_indices
            ],
            dtype=str,
        ),
        delta_time_seconds=delta_time_s,
        delta_time_minutes=delta_time_min,
        shift_x_pixels=shift_x_pixels,
        shift_y_pixels=shift_y_pixels,
        shift_x_km=shift_x_km,
        shift_y_km=shift_y_km,
        peak_correlation=peak_correlation,
        reference_mask=reference_mask,
        fit_mask=fit_results["fit_mask"],
        reference_frame_index=int(reference_index),
        reference_local_datetime=(
            local_datetimes[reference_index].isoformat()
        ),
        vx_m_per_s=fit_results["vx_m_per_s"],
        vy_m_per_s=fit_results["vy_m_per_s"],
        phase_speed_m_per_s=fit_results["speed_m_per_s"],
        direction_deg=fit_results["direction_deg"],
        direction_0_360_deg=fit_results["direction_0_360_deg"],
        x_intercept_km=fit_results["x_intercept_km"],
        y_intercept_km=fit_results["y_intercept_km"],
        r_squared_x=fit_results["r_squared_x"],
        r_squared_y=fit_results["r_squared_y"],
        r_squared_vector=fit_results["r_squared_vector"],
    )

    with open(
        OUT_DIR / "processing_log.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(f"Cube file: {CUBE_FILE}\n")
        file.write(f"Metadata file: {METADATA_FILE}\n")
        file.write(f"Original input: {input_file}\n")
        file.write(f"Cube contents: {cube_contents}\n")
        file.write(f"Time reference: {time_reference}\n")
        file.write(
            f"Requested local time: "
            f"{target_hour:02d}:{target_minute:02d}\n"
        )
        file.write(
            f"Reference frame: {reference_index}\n"
        )
        file.write(
            f"Reference datetime: "
            f"{local_datetimes[reference_index].isoformat()}\n"
        )
        file.write(
            f"Reference clock error: "
            f"{reference_time_error_min:+.6f} min\n"
        )
        file.write(
            f"Neighbor frames each side: "
            f"{NEIGHBOR_FRAMES_EACH_SIDE}\n"
        )
        file.write(
            f"Selected frames: "
            f"{selected_indices.tolist()}\n"
        )
        file.write(
            f"Correlation half width: "
            f"{CORRELATION_HALF_WIDTH_KM}\n"
        )
        file.write(
            f"Maximum shift magnitude: "
            f"{MAX_SHIFT_MAGNITUDE_KM} km\n"
        )
        file.write(
            f"Minimum fit peak correlation: "
            f"{MIN_PEAK_CORRELATION_FOR_FIT}\n"
        )
        file.write(
            f"Gaussian smoothing: "
            f"{APPLY_GAUSSIAN_SMOOTHING}\n"
        )
        if APPLY_GAUSSIAN_SMOOTHING:
            file.write(
                f"Gaussian sigma: "
                f"{GAUSSIAN_SIGMA_PIXELS} pixels\n"
            )

        file.write("\nFIT RESULT\n")
        file.write(
            f"vx = {fit_results['vx_m_per_s']:.6f} m/s\n"
        )
        file.write(
            f"vy = {fit_results['vy_m_per_s']:.6f} m/s\n"
        )
        file.write(
            f"phase speed = "
            f"{fit_results['speed_m_per_s']:.6f} m/s\n"
        )
        file.write(
            f"direction [-180,180) = "
            f"{fit_results['direction_deg']:.6f} deg\n"
        )
        file.write(
            f"direction [0,360) = "
            f"{fit_results['direction_0_360_deg']:.6f} deg\n"
        )
        file.write(
            f"R2 X = {fit_results['r_squared_x']:.6f}\n"
        )
        file.write(
            f"R2 Y = {fit_results['r_squared_y']:.6f}\n"
        )
        file.write(
            f"R2 vector = "
            f"{fit_results['r_squared_vector']:.6f}\n"
        )
        file.write(
            f"fit points = "
            f"{fit_results['number_of_fit_points']}\n"
        )
        file.write(
            f"rejected = "
            f"{fit_results['number_rejected']}\n"
        )

    # --------------------------------------------------
    # Print final result
    # --------------------------------------------------
    print("\n" + "=" * 80)
    print("2-D CORRELATION PHASE-SPEED RESULT")
    print("=" * 80)
    print(
        f"Reference:   frame {reference_index} | "
        f"{local_datetimes[reference_index]}"
    )
    print(
        f"vx:          {fit_results['vx_m_per_s']:+.2f} m/s"
    )
    print(
        f"vy:          {fit_results['vy_m_per_s']:+.2f} m/s"
    )
    print(
        f"Phase speed: {fit_results['speed_m_per_s']:.2f} m/s"
    )
    print(
        f"Direction:   {fit_results['direction_deg']:+.2f} deg "
        "(0=+X/right, +90=+Y/image down)"
    )
    print(
        f"Vector R²:   {fit_results['r_squared_vector']:.4f}"
    )
    print(
        f"Fit points:  {fit_results['number_of_fit_points']} "
        f"(rejected {fit_results['number_rejected']})"
    )
    print(f"Output:      {OUT_DIR.resolve()}")

    return {
        "target_hour": int(target_hour),
        "target_minute": int(target_minute),
        "target_label": f"{int(target_hour):02d}:{int(target_minute):02d}",
        "reference_frame_index": int(reference_index),
        "reference_local_datetime": local_datetimes[reference_index].isoformat(),
        "reference_time_error_min": float(reference_time_error_min),
        "vx_m_per_s": float(fit_results["vx_m_per_s"]),
        "vy_m_per_s": float(fit_results["vy_m_per_s"]),
        "phase_speed_m_per_s": float(fit_results["speed_m_per_s"]),
        "direction_deg": float(fit_results["direction_deg"]),
        "direction_0_360_deg": float(fit_results["direction_0_360_deg"]),
        "r_squared_x": float(fit_results["r_squared_x"]),
        "r_squared_y": float(fit_results["r_squared_y"]),
        "r_squared_vector": float(fit_results["r_squared_vector"]),
        "number_of_fit_points": int(fit_results["number_of_fit_points"]),
        "number_rejected": int(fit_results["number_rejected"]),
        "output_directory": str(OUT_DIR),
    }



def save_all_times_summary_csv(results, output_file):
    fieldnames = [
        "target_local_time",
        "reference_frame_index",
        "reference_local_datetime",
        "reference_time_error_min",
        "vx_m_per_s",
        "vy_m_per_s",
        "phase_speed_m_per_s",
        "direction_deg_minus180_to_180",
        "direction_deg_0_to_360",
        "r_squared_x",
        "r_squared_y",
        "r_squared_vector",
        "number_of_fit_points",
        "number_rejected",
        "output_directory",
    ]

    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()

        for result in results:
            writer.writerow(
                {
                    "target_local_time": result["target_label"],
                    "reference_frame_index": result["reference_frame_index"],
                    "reference_local_datetime": result["reference_local_datetime"],
                    "reference_time_error_min": result["reference_time_error_min"],
                    "vx_m_per_s": result["vx_m_per_s"],
                    "vy_m_per_s": result["vy_m_per_s"],
                    "phase_speed_m_per_s": result["phase_speed_m_per_s"],
                    "direction_deg_minus180_to_180": result["direction_deg"],
                    "direction_deg_0_to_360": result["direction_0_360_deg"],
                    "r_squared_x": result["r_squared_x"],
                    "r_squared_y": result["r_squared_y"],
                    "r_squared_vector": result["r_squared_vector"],
                    "number_of_fit_points": result["number_of_fit_points"],
                    "number_rejected": result["number_rejected"],
                    "output_directory": result["output_directory"],
                }
            )


def save_all_times_comparison_plot(results, output_file):
    if not results:
        return

    target_minutes = np.array(
        [
            result["target_hour"] * 60.0 + result["target_minute"]
            for result in results
        ],
        dtype=np.float64,
    )

    speed = np.array(
        [result["phase_speed_m_per_s"] for result in results],
        dtype=np.float64,
    )
    direction = np.array(
        [result["direction_deg"] for result in results],
        dtype=np.float64,
    )
    vector_r2 = np.array(
        [result["r_squared_vector"] for result in results],
        dtype=np.float64,
    )

    tick_labels = [result["target_label"] for result in results]

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(10, 9.5),
        sharex=True,
    )

    ax_speed, ax_direction, ax_r2 = axes

    ax_speed.plot(
        target_minutes,
        speed,
        marker="o",
        linewidth=1.5,
    )
    ax_speed.set_ylabel("Phase speed (m/s)")
    ax_speed.set_title("2-D correlation phase speed versus reference time")
    ax_speed.grid(True, alpha=0.3)

    ax_direction.plot(
        target_minutes,
        direction,
        marker="o",
        linewidth=1.5,
    )
    ax_direction.axhline(0.0, linewidth=0.8, linestyle="--")
    ax_direction.set_ylabel("Direction (deg)")
    ax_direction.set_title(
        "Propagation direction (0° = +X/right, +90° = +Y/image down)"
    )
    ax_direction.grid(True, alpha=0.3)

    ax_r2.plot(
        target_minutes,
        vector_r2,
        marker="o",
        linewidth=1.5,
    )
    ax_r2.set_ylim(0.0, 1.0)
    ax_r2.set_ylabel("Vector R²")
    ax_r2.set_xlabel("Reference local time")
    ax_r2.grid(True, alpha=0.3)

    ax_r2.set_xticks(target_minutes)
    ax_r2.set_xticklabels(tick_labels)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def main():
    ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("RUNNING BLO 2-D CUBE CORRELATION ANALYSES")
    print("=" * 80)
    print(
        "Target local times: "
        + ", ".join(f"{hour:02d}:{minute:02d}" for hour, minute in TARGET_LOCAL_TIMES)
    )
    print(
        f"Each target uses ±{NEIGHBOR_FRAMES_EACH_SIDE} neighboring frames."
    )
    print(f"Analysis root: {ANALYSIS_ROOT}")

    successful_results = []
    failed_targets = []

    for target_hour, target_minute in TARGET_LOCAL_TIMES:
        print("\n" + "#" * 80)
        print(f"TARGET {target_hour:02d}:{target_minute:02d}")
        print("#" * 80)

        try:
            result = analyze_target_time(target_hour, target_minute)
            successful_results.append(result)
        except Exception as exc:
            failed_targets.append(
                (f"{target_hour:02d}:{target_minute:02d}", str(exc))
            )
            print(
                f"TARGET {target_hour:02d}:{target_minute:02d} FAILED: {exc}"
            )

    if successful_results:
        save_all_times_summary_csv(
            successful_results,
            ANALYSIS_ROOT / "all_times_phase_speed_summary.csv",
        )

        save_all_times_comparison_plot(
            successful_results,
            ANALYSIS_ROOT / "all_times_phase_speed_comparison.png",
        )

    with open(
        ANALYSIS_ROOT / "all_times_processing_log.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            "Requested target local times: "
            + ", ".join(
                f"{hour:02d}:{minute:02d}"
                for hour, minute in TARGET_LOCAL_TIMES
            )
            + "\n"
        )
        file.write(
            f"Neighbor frames each side: {NEIGHBOR_FRAMES_EACH_SIDE}\n"
        )
        file.write(f"Successful targets: {len(successful_results)}\n")
        file.write(f"Failed targets: {len(failed_targets)}\n")

        if successful_results:
            file.write("\nResults:\n")
            for result in successful_results:
                file.write(
                    f"{result['target_label']}: "
                    f"speed={result['phase_speed_m_per_s']:.3f} m/s, "
                    f"direction={result['direction_deg']:+.3f} deg, "
                    f"vector_R2={result['r_squared_vector']:.4f}, "
                    f"reference_frame={result['reference_frame_index']}, "
                    f"reference_time={result['reference_local_datetime']}\n"
                )

        if failed_targets:
            file.write("\nFailed targets:\n")
            for target_label, reason in failed_targets:
                file.write(f"{target_label}: {reason}\n")

    print("\n" + "=" * 80)
    print("ALL REQUESTED REFERENCE TIMES FINISHED")
    print("=" * 80)

    if successful_results:
        print("\nSummary:")
        print(
            f"{'Target':>7s}  {'Speed (m/s)':>12s}  "
            f"{'Dir (deg)':>10s}  {'Vector R²':>10s}"
        )
        print("-" * 47)

        for result in successful_results:
            print(
                f"{result['target_label']:>7s}  "
                f"{result['phase_speed_m_per_s']:12.2f}  "
                f"{result['direction_deg']:10.2f}  "
                f"{result['r_squared_vector']:10.4f}"
            )

    if failed_targets:
        print("\nFailed targets:")
        for target_label, reason in failed_targets:
            print(f"  {target_label}: {reason}")

    print(f"\nAnalysis root: {ANALYSIS_ROOT.resolve()}")


if __name__ == "__main__":
    main()


RUNNING BLO 2-D CUBE CORRELATION ANALYSES
Target local times: 21:10, 21:20, 21:30, 21:40, 21:50
Each target uses ±4 neighboring frames.
Analysis root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/analysis_2d_correlation_2110_to_2150

################################################################################
TARGET 21:10
################################################################################
BLO 2-D CUBE CORRELATION PHASE-SPEED ANALYSIS
Cube:              /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/preprocessed_perturbation_cube.npy
Metadata:          /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_03_25_2025/preprocessed_perturbation_cube_metadata.npz
Original input:    /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Cube contents:     final_preprocessed_perturbation
Time reference:    frame_center
Cube shape:        (280, 100